# CS781 capstone — MASTER notebook (all four architectures, Phases 3–7)

**How to use:** edit only the **CONTROL** cell (architecture, phase, technique, run mode, your hyperparameters), then Run All / commit.
Every phase section starts with `if PHASE == ...` so only the chosen phase runs.

**Who wrote what (for the LLM declaration):**
- *Your code, copied verbatim from your notebooks:* tokenizer (`a1_tokenizer.py`, `load_tokenizer`), the full Phase-1 code of arch 2 / 3 / 4 (one module each), your Phase-2 mask-aware arch-4 code, `load_encoder`, `load_encoder_a4`, `LoRALinear`, `LoRMALinear`, `inject`, `freeze_all_but`, `trainable_report`, and `backbone_states` (= the body of your Phase-2 `Tagger.forward`).
- *Plumbing written by Claude:* control cell layout, data download/loading/statistics, sentence packing, checkpoint finding, training/eval harness, macro-F1, submission writer, teacher-logit caching.
- *YOUR slots (marked `YOUR CODE`):* arch-1 classes (file not received yet), the Phase-3 model (pooling + document layer + head), PEFT techniques other than LoRA/LoRMA, the KD student and KD loss, and every hyperparameter.

## 0. CONTROL — the only cell you edit per run

In [ ]:
ARCH      = "arch4"
PHASE     = 7
TECHNIQUE = "lora"
RUN_MODE  = "dev"
BIDIR     = False        # Phase 6 loads the "lm" checkpoints
SEED      = 0

CKPTS = {
    "arch1": {"lm": "phase1_arch1_ckpt.pt"},
    "arch2": {"lm": "arch2e2_ckpt.pt", "mlm": "arch2_mlm_best.pt"},
    "arch3": {"lm": "arch3_ckpt.pt",   "mlm": "arch3_tapt_best.pt"},
    "arch4": {"lm": "arch4_ckpt.pt",   "mlm": "arch4_mlm_best.pt"},
}
CKPT_KIND = "mlm" if BIDIR else "lm"

HP = dict(
    DEDUP=True, CHUNK_LEN=1024, OVERLAP=0.2, MAX_CHUNKS=8, KEEP="both", GRAD_CKPT=True,
    TRAIN_FRAC=None, TIME_BUDGET_H=None, EVAL_EVERY=None,
    R=8, ALPHA=8, TARGETS=("q_proj", "k_proj", "v_proj", "out_proj"),
    MAX_SENT_TOK=166, PACK_W=1024, SENT_SEP_EOS=True, MAX_TRAIN_WIN=4,
    EPOCHS=1, LR_ENC=None, LR_HEAD=1e-4, WARMUP_FRAC=0.1, WD=0.01,
    ACCUM_DOCS=None, CLASS_W_POWER=0.2, FINAL_EPOCHS=None, STAGE2_EPOCHS=60, STAGE2_LR=2e-5, FOCAL_GAMMA=2,
    LOSS5="bce", POS_W_POWER=0.7, THR_MIN_POS=55, THR_LO=0.05, THR_HI=0.9, MIN_LABELS=1, EVAL_LABELS="gold",
    STUDENT_LAYERS=2, KD_T=3, KD_ALPHA=0.3,
)
TEACHER_TAG = None
CACHE_TEACHER_LOGITS = False
LOSS = "ce"
P5_MODE = "diag3"
RUN_NOTE = None

P6_MODE = "train"
P6 = dict(
    VAL_N=100, SELECT="lead", SRC_LEN=550, TGT_LEN=470, PREFIX_BIDIR=True,
    LR=1e-4, WARMUP_FRAC=0.1, WD=0.01, CLIP=1.0, ACCUM=32, EPOCHS=2,
    TIME_BUDGET_H=11,    # training cap; setup + decoding + scoring must still fit in Kaggle's 12 h
    EVAL_EVERY=50000, VAL_LOSS_N=100, MOE_AUX_W=0.0,
    DECODE="beam", BEAMS=8, LEN_PENALTY=1.8,
    MIN_NEW=200, MAX_NEW=470, NO_REPEAT_NGRAM=4, REP_PENALTY=1.2,
    VAL_GEN_N=100, TRIM=True,
    TEACHER_FILE=None, STUDENT_LAYERS=None, KD_T=None, KD_ALPHA=None,
    LOAD=None, COMBINE=None, MBR_UTILITY=None,
    SANITY=False,        # must be False in commits
)
# ---- PHASE 7 (prior case retrieval): every None is YOUR value ----
P7_MODE = "train"    # "data" (7.1 statistics + TF-IDF baseline, CPU) | "train" (7.4: train -> dev fusion -> test zip) | "predict" (re-score a saved run)
P7 = dict(
    CKPT="mlm",          # "lm" = Phase-1 LM checkpoints (attached on every account via "without mlm dataset") | "mlm" = your MLM/TAPT ones (only where attached)
    BIDIR_ENC=True,     # True: chunk tokens attend both ways (prefix_mask over the whole chunk) | False: causal (arch1 MUST be False)
    POOL="mean",          # "max" (your Phase-5 LSIModel pooling) | "mean" -- over all real tokens of all chunks, then L2-normalised
    CHUNK_LEN=1024, OVERLAP=0.2, MAX_CHUNKS=8, KEEP="both",   # your Phase-4/5 long-document values (HP); evaluation embeds up to MAX_CHUNKS
    TRAIN_CHUNKS=1,  # chunks per document while TRAINING (cost grows linearly with it); evaluation always uses MAX_CHUNKS
    TOPK=100,          # TF-IDF candidates re-ranked per query (your dev data: R@50 .679, R@100 .749, R@200 .819)
    N_NEG=3,         # hard negatives per training pair (each costs TRAIN_CHUNKS extra forward+backward passes)
    NEG_TOP=50,       # hard negatives are drawn from the TF-IDF top-NEG_TOP non-relevant candidates of the query
    TAU=0.05,           # InfoNCE temperature on cosine similarity (softmax over [relevant, negatives] / TAU)
    LR=5e-4, WARMUP_FRAC=0.1, WD=0.01, CLIP=1.0, ACCUM=32,
    EPOCHS=2,        # passes over the ~5.3k (query, relevant) training pairs
    TIME_BUDGET_H=8, # training hours; afterwards ~1 h: embedding dev + test pools, fusion, zips (all inside Kaggle's 12 h)
    TEACHER_FILE=None, STUDENT_LAYERS=None, KD_T=None, KD_ALPHA=None,   # KD run only
    LOAD=None,          # predict mode: saved run, e.g. "p7_arch3_full.pt"
    SANITY=False,       # interactive only: 80 updates on 8 pairs, loss should fall; must be False in commits
)

def need6(*keys):
    missing = [k for k in keys if P6.get(k) is None]
    assert not missing, f"fill these in P6 (CONTROL cell) first: {missing}"

def need(*keys):
    missing = [k for k in keys if HP.get(k) is None]
    assert not missing, f"fill these in the CONTROL cell first: {missing}"

RUN_TAG = f"p{PHASE}_{ARCH}_{TECHNIQUE}_{RUN_MODE}"
print("RUN:", RUN_TAG, "| checkpoint kind:", CKPT_KIND)


## 1. Setup

In [ ]:
import os; os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation on long cases
import os, json, glob, math, time, random, zipfile, collections, re, bisect
import numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, asdict

device = "cuda" if torch.cuda.is_available() else "cpu"
DEV_TYPE = "cuda" if device == "cuda" else "cpu"
EOS_ID, MASK_ID = 65354, 65355
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", device)

def find_input(name):
    if name.startswith("/") and os.path.exists(name): return name
    hits = [os.path.join(r, n) for r, ds, fs in os.walk("/kaggle/input", followlinks=True)
            for n in fs + ds if n == name]
    assert hits, f"{name} not found under /kaggle/input -- attach the dataset/notebook output that holds it (restart the session if you attached it after starting)"
    return sorted(hits, key=os.path.isdir)[0]          # prefer a real file over a folder

In [ ]:
# official data, per phase (files as listed in the capstone PDF)
REPO = "Exploration-Lab/CS-781-Capstone"
PHASE_FILES = {
    3: ("Phase_3_RR",   ["Train/CL_train.json", "Train/CL_dev.json", "Train/IT_train.json", "Train/IT_dev.json",
                         "Train/label_vocab.json", "Test/CL_test.json", "Test/IT_test.json"]),
    4: ("Phase_4_CJPE", ["Train/train.json", "Train/dev.json", "Test/test.json"]),
    5: ("Phase_5_LSI",  ["Train/train.json", "Train/dev.json", "Test/test.json"]),
    6: ("Phase_6_Summ", ["Train/train.json", "Test/test.json"]),
    7: ("Phase_7_PCR",  ["Train/train.json", "Train/train_candidates.json", "Dev/dev.json",
                         "Dev/dev_candidates.json", "Test/test.json", "Test/test_candidates.json"]),
}

def fetch_phase(phase):
    """Uses attached copies if present (offline-safe), otherwise downloads with HF_TOKEN."""
    folder, files = PHASE_FILES[phase]
    paths = {}
    for f in files:
        local = glob.glob(f"/kaggle/input/**/{os.path.basename(f)}", recursive=True)
        if local and phase == 3:                  # phase-3 file names are unique; later phases share names (train.json ...)
            paths[f] = local[0]; continue
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import hf_hub_download
        tokn = UserSecretsClient().get_secret("HF_TOKEN")
        paths[f] = hf_hub_download(REPO, f"{folder}/{f}", repo_type="dataset", token=tokn,
                                   force_download=(phase == 4))      # the Phase-4 data was replaced on Oct 20
    if phase == 5:                                   # list the whole official Phase-5 folder: statute texts or other files?
        from huggingface_hub import HfApi
        allf = [f for f in HfApi().list_repo_files(REPO, repo_type="dataset", token=tokn) if f.startswith(folder + "/")]
        print("official Phase-5 files:", allf)
        for f in allf:
            rel = f[len(folder) + 1:]
            if rel not in paths:
                paths[rel] = hf_hub_download(REPO, f, repo_type="dataset", token=tokn)
    if phase == 4:
        from huggingface_hub import HfApi
        print("dataset revision:", HfApi().dataset_info(REPO, token=tokn).sha)   # note this in the report)
    for f, p in paths.items():
        print(f"{f:32s} {os.path.getsize(p) // 1024:>8} KB  {p}")
    return paths

P = fetch_phase(PHASE)

## 2. Tokenizer (your Assignment-1 BPE, verbatim)

In [ ]:
%%writefile a1_tokenizer.py
"""Expose your Assignment-1 BPE implementation to Assignment 3.

Reuse your A-1 tokenizer code/class here. The important change is that it must be
RETRAINED on the `train_text` supplied by Assignment 3 rather than reusing a tokenizer
trained on the Assignment-1 dataset.
"""
import re
from collections import Counter

PAT=re.compile(r" ?[A-Za-z]+| ?\d+| ?[^\sA-Za-z\d]+|\s+")
class BPETokeniser:#defining the class
    def __init__(self):
        self.merges={}
        self.vocab={i: bytes([i]) for i in range(256)}
        self.cache={}

    @property
    def vocab_size(self):#vocab size
        return len(self.vocab)

    def _count_pairs(self,words,counts):#how actuall work we start counting how many words have same pair 
        pair_counts={}
        for word,count in zip(words,counts):
            for a,b in zip(word,word[1:]):
                pair_counts[(a,b)]=pair_counts.get((a,b),0)+count
        return pair_counts
    def _merge(self,ids,pair,new_id):
        out=[]
        i=0
        while i<len(ids):
            if i+1<len(ids) and ids[i]==pair[0] and ids[i+1]==pair[1]:
                out.append(new_id)
                i+=2
            else:
                out.append(ids[i])
                i+=1
        return out
    def train(self,text,vocab_size):
        chunks=PAT.findall(text)
        chunk_counts=Counter(chunks)
        words=[]
        counts=[]
        for chunk,n in chunk_counts.items():
            words.append(list(chunk.encode("utf-8")))
            counts.append(n)
        for r in range(vocab_size-256):
                pair_counts=self._count_pairs(words,counts)
                if not pair_counts:
                    break
                best = min(pair_counts, key=lambda p: (-pair_counts[p], p))
                new_id=256+r
                self.merges[best]=new_id
                self.vocab[new_id]=self.vocab[best[0]] + self.vocab[best[1]]
                words=[self._merge(w,best,new_id) for w in words]
    def _encode_chunk(self,chunk):
        if chunk in self.cache:
            return self.cache[chunk]
        ids=list(chunk.encode("utf-8"))
        while True:
            candidates=[p for p in zip(ids,ids[1:]) if p in self.merges]
            if not candidates:
                break
            best=min(candidates,key=lambda p:self.merges[p])
            ids=self._merge(ids,best,self.merges[best])
        self.cache[chunk]=ids
        return ids
    def encode(self,text):
        ids=[]
        for chunk in PAT.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids
    def decode(self,ids):
        data=b"".join(self.vocab[i] for i in ids)
        return data.decode("utf-8",errors="replace")
             
         

def train_tokenizer(train_text: str, vocab_size: int):
    """Train your A-1 BPE tokenizer on `train_text` and return the trained tokenizer.

    Required returned interface:
      tokenizer.encode(text) -> sequence of integer ids
      tokenizer.decode(ids) -> string
      tokenizer.vocab_size OR tokenizer.get_vocab_size()

    Replace this stub with your Assignment-1 tokenizer implementation/training call.
    """
    tok=BPETokeniser()
    tok.train(train_text,vocab_size)
    return tok

In [ ]:
import re
from a1_tokenizer import BPETokeniser
mp = find_input("legal_bpe_merges.json")
raw = json.load(open(mp))
print(type(raw).__name__, len(raw))

In [ ]:
def load_tokenizer(raw):
    tok = BPETokeniser()
    if isinstance(raw, dict) and "merges" in raw:
        raw = raw["merges"]
    triples = []
    if isinstance(raw, dict):                                  # {"a,b": new_id}
        for k, v in raw.items():
            a, b = map(int, re.split(r"[,\s]+", k.strip("()[] ")))
            triples.append((a, b, int(v)))
    else:                                                      # list form
        for i, m in enumerate(raw):
            triples.append((int(m[0]), int(m[1]), int(m[2]) if len(m) == 3 else 256 + i))
    for a, b, new in sorted(triples, key=lambda t: t[2]):      # rebuild in learned order
        tok.merges[(a, b)] = new
        tok.vocab[new] = tok.vocab[a] + tok.vocab[b]
    return tok

tok = load_tokenizer(raw)
print("merges:", len(tok.merges), "| vocab:", tok.vocab_size)   # expect 65,098 | 65,354

## 3. The four architectures — each one complete, in its own module
Each cell below writes **one architecture's full Phase-1 code, verbatim from your Phase-1 notebook**, into its own file
(`arch1_model.py` … `arch4_model.py`) and imports it as `A1` … `A4`. Keeping them as separate modules means the four
sets of classes never overwrite each other (all of them define `GQAttention`, `RMSNorm`, …), and each architecture can be
shown on its own. A quick parameter count (on the `meta` device, so no memory is used) checks each one against Phase 1.

### 3.1 Architecture 1 — vanilla decoder (your A3 attention, sinusoidal positions, GELU FFN)
**YOUR CODE:** paste the model classes from `phase1-train (7).ipynb` into this cell, plus `load_encoder_a1(path) -> (encoder, cfg)` and `states_a1(encoder, idx) -> (B, T, d_model)` (hidden states after the final norm).

In [ ]:
%%writefile arch1_model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

# ===== arch 1: your Phase-1 code, verbatim from phase1-train (7).ipynb (attention.py + model_arch.py) =====
#this is the same attention that i built in the A3 question 1


import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        
        self.q_proj = nn.Linear(d_model,d_model,bias=False)
        self.k_proj = nn.Linear(d_model,d_model,bias=False)
        self.v_proj = nn.Linear(d_model,d_model,bias=False)
        self.out_proj = nn.Linear(d_model,d_model,bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask",mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
       
        B, T, C = x.shape
        
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = (q * self.attention_scale) @ k.transpose(-2, -1)
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)
        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))
        return out


"""Decoder-only Transformer LM, built from scratch (no nn.Transformer*)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          
    d_ff: int = 1024
    max_seq_len: int =1024
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   
    attention_scale: float = None      
    norm_first: bool = False            

def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    # TODO: implement PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    #               PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    pos = torch.arange(max_len).unsqueeze(1).float()
    i2 = torch.arange(0, d_model, 2).float()
    div = torch.pow(10000.0, i2 / d_model)
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(pos / div)
    pe[:, 1::2] = torch.cos(pos / div)
    return pe
    


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: two linear layers with a GELU in between, plus dropout
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first  

    def forward(self, x):
        # TODO: implement pre-norm when norm_first=True,
        # and post-norm when norm_first=False.
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x
        


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # (RoPE is applied inside attention instead of here — see Question 3)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=cfg.d_model ** -0.5)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        # TODO: run x through every block, then ln_f, then self.head
        x = self.tok_emb(idx) * math.sqrt(self.cfg.d_model)
        
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T]
            
        for block in self.blocks:
            x = block(x)
            
        x = self.ln_f(x)
        return self.head(x * self.cfg.d_model ** -0.5)


# ---- loader + hidden states (plumbing around your classes) ----
def load_encoder_a1(path):
    """Your Phase-1 checkpoint ({"model", "cfg", ...}) -> (DecoderLM, LMConfig)."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = LMConfig(**ck["cfg"])
    enc = DecoderLM(cfg)
    enc.load_state_dict(ck["model"])
    print(path.split("/")[-1], "| step", ck.get("step"), "| best_val", round(ck.get("best_val") or float("nan"), 3),
          "| vocab", cfg.vocab_size, f"| {sum(p.numel() for p in enc.parameters()):,} params")
    del ck
    return enc, cfg

def states_a1(enc, idx):
    """= your DecoderLM.forward without the LM head: hidden states after ln_f (causal, mask is built in)."""
    T = idx.size(1)
    x = enc.tok_emb(idx) * math.sqrt(enc.cfg.d_model)
    if enc.cfg.pos_emb_type == "sinusoidal":
        x = x + enc.pos_emb[:T]
    for block in enc.blocks:
        x = block(x)
    return enc.ln_f(x)

### 3.2 Architecture 2 — LLaMA-style decoder (RoPE, RMSNorm, SwiGLU, GQA) — from `phase1-arch2-6.ipynb`

In [ ]:
%%writefile arch2_model.py
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        scores=scores.masked_fill(~self.causal_mask[:T,:T],float("-inf"))
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))

from dataclasses import dataclass

@dataclass
class LlamaConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0

class LlamaBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x):
        x=x+self.attn(self.norm1(x))
        x=x+self.ffn(self.norm2(x))
        return x

class LlamaLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([LlamaBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx):
        x=self.tok_emb(idx)
        for block in self.blocks:
            x=block(x)
        x=self.norm_f(x)
        return self.head(x)

### 3.3 Architecture 3 — prefix-LM on the arch-2 blocks (prefix mask, mask-aware attention) — from `phase1-arch3-240218.ipynb`

In [ ]:
%%writefile arch3_model.py
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

def prefix_mask(T,prefix_lens,device=None):
    causal=torch.tril(torch.ones(T,T,dtype=torch.bool,device=device))
    pos=torch.arange(T,device=device)
    in_prefix=pos[None,:]<prefix_lens[:,None]
    both=in_prefix[:,:,None] & in_prefix[:,None,:]
    return (causal[None] | both)[:, None]

class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x,mask=None):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        if mask is None:
            mask=self.causal_mask[:T,:T]
        scores=scores.masked_fill(~mask,float("-inf"))
        
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))

from dataclasses import dataclass

@dataclass
class LlamaConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0

class LlamaBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x,mask=None):
        x=x+self.attn(self.norm1(x),mask)
        x=x+self.ffn(self.norm2(x))
        return x

class PrefixLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([LlamaBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx,prefix_lens=None):
        if prefix_lens is not None:
            mask=prefix_mask(idx.size(1),prefix_lens,idx.device)
        else:
            mask=None
        x=self.tok_emb(idx)
        for block in self.blocks: x=block(x,mask)
        x=self.norm_f(x)
        return self.head(x)

### 3.4 Architecture 4 — dense + MoE hybrid (4 experts, top-2, on odd layers) — from `phase1-arch4.ipynb`

In [ ]:
%%writefile arch4_model.py
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        scores=scores.masked_fill(~self.causal_mask[:T,:T],float("-inf"))
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))

class MoEFFN(nn.Module):
    def __init__(self,d_model,d_ff,n_experts,top_k,dropout=0.0):
        super().__init__()
        self.n_experts=n_experts
        self.top_k=top_k
        self.router=nn.Linear(d_model,n_experts,bias=False)
        self.experts=nn.ModuleList([SwiGLU(d_model,d_ff,dropout) for _ in range(n_experts)])
        self.aux_loss=torch.tensor(0.0)
    def forward(self,x):
        B,T,d=x.shape
        flat=x.reshape(-1,d)
        with torch.autocast("cuda",enabled=False):
            logits=self.router(flat.float())
        probs=F.softmax(logits,dim=-1)
        top_p,top_i=probs.topk(self.top_k,dim=-1)
        top_p=top_p/top_p.sum(-1,keepdim=True)
        out=torch.zeros_like(flat)
        for e,expert in enumerate(self.experts):
            tok,slot=(top_i==e).nonzero(as_tuple=True)
            if tok.numel()==0: continue
            y=expert(flat[tok])
            out.index_add_(0,tok,(y*top_p[tok,slot,None]).to(out.dtype))
        f=torch.bincount(top_i.flatten(),minlength=self.n_experts).float()/top_i.numel()
        P=probs.mean(0)
        self.aux_loss=self.n_experts*(f*P).sum()
        return out.view(B,T,d)

@dataclass
class HybridConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0
    n_experts:int=4
    top_k:int=2
    d_ff_expert:int=1024

class HybridBlock(nn.Module):
    def __init__(self,cfg,layer_id):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.is_moe=(layer_id%2==1)
        if self.is_moe:
            self.ffn=MoEFFN(cfg.d_model,cfg.d_ff_expert,cfg.n_experts,cfg.top_k,cfg.dropout)
        else:
            self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x):
        x=x+self.attn(self.norm1(x))
        x=x+self.ffn(self.norm2(x))
        return x

class HybridLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([HybridBlock(cfg,i) for i in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx):
        x=self.tok_emb(idx)
        for block in self.blocks:
            x=block(x)
        x=self.norm_f(x)
        return self.head(x)
    @property
    def aux_loss(self):
        moe=[b.ffn.aux_loss for b in self.blocks if b.is_moe]
        return sum(moe)/len(moe)

In [ ]:
import importlib, arch1_model, arch2_model, arch3_model, arch4_model
for _m in (arch1_model, arch2_model, arch3_model, arch4_model): importlib.reload(_m)
A1, A2, A3, A4 = arch1_model, arch2_model, arch3_model, arch4_model

with torch.device("meta"):          # builds the Phase-1 configs without allocating memory
    n2 = sum(p.numel() for p in A2.LlamaLM(A2.LlamaConfig(65355, 1024, 10, 16, 8, 2816)).parameters())
    n3 = sum(p.numel() for p in A3.PrefixLM(A3.LlamaConfig(65355, 1024, 10, 16, 8, 2816)).parameters())
    n4 = sum(p.numel() for p in A4.HybridLM(A4.HybridConfig(65355, 768, 16, 12, 6, 2048)).parameters())
print(f"arch2 {n2:,} (Phase 1: 184,909,824) | arch3 {n3:,} (184,909,824) | arch4 {n4:,} (191,800,320)")

### 3.5 Fine-tuning versions (the `mask` argument you added in Phase 2)
Arch 2's and arch 4's Phase-1 attention is causal-only. In Phase 2 you ran them through the **mask-aware** blocks
(arch 3's `GQAttention`, and `HybridBlock.forward(x, mask)`), which load the same weights: the parameter names are identical,
and with no mask they compute exactly the causal Phase-1 model. The cell below is your Phase-2 arch-4 code, verbatim.

In [ ]:
%%writefile arch4_ft.py
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

from arch3_model import RMSNorm, rope_tables, apply_rope, prefix_mask, GQAttention, SwiGLU

# ===== arch 4: dense + MoE hybrid (classes copied from my Phase 1 notebook phase1-arch4) =====
class MoEFFN(nn.Module):
    def __init__(self,d_model,d_ff,n_experts,top_k,dropout=0.0):
        super().__init__()
        self.n_experts=n_experts
        self.top_k=top_k
        self.router=nn.Linear(d_model,n_experts,bias=False)
        self.experts=nn.ModuleList([SwiGLU(d_model,d_ff,dropout) for _ in range(n_experts)])
        self.aux_loss=torch.tensor(0.0)
    def forward(self,x):
        B,T,d=x.shape
        flat=x.reshape(-1,d)
        with torch.autocast("cuda",enabled=False):
            logits=self.router(flat.float())
        probs=F.softmax(logits,dim=-1)
        top_p,top_i=probs.topk(self.top_k,dim=-1)
        top_p=top_p/top_p.sum(-1,keepdim=True)
        out=torch.zeros_like(flat)
        for e,expert in enumerate(self.experts):
            tok,slot=(top_i==e).nonzero(as_tuple=True)
            if tok.numel()==0: continue
            y=expert(flat[tok])
            out.index_add_(0,tok,(y*top_p[tok,slot,None]).to(out.dtype))
        f=torch.bincount(top_i.flatten(),minlength=self.n_experts).float()/top_i.numel()
        P_=probs.mean(0)
        self.aux_loss=self.n_experts*(f*P_).sum()
        return out.view(B,T,d)

@dataclass
class HybridConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0
    n_experts:int=4
    top_k:int=2
    d_ff_expert:int=1024

class HybridBlock(nn.Module):
    def __init__(self,cfg,layer_id):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.is_moe=(layer_id%2==1)
        if self.is_moe:
            self.ffn=MoEFFN(cfg.d_model,cfg.d_ff_expert,cfg.n_experts,cfg.top_k,cfg.dropout)
        else:
            self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x,mask=None):            # mask arg added so the Phase-2 Tagger can call block(x, mask)
        x=x+self.attn(self.norm1(x),mask)
        x=x+self.ffn(self.norm2(x))
        return x

class HybridLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([HybridBlock(cfg,i) for i in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx):
        x=self.tok_emb(idx)
        for block in self.blocks:
            x=block(x)
        return self.head(self.norm_f(x))

def load_encoder_a4(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = HybridConfig(**ck["cfg"])
    enc = HybridLM(cfg); enc.load_state_dict(ck["model"])
    print(path.split("/")[-1], "| step", ck.get("step"), "| best_val", round(ck.get("best_val", float("nan")), 3),
          "| vocab", cfg.vocab_size, f"| {sum(p.numel() for p in enc.parameters()):,} params",
          "| MoE layers", sum(b.is_moe for b in enc.blocks), "of", len(enc.blocks))
    del ck
    return enc, cfg

## 4. MLM adaptation — kept aside
MLM training runs in its own notebooks (`arch3-mlm`, and the `arch2-mlm` copy of it), not here.
Their outputs (`arch2_mlm_best.pt`, `arch3_mlm_best.pt`, later `arch4_mlm_best.pt`) are plain checkpoints in the usual
`{"model", "cfg"}` format, so section 5 loads them exactly like the Phase-1 checkpoints.

## 5. Backbone loader

In [ ]:
import torch.utils.checkpoint
import arch4_ft; importlib.reload(arch4_ft)
from arch3_model import LlamaConfig, PrefixLM, prefix_mask      # used by your load_encoder below
from arch4_ft import load_encoder_a4

def load_encoder(path):
    ck = torch.load(path, map_location="cpu")
    cfg = LlamaConfig(**ck["cfg"])
    enc = PrefixLM(cfg); enc.load_state_dict(ck["model"])
    print(path.split("/")[-1], "| step", ck.get("step"), "| best_val", round(ck.get("best_val", float("nan")), 3),
          "| vocab", cfg.vocab_size, f"| {sum(p.numel() for p in enc.parameters()):,} params")
    del ck
    return enc, cfg

def backbone_states(enc, idx, prefix_lens=None):
    """Your Phase-2 Tagger.forward without the tag head: hidden states after norm_f.
    prefix_lens = window lengths -> fully bidirectional over the real tokens; None -> causal."""
    mask = prefix_mask(idx.size(1), prefix_lens, idx.device) if prefix_lens is not None else None
    x = enc.tok_emb(idx)
    for block in enc.blocks:
        if HP.get("GRAD_CKPT") and enc.training and torch.is_grad_enabled():
            x = torch.utils.checkpoint.checkpoint(block, x, mask, use_reentrant=False)   # memory switch: recompute in backward
        else:
            x = block(x, mask)
    return enc.norm_f(x)

def load_backbone(arch=None, kind=None):
    """arch2 / arch3 -> PrefixLM (arch-3 module), arch4 -> mask-aware HybridLM (arch4_ft), arch1 -> your arch1_model."""
    arch, kind = arch or ARCH, kind or CKPT_KIND
    assert kind in CKPTS[arch], f"{arch} has no '{kind}' checkpoint listed in CKPTS"
    path = find_input(CKPTS[arch][kind])
    if arch in ("arch2", "arch3"):
        enc, cfg = load_encoder(path)
    elif arch == "arch4":
        enc, cfg = load_encoder_a4(path)
    else:
        assert hasattr(A1, "load_encoder_a1"), "fill section 3.1 (arch 1) first"
        enc, cfg = A1.load_encoder_a1(path)
    return enc, cfg.d_model

def encoder_states(enc, win_ids, win_lens):
    """What your Phase-3 model calls: (n_win, L) ids + (n_win,) lengths -> (n_win, L, d_model)."""
    if ARCH == "arch1":
        return A1.states_a1(enc, win_ids)
    return backbone_states(enc, win_ids, win_lens if BIDIR else None)

## 6. Fine-tuning techniques
Your LoRA / LoRMA and helpers (verbatim), then `apply_technique`, then the slot for the techniques not written yet.

In [ ]:
# ===== PEFT plumbing: freeze the backbone, report what is trainable =====
def freeze_all_but(model, keep):
    """keep(name, param) -> True if this parameter should train."""
    for n, p in model.named_parameters():
        p.requires_grad = bool(keep(n, p))

def trainable_report(model, tag=""):
    seen = {id(p): p for p in model.parameters()}.values()
    tot = sum(p.numel() for p in seen); tr = sum(p.numel() for p in seen if p.requires_grad)
    print(f"{tag} trainable {tr:,} / {tot:,} params ({tr / tot:.3%})")

In [ ]:
# ===== LoRA layer (my implementation): W' = W + (alpha/r) B A, low-rank term on the INPUT x =====
import torch
import torch.nn as nn
import math
class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,r:int,alpha:float):
        super().__init__()
        self.b=base
        for p in self.b.parameters():p.requires_grad=False
        i,o=base.in_features,base.out_features
        self.lora_A=nn.Parameter(torch.empty(r,i))
        self.lora_B=nn.Parameter(torch.zeros(o,r))
        nn.init.kaiming_uniform_(self.lora_A,a=math.sqrt(5))
        self.s=alpha/r
    def forward(self,x):
        return self.b(x)+self.s*(x@self.lora_A.T@self.lora_B.T)

In [ ]:
# ===== LoRMA layer (my implementation): W' = (I + (alpha/r) B A) W, applied to the frozen Linear's output =====
import torch
import torch.nn as nn
import math

class LoRMALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float):
        super().__init__()
        self.base = base

        for param in self.base.parameters():
            param.requires_grad = False

        d_out = base.out_features

        self.lorma_A = nn.Parameter(torch.empty(r, d_out))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))

        self.lorma_B = nn.Parameter(torch.zeros(d_out, r))

        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + self.scale * (h @ self.lorma_A.T @ self.lorma_B.T)

In [ ]:
def inject(model, wrap_fn, targets):
    """Replace every nn.Linear child whose attribute name is in `targets` with wrap_fn(linear)."""
    n = 0
    for _, mod in list(model.named_modules()):
        for t in targets:
            child = getattr(mod, t, None)
            if isinstance(child, nn.Linear):
                setattr(mod, t, wrap_fn(child)); n += 1
    return n

In [ ]:
def apply_technique(model, technique=None):
    """Wraps model.encoder for the chosen technique and freezes accordingly.
    Rule used for every PEFT method: the technique's own parameters + everything outside model.encoder
    (your pooling / document layer / head) train; the rest of the encoder is frozen."""
    t = technique or TECHNIQUE
    if t == "full":
        for p in model.parameters(): p.requires_grad = True
    elif t == "lora":
        need("R", "ALPHA")
        n = inject(model.encoder, lambda lin: LoRALinear(lin, HP["R"], HP["ALPHA"]), HP["TARGETS"])
        freeze_all_but(model, lambda n_, p: ("lora" in n_) or not n_.startswith("encoder."))
        print("LoRA wrapped", n, "Linears")
    elif t == "lorma":
        need("R", "ALPHA")
        n = inject(model.encoder, lambda lin: LoRMALinear(lin, HP["R"], HP["ALPHA"]), HP["TARGETS"])
        freeze_all_but(model, lambda n_, p: ("lorma" in n_) or not n_.startswith("encoder."))
        print("LoRMA wrapped", n, "Linears")
    elif t in EXTRA_TECHNIQUES:
        EXTRA_TECHNIQUES[t](model)
    else:
        raise NotImplementedError(f"technique '{t}' not written yet -- add it to EXTRA_TECHNIQUES in the next cell")
    trainable_report(model, f"{ARCH}+{t}")
    return model

In [ ]:
def apply_rslora(model):
    need("R", "ALPHA")
    inject(model.encoder, lambda lin: LoRALinear(lin, HP["R"], HP["ALPHA"]), HP["TARGETS"])
    
    count = 0
    scale = HP["ALPHA"] / (HP["R"] ** 0.5)
    
    for module in model.encoder.modules():
        if isinstance(module, LoRALinear):
            module.s = scale
            count += 1
            
    freeze_all_but(model, lambda n_, p: ("lora" in n_) or not n_.startswith("encoder."))
    print("rsLoRA wrapped", count, "Linears | scale alpha/sqrt(r) =", round(scale, 4))

EXTRA_TECHNIQUES = {
    "rslora": apply_rslora,
}

In [ ]:
def apply_rslora(model):
    need("R", "ALPHA")
    inject(model.encoder, lambda lin: LoRALinear(lin, HP["R"], HP["ALPHA"]), HP["TARGETS"])
    
    count = 0
    scale = HP["ALPHA"] / (HP["R"] ** 0.5)
    
    for module in model.encoder.modules():
        if isinstance(module, LoRALinear):
            module.s = scale
            count += 1
            
    freeze_all_but(model, lambda n_, p: ("lora" in n_) or not n_.startswith("encoder."))
    print("rsLoRA wrapped", count, "Linears | scale alpha/sqrt(r) =", round(scale, 4))

def apply_partial(model):
    """layer-wise fine-tuning: only the top PARTIAL_TOP_K encoder blocks, the final norm and the task head train.
    PARTIAL_TOP_K = a block count (1..n) or a fraction of the blocks (0 < x < 1, rounded, at least 1 block)."""
    need("PARTIAL_TOP_K")
    n = len(model.encoder.blocks); k = HP["PARTIAL_TOP_K"]
    if 0 < k < 1: k = max(1, int(round(k * n)))
    k = int(k); assert 1 <= k <= n, f"PARTIAL_TOP_K must be 1..{n} blocks (or a fraction 0..1)"
    keep = tuple(f"encoder.blocks.{i}." for i in range(n - k, n)) + ("encoder.ln_f.", "encoder.norm_f.")
    freeze_all_but(model, lambda n_, p: not n_.startswith("encoder.") or n_.startswith(keep))
    n_drop = 0
    for i in range(n - k):                                  # frozen blocks: no dropout -> stable inputs for the trainable top blocks
        for mod in model.encoder.blocks[i].modules():
            if isinstance(mod, torch.nn.Dropout): mod.p = 0.0; n_drop += 1
    print(f"partial: training top {k} of {n} encoder blocks + final norm + head | dropout off in {n_drop} layers of the {n - k} frozen blocks")

EXTRA_TECHNIQUES = {
    "rslora": apply_rslora,
    "partial": apply_partial,
}

EXTRA_TECHNIQUES["partial"] = apply_partial


# PHASE 3 — Rhetorical Role prediction

### 3.1 Data + statistics

In [ ]:
if PHASE == 3:
    def to_docs(obj):
        if isinstance(obj, list):                                            # [ {id, text, labels}, ... ]
            return obj
        vals = list(obj.values())
        if all(isinstance(v, dict) for v in vals):                           # {doc_id: {text, labels}}
            return [dict(id=k, **v) for k, v in obj.items()]
        if "text" in obj and isinstance(obj["text"], list) and obj["text"] and isinstance(obj["text"][0], list):
            n = len(obj["text"])                                             # columnar {id:[...], text:[[...]], labels:[[...]]}
            docs = [{k: obj[k][i] for k in obj if isinstance(obj[k], list) and len(obj[k]) == n} for i in range(n)]
            for i, d in enumerate(docs): d.setdefault("id", str(i))
            return docs
        for v in vals:
            if isinstance(v, list) and v and isinstance(v[0], dict):         # {"data": [ {...}, ... ]}
                return v
        if all(isinstance(v, list) and (not v or isinstance(v[0], str)) for v in vals):
            return [{"id": k, "text": v} for k, v in obj.items()]            # {doc_id: [sentences]}
        raise ValueError(f"unknown layout: top keys {list(obj)[:5]}, first value type {type(vals[0])}")

    vocab = json.load(open(P["Train/label_vocab.json"]))
    if isinstance(vocab, dict) and all(str(k).isdigit() for k in vocab):          # {"0": "Fact", ...}
        LABELS = [vocab[k] for k in sorted(vocab, key=int)]
    elif isinstance(vocab, dict):                                                  # {"Fact": 0, ...}
        LABELS = sorted(vocab, key=lambda k: vocab[k]) if all(isinstance(v, int) for v in vocab.values()) else list(vocab)
    else:
        LABELS = list(vocab)
    L2I = {l: i for i, l in enumerate(LABELS)}
    print("labels:", LABELS)

    RAW = {}
    for dom in ("CL", "IT"):
        for split in ("train", "dev", "test"):
            key = f"{'Test' if split == 'test' else 'Train'}/{dom}_{split}.json"
            obj = json.load(open(P[key]))
            try:
                RAW[(dom, split)] = to_docs(obj)
            except Exception as e:
                print(f"!! {key}: top-level type {type(obj).__name__}, "
                      f"first keys {list(obj)[:3] if isinstance(obj, dict) else '-'}, "
                      f"first item {str(obj[0] if isinstance(obj, list) else next(iter(obj.values())))[:200]}")
                raise
            d0 = RAW[(dom, split)][0]
            print(f"{dom}_{split:5s}: loaded {len(RAW[(dom, split)])} docs | keys {list(d0.keys())}")

    ids_cl = {d["id"] for s in ("train", "dev", "test") for d in RAW[("CL", s)]}
    ids_it = {d["id"] for s in ("train", "dev", "test") for d in RAW[("IT", s)]}
    test_ids = [d["id"] for dom in ("CL", "IT") for d in RAW[(dom, "test")]]
    assert len(test_ids) == len(set(test_ids)), "duplicate test ids across CL/IT -- the output dict would collide"
    print("CL/IT id overlap:", len(ids_cl & ids_it))

    def pct(xs, q):
        xs = sorted(xs); return xs[min(len(xs) - 1, int(q * len(xs)))]
    for (dom, split), docs in RAW.items():
        ns = [len(d["text"]) for d in docs]
        print(f"{dom}_{split:5s}: {len(docs):4d} docs | {sum(ns):6d} sentences | sent/doc p50 {pct(ns, .5)} max {max(ns)}")
        if "labels" in docs[0]:
            assert all(len(d["labels"]) == len(d["text"]) for d in docs), "label/sentence length mismatch"
            assert not ({l for d in docs for l in d["labels"]} - set(LABELS)), "unknown label in data"
    for dom in ("CL", "IT"):
        c = collections.Counter(l for s in ("train", "dev") for d in RAW[(dom, s)] for l in d["labels"])
        tot = sum(c.values())
        print(f"\n{dom} train+dev label share:", {l: f"{100 * c[l] / tot:.1f}%" for l in LABELS})

In [ ]:
def pct(xs, q): xs = sorted(xs); return xs[min(len(xs) - 1, int(q * len(xs)))]
if PHASE == 3:
    for split in ("train", "dev", "test"):
        sl = [len(tok.encode(" " + s.strip())) for dom in ("CL", "IT") for d in RAW[(dom, split)] for s in d["text"]]
        dl = [sum(len(tok.encode(" " + s.strip())) for s in d["text"]) for dom in ("CL", "IT") for d in RAW[(dom, split)]]
        print(f"{split:5s} tokens/sentence p50 {pct(sl,.5)} p90 {pct(sl,.9)} p95 {pct(sl,.95)} p99 {pct(sl,.99)} max {max(sl)} "
              f"| tokens/doc p50 {pct(dl,.5)} p90 {pct(dl,.9)} max {max(dl)}")

### 3.2 Sentence encoding + packing into encoder windows
Every sentence keeps a span `(window, start, end)`; whole sentences are packed into windows of at most `PACK_W` tokens, so a document is encoded with neighbouring sentences as context.

In [ ]:
if PHASE == 3:
    need("MAX_SENT_TOK", "PACK_W", "SENT_SEP_EOS")
    assert HP["MAX_SENT_TOK"] + 1 <= HP["PACK_W"] <= 1024

    SENT_CACHE = {}
    def sent_ids(s):
        if s not in SENT_CACHE:
            SENT_CACHE[s] = tok.encode(" " + s.strip())
        return SENT_CACHE[s]

    def pack_doc(sentences):
        windows, spans, cur, n_trunc, n_empty = [], [], [], 0, 0
        for s in sentences:
            ids = sent_ids(s)
            if not ids: ids, n_empty = [EOS_ID], n_empty + 1
            if len(ids) > HP["MAX_SENT_TOK"]: ids, n_trunc = ids[:HP["MAX_SENT_TOK"]], n_trunc + 1
            piece = ids + ([EOS_ID] if HP["SENT_SEP_EOS"] else [])
            if cur and len(cur) + len(piece) > HP["PACK_W"]:
                windows.append(cur); cur = []
            spans.append((len(windows), len(cur), len(cur) + len(ids)))
            cur = cur + piece
        if cur: windows.append(cur)
        return windows, spans, n_trunc, n_empty

    def build_items(dom, split):
        items, tr, em = [], 0, 0
        for d in RAW[(dom, split)]:
            w, sp, t_, e_ = pack_doc(d["text"]); tr += t_; em += e_
            items.append(dict(id=d["id"], domain=dom, windows=w, spans=sp,
                              labels=[L2I[l] for l in d["labels"]] if "labels" in d else None,
                              sent_idx=list(range(len(sp)))))
        return items, tr, em

    ITEMS = {}
    for dom in ("CL", "IT"):
        for split in ("train", "dev", "test"):
            ITEMS[(dom, split)], tr, em = build_items(dom, split)
            nw = [len(it["windows"]) for it in ITEMS[(dom, split)]]
            print(f"{dom}_{split:5s}: windows/doc p50 {sorted(nw)[len(nw)//2]} max {max(nw)} | "
                  f"truncated sentences {tr} | empty sentences {em}")
    TRAIN = ITEMS[("CL", "train")] + ITEMS[("IT", "train")]
    DEV   = ITEMS[("CL", "dev")]   + ITEMS[("IT", "dev")]
    TEST  = ITEMS[("CL", "test")]  + ITEMS[("IT", "test")]

    # check: spans point at exactly the (possibly truncated) sentence tokens
    it = TRAIN[0]
    ok = all(it["windows"][w][s:e] == sent_ids(it_s)[:e - s] for (w, s, e), it_s in
             zip(it["spans"], RAW[("CL", "train")][0]["text"]) if sent_ids(it_s))
    print("span check on first doc:", ok)

### 3.3 YOUR CODE — the Phase-3 model
**Contract the harness relies on:**
- the Phase-1 backbone is stored as `self.encoder` (so the freezing rule in `apply_technique` works);
- `forward(win_ids, win_lens, spans)` with `win_ids` LongTensor `(n_win, L)` (padded with EOS), `win_lens` LongTensor `(n_win,)`, `spans` = list of `(window, start, end)` per sentence, in document order;
- returns logits `(n_sentences, len(LABELS))`, one row per span, same order;
- get token states with `encoder_states(self.encoder, win_ids, win_lens)` → `(n_win, L, d_model)`.

In [ ]:
import torch
import torch.nn as nn

class RRModel(nn.Module):
    def __init__(self, encoder, d_model, n_labels, dropout=0.1, lstm_hidden=256):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)
        
        self.doc_lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        self.head = nn.Linear(2 * lstm_hidden, n_labels)

    def forward(self, win_ids, win_lens, spans):
        H = encoder_states(self.encoder, win_ids, win_lens)
        
        s_list = []
        for w_idx, start, end in spans:
            s_list.append(H[w_idx, start:end].mean(dim=0))
        S = torch.stack(s_list)
        
        S = S.unsqueeze(0)
        lstm_out, _ = self.doc_lstm(S)
        S = lstm_out.squeeze(0)
        
        S = self.dropout(S)
        logits = self.head(S)
        
        return logits

In [ ]:
def build_student():
    need("STUDENT_LAYERS")
    enc, d_model = load_backbone()
    
    enc.blocks = torch.nn.ModuleList(list(enc.blocks)[:HP["STUDENT_LAYERS"]])
    
    m = RRModel(enc, d_model, len(LABELS))
    
    for param in m.parameters():
        param.requires_grad = True
        
    return m.to(device)

def kd_loss(logits, y, cw, teacher=None):
    hard_loss = torch.nn.functional.cross_entropy(logits, y, weight=cw)
    
    if teacher is None:
        return hard_loss
        
    T = HP["KD_T"]
    alpha = HP["KD_ALPHA"]
    
    student_log_probs = torch.nn.functional.log_softmax(logits / T, dim=-1)
    teacher_probs = torch.nn.functional.softmax(teacher / T, dim=-1)
    
    soft_loss = torch.nn.functional.kl_div(
        student_log_probs, 
        teacher_probs, 
        reduction='batchmean'
    ) * (T ** 2)
    
    return (1.0 - alpha) * hard_loss + alpha * soft_loss

### 3.4 Harness (training, evaluation, submission)

In [ ]:
def weighted_ce(logits, y, cw, teacher=None):
    # per-item loss x weight of its own class, then mean. (F.cross_entropy(weight=cw) divides by the batch's
    # weight sum, which cancels the weight completely when a batch holds ONE item -- the Phase-4 case.)
    return (F.cross_entropy(logits, y, reduction="none") * cw[y]).mean()

def crop(item, max_win):
    """Training-time crop to <= max_win consecutive windows (whole sentences only)."""
    n = len(item["windows"])
    if max_win is None or n <= max_win:
        return item
    w0 = random.randint(0, n - max_win)
    keep = [i for i, (w, _, _) in enumerate(item["spans"]) if w0 <= w < w0 + max_win]
    return dict(item, windows=item["windows"][w0:w0 + max_win],
                spans=[(item["spans"][i][0] - w0, item["spans"][i][1], item["spans"][i][2]) for i in keep],
                labels=[item["labels"][i] for i in keep] if item["labels"] is not None else None,
                sent_idx=[item["sent_idx"][i] for i in keep])

def to_tensors(item):
    L = max(len(w) for w in item["windows"])
    x = torch.full((len(item["windows"]), L), EOS_ID, dtype=torch.long)
    for i, w in enumerate(item["windows"]):
        x[i, :len(w)] = torch.tensor(w)
    lens = torch.tensor([len(w) for w in item["windows"]])
    return x.to(device), lens.to(device)

def class_weights_rr(items, power):
    cnt = torch.zeros(len(LABELS))
    for it in items:
        cnt += torch.bincount(torch.tensor(it["labels"]), minlength=len(LABELS)).float()
    w = (cnt.sum() / cnt.clamp(min=1)) ** power
    return (w / w.mean()).to(device)

from sklearn.metrics import f1_score
def macro_f1(gold, pred):
    """Official Phase-3 metric (same as evaluate_phase3.py): macro over ALL 13 labels, zero_division=0."""
    g = [LABELS[x] for d in gold for x in d]
    p = [LABELS[x] for d in pred for x in d]
    per = f1_score(g, p, labels=LABELS, average=None, zero_division=0)
    return float(per.mean()), {l: float(v) for l, v in zip(LABELS, per)}

@torch.no_grad()
def predict_rr(model, items):
    model.eval(); out = {}
    for it in items:
        x, lens = to_tensors(it)
        with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
            lg = model(x, lens, it["spans"])
        out[it["id"]] = lg.float().cpu()
    model.train()
    return out                          # doc id -> (n_sent, n_labels) logits

def evaluate(items, logits_by_id, show_per_class=False):
    res = {}
    for dom in ("CL", "IT", "ALL"):
        sub = [it for it in items if dom == "ALL" or it["domain"] == dom]
        if not sub: continue
        f, per = macro_f1([it["labels"] for it in sub], [logits_by_id[it["id"]].argmax(-1).tolist() for it in sub])
        res[dom] = f
        if show_per_class and dom == "ALL":
            for l, v in per.items(): print(f"    {l:24s} {v:.3f}")
    return res

def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

def train_rr(model, train_items, dev_items, loss_fn=weighted_ce, teacher=None, *, EPOCHS, LR_ENC, LR_HEAD,
             WARMUP_FRAC, WD, ACCUM_DOCS, CLASS_W_POWER, MAX_TRAIN_WIN, CLIP=1.0, seed=0):
    """Per-document training with gradient accumulation. teacher: doc id -> (n_sent, n_labels) logits (KD runs).
    With dev_items: keeps the best-dev trainable weights. Without: keeps the last epoch."""
    torch.manual_seed(seed); random.seed(seed)
    train_items = list(train_items)
    cw = class_weights_rr(train_items, CLASS_W_POWER)
    enc_p  = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("encoder.")]
    head_p = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("encoder.")]
    groups = [g for g in ({"params": enc_p, "lr": LR_ENC}, {"params": head_p, "lr": LR_HEAD}) if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=WD)
    steps_ep = math.ceil(len(train_items) / ACCUM_DOCS); total = EPOCHS * steps_ep
    warm = max(1, int(WARMUP_FRAC * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    scaler = torch.amp.GradScaler(DEV_TYPE, enabled=(DEV_TYPE == "cuda"))
    params = [p for g in groups for p in g["params"]]
    best_f, best_state, hist = -1.0, None, []
    model.train()
    for ep in range(EPOCHS):
        random.shuffle(train_items); t0 = time.time(); tot = 0.0
        for i, item in enumerate(train_items):
            it = crop(item, MAX_TRAIN_WIN)
            x, lens = to_tensors(it)
            y = torch.tensor(it["labels"], device=device)
            tch = teacher[it["id"]][it["sent_idx"]].to(device) if teacher is not None else None
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                lg = model(x, lens, it["spans"])
            loss = loss_fn(lg.float(), y, cw, teacher=tch) / ACCUM_DOCS
            scaler.scale(loss).backward(); tot += loss.item() * ACCUM_DOCS
            if (i + 1) % ACCUM_DOCS == 0 or i + 1 == len(train_items):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        msg = f"epoch {ep + 1}/{EPOCHS} | train loss {tot / len(train_items):.4f} | {time.time() - t0:.0f}s"
        if dev_items:
            res = evaluate(dev_items, predict_rr(model, dev_items))
            msg += " | dev macro-F1 " + " ".join(f"{k} {v:.4f}" for k, v in res.items())
            hist.append(res)
            if res["ALL"] > best_f:
                best_f, best_state = res["ALL"], trainable_state(model)
        print(msg)
    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
        print("BEST dev macro-F1 (ALL):", round(best_f, 4))
    return best_f, hist

def write_submission(logits_by_id, tag):
    pred = {}
    for it in TEST:
        lab = [LABELS[i] for i in logits_by_id[it["id"]].argmax(-1).tolist()]
        assert len(lab) == len(it["spans"]), f"length mismatch for {it['id']}"
        pred[it["id"]] = lab
    want = {d["id"]: len(d["text"]) for dom in ("CL", "IT") for d in RAW[(dom, "test")]}
    assert set(pred) == set(want) and all(len(pred[k]) == want[k] for k in want), "ids / sentence counts do not match the test files"
    assert all(l in L2I for v in pred.values() for l in v)
    jp, zp = f"/kaggle/working/prediction_{tag}.json", f"/kaggle/working/submission_{tag}.zip"
    json.dump(pred, open(jp, "w"))
    with zipfile.ZipFile(zp, "w") as z:
        z.write(jp, arcname="prediction.json")          # must sit at the zip root
    print(f"wrote {zp} | docs {len(pred)} | sentences {sum(map(len, pred.values()))}")

In [ ]:
def focal_loss(logits, y, cw, teacher=None):
    log_pt = torch.nn.functional.log_softmax(logits, dim=-1)
    log_p = log_pt.gather(1, y.unsqueeze(1)).squeeze(1)
    p = torch.exp(log_p)
    focal_factor = (1 - p) ** HP["FOCAL_GAMMA"]
    per_sentence_loss = -cw[y] * focal_factor * log_p
    return per_sentence_loss.mean()

def train_loss():
    if LOSS == "focal":
        need("FOCAL_GAMMA")
        return focal_loss
    return weighted_ce

### 3.5 Run (uses the CONTROL cell)

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    for n in dirs + files:
        if "arch3" in n or "mlm" in n:
            p = os.path.join(root, n)
            print("DIR " if os.path.isdir(p) else "FILE", p, os.path.getsize(p) if os.path.isfile(p) else "")

In [ ]:
if PHASE == 3:
    need("EPOCHS", "LR_ENC", "LR_HEAD", "WARMUP_FRAC", "WD", "ACCUM_DOCS", "CLASS_W_POWER", "MAX_TRAIN_WIN")
    settings = {k: HP[k] for k in ("EPOCHS", "LR_ENC", "LR_HEAD", "WARMUP_FRAC", "WD", "ACCUM_DOCS",
                                   "CLASS_W_POWER", "MAX_TRAIN_WIN")}
    teacher, loss_fn = None, train_loss()
    if TECHNIQUE == "kd":
        assert TEACHER_TAG, "set TEACHER_TAG in the CONTROL cell and attach that run's teacher-logits file"
        teacher = torch.load(find_input(f"teacher_logits_{TEACHER_TAG}_{RUN_MODE}.pt"))
        model, loss_fn = build_student().to(device), kd_loss
        trainable_report(model, "KD student")
    else:
        model = build_rr_model()

    if RUN_MODE == "dev":
        train_items, dev_items = TRAIN, DEV
    else:
        need("FINAL_EPOCHS"); settings["EPOCHS"] = HP["FINAL_EPOCHS"]
        train_items, dev_items = TRAIN + DEV, None

    t_start = time.time()
    best_f, hist = train_rr(model, train_items, dev_items, loss_fn, teacher, seed=SEED, **settings)
    print(f"training time {(time.time() - t_start) / 60:.1f} min")
    torch.save(trainable_state(model), f"/kaggle/working/weights_{RUN_TAG}.pt")   # trainable params only

    if RUN_MODE == "dev":
        dev_logits = predict_rr(model, DEV)
        print("DEV:", evaluate(DEV, dev_logits, show_per_class=True))
        torch.save(dev_logits, f"/kaggle/working/devlogits_{RUN_TAG}.pt")          # for ensembling later
        test_logits = predict_rr(model, TEST)                                         # submit straight from the best-dev model
        torch.save(test_logits, f"/kaggle/working/testlogits_{RUN_TAG}.pt")
        write_submission(test_logits, RUN_TAG)
        json.dump(hist, open(f"/kaggle/working/hist_{RUN_TAG}.json", "w"))
    else:
        test_logits = predict_rr(model, TEST)
        torch.save(test_logits, f"/kaggle/working/testlogits_{RUN_TAG}.pt")        # for ensembling later
        write_submission(test_logits, RUN_TAG)

    if CACHE_TEACHER_LOGITS:                       # this run will serve as a KD teacher
        tl = predict_rr(model, train_items)
        torch.save(tl, f"/kaggle/working/teacher_logits_{ARCH}_{TECHNIQUE}_{RUN_MODE}.pt")
        print("saved teacher logits for", len(tl), "training docs")

### 3.6 Stage 2 — document layer on full documents
Run right after 3.5 (same session). Writes `submission_<tag>_s2.zip` plus dev/test logits for ensembling.

In [ ]:
# ===== 3.6 Stage 2: train doc_lstm + head on FULL documents (plumbing; your layers, unchanged) =====
# Run right after 3.5 in the same session: it reuses `model`. Encoder + LoRA are frozen; sentence vectors are
# computed once per document (same mean-over-span pooling as RRModel.forward), then only your doc_lstm + head train,
# on whole-document sequences (no MAX_TRAIN_WIN crop).
RUN_STAGE2 = True
if PHASE == 3 and RUN_STAGE2:
    need("STAGE2_EPOCHS", "STAGE2_LR")
    assert all(hasattr(model, a) for a in ("doc_lstm", "head", "dropout")), "needs your BiLSTM RRModel"

    @torch.no_grad()
    def sentence_vectors(m, items, chunk=8):
        m.eval(); out = {}
        for it in items:
            x, lens = to_tensors(it)
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                H = torch.cat([encoder_states(m.encoder, x[i:i + chunk], lens[i:i + chunk])
                               for i in range(0, x.size(0), chunk)])
            out[it["id"]] = torch.stack([H[w, s:e].float().mean(0) for w, s, e in it["spans"]]).cpu()
        return out

    def doc_logits(m, S):                         # = your RRModel.forward after pooling
        out, _ = m.doc_lstm(S.unsqueeze(0).to(device))
        return m.head(m.dropout(out.squeeze(0)))

    s2_train = list(TRAIN) if RUN_MODE == "dev" else list(TRAIN + DEV)
    SV = sentence_vectors(model, s2_train + DEV + TEST)
    print("sentence vectors cached for", len(SV), "docs")

    for p in model.parameters(): p.requires_grad = False
    doc_params = list(model.doc_lstm.parameters()) + list(model.head.parameters())
    for p in doc_params: p.requires_grad = True
    opt2 = torch.optim.AdamW(doc_params, lr=HP["STAGE2_LR"], weight_decay=HP["WD"])
    cw2 = class_weights_rr(s2_train, HP["CLASS_W_POWER"])

    @torch.no_grad()
    def s2_predict(items):
        model.eval(); out = {it["id"]: doc_logits(model, SV[it["id"]]).float().cpu() for it in items}; model.train()
        return out

    best2, state2 = -1.0, None
    for ep in range(HP["STAGE2_EPOCHS"]):
        random.shuffle(s2_train); model.train(); tot = 0.0
        for it in s2_train:
            lg = doc_logits(model, SV[it["id"]])
            loss = train_loss()(lg.float(), torch.tensor(it["labels"], device=device), cw2)
            opt2.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(doc_params, 1.0); opt2.step(); tot += loss.item()
        res = evaluate(DEV, s2_predict(DEV))
        print(f"stage2 epoch {ep + 1} | loss {tot / len(s2_train):.4f} | dev " +
              " ".join(f"{k} {v:.4f}" for k, v in res.items()) + ("" if RUN_MODE == "dev" else "  (dev is in training data)"))
        if RUN_MODE == "dev" and res["ALL"] > best2:
            best2, state2 = res["ALL"], {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
    if state2 is not None:
        model.load_state_dict(state2, strict=False); print("stage2 BEST dev macro-F1:", round(best2, 4))

    print("DEV:", evaluate(DEV, s2_predict(DEV), show_per_class=True))
    tl2 = s2_predict(TEST)
    torch.save(s2_predict(DEV), f"/kaggle/working/devlogits_{RUN_TAG}_s2.pt")
    torch.save(tl2, f"/kaggle/working/testlogits_{RUN_TAG}_s2.pt")
    write_submission(tl2, RUN_TAG + "_s2")


### 3.6b Stage-2 comparison table
List (loss, CLASS_W_POWER) combos; one run prints the official dev score for each and saves their logits for 3.7 / 3.8.

In [ ]:
# ===== 3.6b Stage-2 comparison table (plumbing): one run tries every combo YOU list, prints the official dev score =====
# Needs 3.5 and 3.6 run first in this session (uses their cached sentence vectors). Each combo restarts the doc layer
# from the same post-3.5 weights, so results are comparable. Nothing is chosen automatically: you read the table and pick.
SWEEP = [("ce", 0.2), ("focal", 0.2)]   # YOUR combos as (loss, CLASS_W_POWER), e.g. [("ce", 0.2), ("focal", 0.2)]
if PHASE == 3 and SWEEP:
    s1 = torch.load(f"/kaggle/working/weights_{RUN_TAG}.pt")
    doc_s1 = {k: v for k, v in s1.items() if k.startswith(("doc_lstm.", "head."))}
    loss_fns = {"ce": weighted_ce, "focal": focal_loss}
    table = []
    for loss_name, cwp in SWEEP:
        if loss_name == "focal": need("FOCAL_GAMMA")
        model.load_state_dict(doc_s1, strict=False)
        for p in model.parameters(): p.requires_grad = False
        dps = list(model.doc_lstm.parameters()) + list(model.head.parameters())
        for p in dps: p.requires_grad = True
        opt_s = torch.optim.AdamW(dps, lr=HP["STAGE2_LR"], weight_decay=HP["WD"])
        cw_s = class_weights_rr(s2_train, cwp)
        best_s, state_s = -1.0, None
        for ep in range(HP["STAGE2_EPOCHS"]):
            random.shuffle(s2_train); model.train()
            for it in s2_train:
                lg = doc_logits(model, SV[it["id"]])
                loss = loss_fns[loss_name](lg.float(), torch.tensor(it["labels"], device=device), cw_s)
                opt_s.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(dps, 1.0); opt_s.step()
            f = evaluate(DEV, s2_predict(DEV))["ALL"]
            if f > best_s:
                best_s, state_s = f, {n: q.detach().cpu().clone() for n, q in model.named_parameters() if q.requires_grad}
        model.load_state_dict(state_s, strict=False)
        tag = f"{RUN_TAG}_s2_{loss_name}_cw{cwp}"
        torch.save(s2_predict(DEV),  f"/kaggle/working/devlogits_{tag}.pt")
        torch.save(s2_predict(TEST), f"/kaggle/working/testlogits_{tag}.pt")
        table.append((loss_name, cwp, best_s))
        print(f"{loss_name:6s} CLASS_W_POWER {cwp}: best dev macro-F1 (official) {best_s:.4f}  -> devlogits/testlogits_{tag}.pt")
    print("\nsummary:"); [print(f"  {l:6s} cw {c_}: {f:.4f}") for l, c_, f in table]
    # to submit one: put its testlogits/devlogits file names into 3.7 (single file is fine) or 3.8 (with TAU)


### 3.7 Ensemble
List `testlogits_*.pt` (and optionally `devlogits_*.pt`) files from your runs; writes `submission_p3_ensemble.zip`.

In [ ]:
# ===== 3.7 Ensemble: average probabilities of several runs (plumbing, equal weights) =====
ENS_TEST = []        # e.g. ["testlogits_p3_arch3_lora_dev_s2.pt", "testlogits_p3_arch2_lorma_dev_s2.pt"]
ENS_DEV  = []        # matching devlogits_*.pt files (optional: scores the ensemble on dev)

def _load_logits(name):
    for p in (name, f"/kaggle/working/{name}"):
        if os.path.exists(p): return torch.load(p)
    return torch.load(find_input(name))

def _avg_probs(names):
    acc = None
    for n in names:
        P_ = {k: torch.softmax(v.float(), -1) for k, v in _load_logits(n).items()}
        acc = P_ if acc is None else {k: acc[k] + P_[k] for k in acc}
    return {k: v / len(names) for k, v in acc.items()}

if PHASE == 3 and ENS_DEV:
    print("ENSEMBLE DEV:", evaluate(DEV, _avg_probs(ENS_DEV), show_per_class=True))
if PHASE == 3 and ENS_TEST:
    write_submission(_avg_probs(ENS_TEST), "p3_ensemble")


### 3.8 Logit adjustment (post-processing for rare labels)
Runs on saved logits in seconds; compare TAU values on dev, then set TAU to write a submission.

In [ ]:
# ===== 3.8 Logit adjustment (post-processing; Menon et al. 2021): score_c = log p_c - TAU * log prior_c =====
# Works on saved logits: one file = single model, several files = ensemble average (same as 3.7).
# prior_c = label frequency in TRAIN. TAU = 0 changes nothing; TAU > 0 shifts predictions toward rarer labels.
LA_DEV    = []      # e.g. ["devlogits_p3_arch3_lora_dev_s2.pt"]
LA_TEST   = []      # matching testlogits files, same order
TAU_SWEEP = []      # YOUR candidate values to compare on dev (official metric)
TAU       = None    # YOUR choice after looking at the sweep -> writes the submission

if PHASE == 3 and LA_DEV:
    cnt = torch.zeros(len(LABELS))
    for it in TRAIN:
        cnt += torch.bincount(torch.tensor(it["labels"]), minlength=len(LABELS)).float()
    log_prior = torch.log((cnt + 1) / (cnt + 1).sum())

    def adjust(probs_by_id, tau):
        return {k: torch.log(v.clamp_min(1e-12)) - tau * log_prior for k, v in probs_by_id.items()}

    dev_p = _avg_probs(LA_DEV)
    print("no adjustment:", evaluate(DEV, adjust(dev_p, 0.0)))
    for tau in TAU_SWEEP:
        print(f"tau {tau}: dev", evaluate(DEV, adjust(dev_p, tau)))
    if TAU is not None and LA_TEST:
        print(f"chosen tau {TAU}: dev", evaluate(DEV, adjust(dev_p, TAU), show_per_class=True))
        write_submission(adjust(_avg_probs(LA_TEST), TAU), f"p3_la_tau{TAU}")


# PHASE 4 — Court Judgment Prediction
Run order: 4.1 → 4.5. Per account, edit only the CONTROL cell (`ARCH`, `TECHNIQUE`, `BIDIR`, `KEEP`, `TIME_BUDGET_H`, `EPOCHS`, `CACHE_TEACHER_LOGITS`, `TEACHER_TAG`).

### 4.1 Data + checks
Loads train/dev/test, reports duplicates, stops if any case is shared with test (leak check), prints label balance and lengths.

In [ ]:
if PHASE == 4:
    import hashlib

    def read_json_any(path):
        try:
            return json.load(open(path))
        except json.JSONDecodeError:                                   # JSON-lines fallback
            return [json.loads(l) for l in open(path) if l.strip()]

    def to_records(obj, name):
        if isinstance(obj, dict):
            vals = list(obj.values())
            if vals and all(isinstance(v, dict) for v in vals):        # {case_id: {text, label, ...}}
                obj = [dict(id=k, **v) for k, v in obj.items()]
            elif any(isinstance(v, list) and v and isinstance(v[0], dict) for v in vals):
                obj = next(v for v in vals if isinstance(v, list) and v and isinstance(v[0], dict))  # {"data": [...]}
            elif "id" in obj and isinstance(obj["id"], list):          # columnar {id: [...], text: [...], label: [...]}
                n = len(obj["id"])
                obj = [{k: obj[k][i] for k in obj if isinstance(obj[k], list) and len(obj[k]) == n} for i in range(n)]
            else:
                raise ValueError(f"{name}: unknown layout, top keys {list(obj)[:5]}")
        assert isinstance(obj, list) and obj and isinstance(obj[0], dict), f"{name}: expected a list of records"
        return obj

    def pick(keys, cands, what, required=True):
        for c in cands:
            if c in keys: return c
        if required: raise KeyError(f"no {what} field among {sorted(keys)}")
        return None

    def to_label(v):
        if isinstance(v, list): v = v[0]
        v = int(float(v))
        assert v in (0, 1), f"non-binary label {v}"
        return v

    # ---- load the three splits ----
    RAW4 = {}
    for split, key in (("train", "Train/train.json"), ("dev", "Train/dev.json"), ("test", "Test/test.json")):
        recs = to_records(read_json_any(P[key]), key)
        keys = set(recs[0].keys())
        ID_K   = pick(keys, ("id", "case_id", "doc_id", "name"), "id")
        TEXT_K = pick(keys, ("text", "case_text", "document", "judgment"), "text")
        LAB_K  = pick(keys, ("label", "labels", "outcome", "decision", "pred"), "label", required=(split != "test"))
        out = []
        for r in recs:
            t = r[TEXT_K]
            if isinstance(t, list): t = " ".join(map(str, t))           # sentence-list form -> one string
            d = {"id": r[ID_K], "text": t}                                # id kept with its ORIGINAL type for the writer
            if LAB_K and split != "test": d["label"] = to_label(r[LAB_K])
            out.append(d)
        RAW4[split] = out
        extra = sorted(keys - {ID_K, TEXT_K, LAB_K})
        print(f"{split:5s}: {len(out):6d} cases | id='{ID_K}' text='{TEXT_K}' label='{LAB_K}' | other fields (not used): {extra}")

    # ---- integrity checks: duplicates inside a split are reported; anything shared with test stops the cell ----
    norm = lambda s: re.sub(r"\s+", " ", s).strip().lower()
    md5  = lambda s: hashlib.md5(norm(s).encode()).hexdigest()
    IDS  = {s: [str(d["id"]) for d in RAW4[s]] for s in RAW4}
    HASH = {s: {md5(d["text"]) for d in RAW4[s]} for s in RAW4}
    for s in RAW4:
        dup = [k for k, n in collections.Counter(IDS[s]).items() if n > 1]
        print(f"{s:5s}: unique ids {len(set(IDS[s]))} / {len(IDS[s])} | duplicated ids {len(dup)} | unique texts {len(HASH[s])}")
        if dup:
            by = collections.defaultdict(list)
            for d in RAW4[s]: by[str(d["id"])].append(d)
            same_t = sum(len({md5(x["text"]) for x in by[k]}) == 1 for k in dup)
            same_l = sum(len({x.get("label") for x in by[k]}) == 1 for k in dup)
            print(f"   of the {len(dup)} duplicated ids: identical text {same_t} | same label {same_l}")
    assert len(IDS["test"]) == len(set(IDS["test"])), "duplicate ids inside test -- one prediction per id would be ambiguous"
    for a, b in (("train", "dev"), ("train", "test"), ("dev", "test")):
        ni, nt = len(set(IDS[a]) & set(IDS[b])), len(HASH[a] & HASH[b])
        print(f"overlap {a}-{b}: ids {ni} | identical texts {nt}")
        if b == "test":
            assert ni == 0 and nt == 0, f"{a} shares cases with test -- old (leaked) data? re-download"

    # ---- label balance + length statistics ----
    def pct(xs, q): xs = sorted(xs); return xs[min(len(xs) - 1, int(q * len(xs)))]
    for s in ("train", "dev"):
        c = collections.Counter(d["label"] for d in RAW4[s])
        print(f"{s:5s} labels: 0 -> {c[0]} ({c[0]/len(RAW4[s]):.1%}) | 1 -> {c[1]} ({c[1]/len(RAW4[s]):.1%})")
    for s in RAW4:
        wl = [len(d["text"].split()) for d in RAW4[s]]
        print(f"{s:5s} words/case p50 {pct(wl,.5)} p90 {pct(wl,.9)} p99 {pct(wl,.99)} max {max(wl)} | total {sum(wl)/1e6:.1f}M words")

    # ---- tokenizer cost on a sample (your BPE is pure Python, so full-text encoding of every case is the slow step) ----
    rng = random.Random(SEED)
    sample = rng.sample(RAW4["train"], min(100, len(RAW4["train"])))
    t0 = time.time()
    lens = [len(tok.encode(d["text"])) for d in sample]
    dt = time.time() - t0
    ntok = sum(lens)
    nw = sum(len(d["text"].split()) for d in sample)
    tpw = ntok / nw
    total_w = sum(len(d["text"].split()) for s in RAW4 for d in RAW4[s])
    print(f"sample of {len(sample)}: {tpw:.2f} tokens/word | {ntok/dt:,.0f} tokens/s encode speed")
    print(f"tokens/case (sample): p50 {pct(lens,.5)} p90 {pct(lens,.9)} max {max(lens)}")
    print(f"projected full-text tokens: {total_w*tpw/1e6:.0f}M | projected encode time for ALL splits: {total_w*tpw/(ntok/dt)/3600:.1f} h")

    ex = RAW4["train"][0]

### 4.2 De-duplicate, tokenize once (cached), chunking
Your settings: `DEDUP`, `CHUNK_LEN`, `OVERLAP`, `MAX_CHUNKS`, `KEEP`. Prints coverage and hours per epoch.

In [ ]:
if PHASE == 4:
    # ---- 4.2  tokenize every case ONCE, cache to a file, and define the chunking (plumbing) ----
    # Your choices (CONTROL / HP): CHUNK_LEN, OVERLAP, MAX_CHUNKS, KEEP, DEDUP.  The cache stores FULL token
    # sequences, so changing MAX_CHUNKS / KEEP later needs no re-tokenization; changing CHUNK_LEN/OVERLAP doesn't either.
    CACHE_NAME = "p4_tokens.pt"

    # 1) de-duplicate (by id; all repeated ids were verified identical in text and label in 4.1)
    def dedup(recs):
        seen, out = set(), []
        for d in recs:
            k = str(d["id"])
            if k not in seen: seen.add(k); out.append(d)
        return out
    DATA4 = {s: (dedup(RAW4[s]) if HP["DEDUP"] else list(RAW4[s])) for s in RAW4}
    for s in DATA4: print(f"{s:5s}: {len(RAW4[s])} rows -> {len(DATA4[s])} cases")
    by_text = collections.defaultdict(list)                       # report only: same text under different ids
    for d in DATA4["train"]: by_text[md5(d["text"])].append(d["label"])
    shared = [v for v in by_text.values() if len(v) > 1]
    print(f"train texts shared by >1 id: {len(shared)} | of these with conflicting labels: {sum(len(set(v)) > 1 for v in shared)}  (kept as-is)")

    # 2) tokenize once (or load the cache if it is attached from another account)
    hits = glob.glob(f"/kaggle/input/**/{CACHE_NAME}", recursive=True) + glob.glob(f"/kaggle/working/{CACHE_NAME}")
    TOK4 = None
    if hits:
        TOK4 = torch.load(hits[0], weights_only=False)
        if TOK4["cfg"]["DEDUP"] != HP["DEDUP"] or any(TOK4["ids"][s] != [str(d["id"]) for d in DATA4[s]] for s in DATA4):
            print("cache does not match the current data/DEDUP setting -> re-tokenizing"); TOK4 = None
        else:
            print("loaded token cache:", hits[0])
    if TOK4 is None:
        TOK4 = {"cfg": {"DEDUP": HP["DEDUP"]}, "ids": {}, "labels": {}, "offsets": {}, "tokens": {}}
        for s in DATA4:
            t0 = time.time(); seqs = []
            for i, d in enumerate(DATA4[s]):
                seqs.append(np.asarray(tok.encode(d["text"]), dtype=np.uint16))     # vocab 65,355 < 65,536
                if i % 5000 == 0: print(f"  {s}: {i}/{len(DATA4[s])}  {time.time()-t0:.0f}s")
            TOK4["ids"][s]     = [str(d["id"]) for d in DATA4[s]]
            TOK4["labels"][s]  = [d.get("label", -1) for d in DATA4[s]]
            TOK4["offsets"][s] = np.concatenate([[0], np.cumsum([len(x) for x in seqs])]).astype(np.int64)
            TOK4["tokens"][s]  = np.concatenate(seqs)
            print(f"{s:5s}: {len(seqs)} cases, {len(TOK4['tokens'][s])/1e6:.1f}M tokens, {time.time()-t0:.0f}s")
        torch.save(TOK4, f"/kaggle/working/{CACHE_NAME}")
        print(f"saved /kaggle/working/{CACHE_NAME} ({os.path.getsize(f'/kaggle/working/{CACHE_NAME}')/1e6:.0f} MB)"
              " -> make it a dataset and attach it on the other accounts to skip this step")
    ORIG_ID = {s: {str(d["id"]): d["id"] for d in DATA4[s]} for s in DATA4}   # original id type, for the writer

    # 3) chunking
    def doc_tokens(split, i):
        o = TOK4["offsets"][split]
        return TOK4["tokens"][split][o[i]:o[i + 1]]

    def chunk_starts(n, L, stride):
        """start offsets of overlapping windows covering all n tokens; the last window is aligned to the end"""
        if n <= L: return [0]
        st = list(range(0, n - L + 1, stride))
        if st[-1] + L < n: st.append(n - L)
        return st

    def select_chunks(st, max_chunks, keep):
        if len(st) <= max_chunks: return st
        if keep == "first": return st[:max_chunks]
        if keep == "last":  return st[-max_chunks:]
        if keep == "both":
            a = (max_chunks + 1) // 2
            return st[:a] + st[len(st) - (max_chunks - a):]
        raise ValueError("HP['KEEP'] must be 'first', 'last' or 'both'")

    def doc_chunks(split, i):
        """list of int64 tensors (each <= CHUNK_LEN tokens) for case i of a split, after the cap"""
        x = doc_tokens(split, i)
        L = HP["CHUNK_LEN"]; stride = L - int(round(L * HP["OVERLAP"]))
        st = select_chunks(chunk_starts(len(x), L, stride), HP["MAX_CHUNKS"], HP["KEEP"])
        return [torch.from_numpy(x[a:a + L].astype(np.int64)) for a in st]

    # 4) what the settings imply: coverage, tokens per epoch, time at your measured T4 speed (~5k tok/s, Phase-1 arch 1)
    L = HP["CHUNK_LEN"]; stride = L - int(round(L * HP["OVERLAP"]))
    print(f"\nCHUNK_LEN {L} | overlap {L - stride} tokens | stride {stride} | MAX_CHUNKS {HP['MAX_CHUNKS']} "
          f"-> a case is fully covered up to {L + (HP['MAX_CHUNKS'] - 1) * stride} tokens")
    for s in DATA4:
        o = TOK4["offsets"][s]; n = np.diff(o)
        nch = np.array([len(chunk_starts(int(k), L, stride)) for k in n])
        cap = np.minimum(nch, HP["MAX_CHUNKS"])
        used = sum(min(int(k), L) if c == 1 else c * L for k, c in zip(n, cap))
        print(f"{s:5s}: chunks/case p50 {int(np.median(nch))} p90 {int(np.percentile(nch, 90))} max {nch.max()} | "
              f"cases over the cap {np.mean(nch > HP['MAX_CHUNKS']):.1%} | tokens through the encoder {used/1e6:.1f}M")
        if s == "train": train_used = used
        else: globals()[f"used_{s}"] = used
    SPEED = 5000
    print(f"\ntrain epoch, fine-tuning (fwd+bwd) at ~{SPEED} tok/s: {train_used/SPEED/3600:.1f} h for all train cases "
          f"(scales linearly with the fraction of train cases used)")
    print(f"dev+test prediction (forward only, roughly 3x faster -- an assumption): {(used_dev + used_test)/(3*SPEED)/3600:.1f} h")
    print(f"frozen-encoder caching of train+dev+test (forward only): {(train_used + used_dev + used_test)/(3*SPEED)/3600:.1f} h")

### 4.3 YOUR CODE — the Phase-4 model
**Contract the harness relies on:** backbone stored as `self.encoder`; `__init__(encoder, d_model)`; `forward(win_ids, win_lens)` with `win_ids` `(n_chunks, L)` padded with EOS and `win_lens` `(n_chunks,)`; returns logits `(1, 2)`; token states via `encoder_states(self.encoder, win_ids, win_lens)`.

In [ ]:
if PHASE == 4:
    class CJPEModel(nn.Module):
        def __init__(self, encoder, d_model):
            super().__init__()
            self.encoder = encoder
            self.doc_rnn = nn.GRU(d_model, d_model // 2, bidirectional=True, batch_first=True)
            self.classifier = nn.Linear(d_model, 2)

        def forward(self, win_ids, win_lens):
            H = encoder_states(self.encoder, win_ids, win_lens)
            
            mask = torch.arange(H.size(1), device=H.device).unsqueeze(0) < win_lens.unsqueeze(1)
            H = H * mask.unsqueeze(2).float()
            chunk_reps = H.sum(dim=1) / win_lens.unsqueeze(1).clamp(min=1).float()
            
            chunk_reps = chunk_reps.unsqueeze(0)
            doc_out, _ = self.doc_rnn(chunk_reps)
            doc_rep = doc_out.mean(dim=1)
            
            logits = self.classifier(doc_rep)
            return logits

    def build_cjpe_model():
        enc, d_model = load_backbone()
        return apply_technique(CJPEModel(enc, d_model)).to(device)

    def build_student4():
        enc, d_model = load_backbone()
        
        n_before = len(enc.blocks)
        enc.blocks = torch.nn.ModuleList(list(enc.blocks)[:HP["STUDENT_LAYERS"]])
        assert len(enc.blocks) == HP["STUDENT_LAYERS"] < n_before, "student was not truncated"
        print(f"KD student: kept {len(enc.blocks)} of {n_before} blocks")
            
        model = CJPEModel(enc, d_model)
        
        for param in model.parameters():
            param.requires_grad = True
            
        return model.to(device)

### 4.4 Harness (training, evaluation, submission)
Speed check after 50 cases, time-budget stop, best-dev weights saved on every improvement, OOM cases skipped and counted.

In [ ]:
if PHASE == 4:
    # ===================== 4.4  Harness: data -> tensors, training, evaluation, submission (plumbing) =====================
    from sklearn.metrics import f1_score, precision_score, recall_score

    def p4_items(split, frac=None, seed=0):
        """indices of the cases of a split (optionally a random fraction, train only)"""
        idx = list(range(len(TOK4["ids"][split])))
        if frac is not None and frac < 1.0:
            idx = sorted(random.Random(seed).sample(idx, int(round(frac * len(idx)))))
        return idx

    def p4_tensors(split, i):
        ch = doc_chunks(split, i)
        L = max(len(c) for c in ch)
        x = torch.full((len(ch), L), EOS_ID, dtype=torch.long)
        for k, c in enumerate(ch): x[k, :len(c)] = c
        return x.to(device), torch.tensor([len(c) for c in ch]).to(device)

    def p4_label(split, i): return int(TOK4["labels"][split][i])
    def p4_id(split, i):    return TOK4["ids"][split][i]

    def class_weights4(split, idx, power):
        cnt = torch.bincount(torch.tensor([p4_label(split, i) for i in idx]), minlength=2).float()
        w = (cnt.sum() / cnt.clamp(min=1)) ** power
        return (w / w.mean()).to(device)

    def metrics4(gold, pred):
        """Macro-F1 (primary) + macro precision / recall over the two classes."""
        return dict(f1=float(f1_score(gold, pred, labels=[0, 1], average="macro", zero_division=0)),
                    p=float(precision_score(gold, pred, labels=[0, 1], average="macro", zero_division=0)),
                    r=float(recall_score(gold, pred, labels=[0, 1], average="macro", zero_division=0)))

    @torch.no_grad()
    def predict4(model, split, idx):
        model.eval(); out = {}
        for i in idx:
            x, lens = p4_tensors(split, i)
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                lg = model(x, lens)
            out[p4_id(split, i)] = lg.float().reshape(1, 2).cpu()
        model.train()
        return out                                          # case id -> (1, 2) logits

    def evaluate4(split, idx, logits_by_id):
        gold = [p4_label(split, i) for i in idx]
        pred = [int(logits_by_id[p4_id(split, i)].argmax(-1)) for i in idx]
        return metrics4(gold, pred)

    def trainable_state(model):
        return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

    def train4(model, train_idx, dev_idx, loss_fn, teacher=None, *, EPOCHS, LR_ENC, LR_HEAD, WARMUP_FRAC, WD,
               ACCUM_DOCS, CLASS_W_POWER, TIME_BUDGET_H, EVAL_EVERY=None, CLIP=1.0, seed=0, tag="run"):
        """One case per forward pass, gradient accumulation over ACCUM_DOCS cases.
        Stops cleanly when TIME_BUDGET_H hours of training have passed. Keeps the best-dev trainable weights."""
        torch.manual_seed(seed); rng = random.Random(seed)
        train_idx = list(train_idx)
        cw = class_weights4("train", train_idx, CLASS_W_POWER)
        enc_p  = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("encoder.")]
        head_p = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("encoder.")]
        groups = [g for g in ({"params": enc_p, "lr": LR_ENC}, {"params": head_p, "lr": LR_HEAD}) if g["params"]]
        opt = torch.optim.AdamW(groups, weight_decay=WD)
        steps_ep = math.ceil(len(train_idx) / ACCUM_DOCS); total = EPOCHS * steps_ep
        warm = max(1, int(WARMUP_FRAC * total))
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
        scaler = torch.amp.GradScaler(DEV_TYPE, enabled=(DEV_TYPE == "cuda"))
        params = [p for g in groups for p in g["params"]]
        best = {"f1": -1.0}; best_state = None; hist = []; n_oom = 0
        t_start = time.time(); stop = False; seen_tok = 0; done = 0
        _probe = random.Random(seed + 1).sample(train_idx, min(200, len(train_idx)))       # random sample, not file order
        n_tok_epoch = sum(int(sum(len(c) for c in doc_chunks("train", i))) for i in _probe) / len(_probe) * len(train_idx)
        win_l, win_n, win_p1, win_y1 = 0.0, 0, 0, 0                                        # recent-window diagnostics

        def dev_check(label):
            nonlocal best, best_state
            if not dev_idx: return
            res = evaluate4("dev", dev_idx, predict4(model, "dev", dev_idx)); res["at"] = label; hist.append(res)
            print(f"  [{label}] dev macro-F1 {res['f1']:.4f} | P {res['p']:.4f} | R {res['r']:.4f}")
            if res["f1"] > best["f1"]:
                best, best_state = res, trainable_state(model)
                torch.save(best_state, f"/kaggle/working/weights_{tag}.pt")       # survives a session kill

        model.train()
        for ep in range(EPOCHS):
            rng.shuffle(train_idx); t0 = time.time(); tot = 0.0
            for k, i in enumerate(train_idx):
                x, lens = p4_tensors("train", i)
                y = torch.tensor([p4_label("train", i)], device=device)
                tch = teacher[p4_id("train", i)].to(device) if teacher is not None else None
                try:
                    with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                        lg = model(x, lens)
                    loss = loss_fn(lg.float().reshape(1, 2), y, cw, teacher=tch) / ACCUM_DOCS
                    scaler.scale(loss).backward(); tot += loss.item() * ACCUM_DOCS
                    win_l += loss.item() * ACCUM_DOCS; win_n += 1
                    win_p1 += int(lg.reshape(1, 2).argmax(-1).item() == 1); win_y1 += int(y.item() == 1)
                except torch.cuda.OutOfMemoryError:
                    n_oom += 1; opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                    print(f"  OOM on case {p4_id('train', i)} ({x.shape[0]} chunks) -- skipped ({n_oom} so far)")
                    continue
                seen_tok += int(lens.sum()); done += 1
                if (k + 1) % ACCUM_DOCS == 0 or k + 1 == len(train_idx):
                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, CLIP)
                    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
                el = time.time() - t_start
                if done == 50:                                                     # early speed check
                    sp = seen_tok / el
                    print(f"  speed check: {sp:,.0f} tokens/s | projected epoch {n_tok_epoch / sp / 3600:.2f} h "
                          f"| budget {TIME_BUDGET_H} h | GPU mem peak {torch.cuda.max_memory_allocated() / 1e9:.1f} GB"
                          if DEV_TYPE == "cuda" else f"  speed check: {sp:,.0f} tokens/s")
                    steps_fit = int(TIME_BUDGET_H * 3600 * done / el / ACCUM_DOCS)
                    print(f"  LR schedule: {total} planned steps (EPOCHS={EPOCHS}), warmup {warm} | about {steps_fit} steps fit in the budget")
                    if steps_fit < warm:
                        print("  !! the budget ends INSIDE warmup: LR never reaches LR_ENC/LR_HEAD -- check EPOCHS / WARMUP_FRAC vs TIME_BUDGET_H")
                    elif steps_fit < total:
                        print("  !! the budget ends before the planned schedule finishes: LR will not decay to 0 -- check EPOCHS vs TIME_BUDGET_H")
                if (k + 1) % 1000 == 0 and win_n:
                    print(f"  ep {ep + 1} | {k + 1}/{len(train_idx)} cases | last-{win_n} loss {win_l / win_n:.4f} "
                          f"(50/50 guess = 0.693) | predicted 1: {win_p1 / win_n:.0%} vs true 1: {win_y1 / win_n:.0%} "
                          f"| LR {sched.get_last_lr()[0]:.2e} | {el / 3600:.2f} h")
                    win_l, win_n, win_p1, win_y1 = 0.0, 0, 0, 0
                if EVAL_EVERY and (k + 1) % EVAL_EVERY == 0:
                    dev_check(f"ep{ep + 1} case{k + 1}")
                if el > TIME_BUDGET_H * 3600:
                    print(f"  time budget reached after {k + 1} cases of epoch {ep + 1}"); stop = True; break
            print(f"epoch {ep + 1}/{EPOCHS} | train loss {tot / max(1, k + 1):.4f} | {time.time() - t0:.0f}s | OOM-skipped {n_oom}")
            dev_check(f"end ep{ep + 1}" if not stop else f"stop ep{ep + 1}")
            if stop: break
        if best_state is not None:
            model.load_state_dict(best_state, strict=False)
            print("BEST dev:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in best.items()})
        return best, hist

    def write_submission4(logits_by_id, tag):
        pred = []
        for cid in TOK4["ids"]["test"]:
            pred.append({"id": ORIG_ID["test"][cid], "pred": int(logits_by_id[cid].argmax(-1))})
        want = [str(d["id"]) for d in RAW4["test"]]
        assert len(pred) == len(set(want)) == len(want) and {str(p["id"]) for p in pred} == set(want), "test id mismatch"
        assert all(p["pred"] in (0, 1) for p in pred)
        js = f"/kaggle/working/prediction.json"; zp = f"/kaggle/working/submission_{tag}.zip"
        json.dump(pred, open(js, "w"))
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(js, arcname="prediction.json")
        c = collections.Counter(p["pred"] for p in pred)
        print(f"wrote {zp}: {len(pred)} predictions | pred 0: {c[0]}  pred 1: {c[1]}")

### 4.5 Run (uses the CONTROL cell)
Writes `submission_p4_<arch>_<technique>.zip`, dev/test logits, history; teacher logits when `CACHE_TEACHER_LOGITS = True`.

In [ ]:
if PHASE == 4:
    # ===================== 4.5  Run (uses the CONTROL cell) =====================
    P4_TECH = ("full", "partial", "freezing", "bitfit", "lorma", "adalora", "adapters", "kd")
    assert TECHNIQUE in P4_TECH, f"TECHNIQUE={TECHNIQUE!r} is not on the Phase-4 list {P4_TECH}"
    assert TECHNIQUE in ("full", "lorma", "kd") or TECHNIQUE in EXTRA_TECHNIQUES, f"write '{TECHNIQUE}' into EXTRA_TECHNIQUES first"
    need("EPOCHS", "LR_ENC", "LR_HEAD", "WARMUP_FRAC", "WD", "ACCUM_DOCS", "CLASS_W_POWER", "TIME_BUDGET_H", "KEEP")
    tag4 = f"p4_{ARCH}_{TECHNIQUE}"
    settings = {k: HP[k] for k in ("EPOCHS", "LR_ENC", "LR_HEAD", "WARMUP_FRAC", "WD", "ACCUM_DOCS",
                                   "CLASS_W_POWER", "TIME_BUDGET_H")}
    settings["EVAL_EVERY"] = HP.get("EVAL_EVERY")
    train_idx = p4_items("train", HP.get("TRAIN_FRAC"), SEED)
    dev_idx, test_idx = p4_items("dev"), p4_items("test")

    teacher, loss_fn = None, train_loss()
    if TECHNIQUE == "kd":
        assert TEACHER_TAG, "set TEACHER_TAG in the CONTROL cell (e.g. 'arch3_full') and attach that run's teacher file"
        teacher = torch.load(find_input(f"teacher_logits_p4_{TEACHER_TAG}.pt"))
        train_idx = [i for i in train_idx if p4_id("train", i) in teacher]          # cases the teacher labelled
        model, loss_fn = build_student4().to(device), kd_loss
        trainable_report(model, "KD student")
    else:
        model = build_cjpe_model()
    print(f"{tag4}: train cases {len(train_idx)} | dev {len(dev_idx)} | test {len(test_idx)} | KEEP={HP['KEEP']}")

    t_start = time.time()
    best, hist = train4(model, train_idx, dev_idx, loss_fn, teacher, seed=SEED, tag=tag4, **settings)
    print(f"training time {(time.time() - t_start) / 3600:.2f} h")

    dev_logits = predict4(model, "dev", dev_idx)
    print("DEV (best weights):", evaluate4("dev", dev_idx, dev_logits))
    torch.save(dev_logits, f"/kaggle/working/devlogits_{tag4}.pt")
    test_logits = predict4(model, "test", test_idx)
    torch.save(test_logits, f"/kaggle/working/testlogits_{tag4}.pt")
    write_submission4(test_logits, tag4)
    json.dump(dict(hist=hist, settings=settings, hp={k: v for k, v in HP.items() if not callable(v)}),
              open(f"/kaggle/working/hist_{tag4}.json", "w"), default=str)

    if CACHE_TEACHER_LOGITS:                                    # this run will serve as the KD teacher
        t0 = time.time()
        tl = predict4(model, "train", train_idx)
        torch.save(tl, f"/kaggle/working/teacher_logits_p4_{ARCH}_{TECHNIQUE}.pt")
        print(f"saved teacher logits for {len(tl)} train cases in {(time.time() - t0) / 3600:.2f} h")

# PHASE 5 — Legal Statute Identification (multi-label)
Run order 5.1 → 5.5 (5.4b sanity only interactively; 5.6 ensemble after runs finish). Per account edit only the CONTROL cell.

### 5.1 Data, label statistics, integrity checks

In [ ]:
if PHASE == 5:
    # ===================== 5.1  Data, label statistics, integrity checks (plumbing) =====================
    import hashlib

    def read_json_any(path):
        try:
            return json.load(open(path))
        except json.JSONDecodeError:
            return [json.loads(l) for l in open(path) if l.strip()]

    def to_records(obj, name):
        if isinstance(obj, dict):
            vals = list(obj.values())
            if vals and all(isinstance(v, dict) for v in vals):
                obj = [dict(id=k, **v) for k, v in obj.items()]
            elif any(isinstance(v, list) and v and isinstance(v[0], dict) for v in vals):
                obj = next(v for v in vals if isinstance(v, list) and v and isinstance(v[0], dict))
            else:
                raise ValueError(f"{name}: unknown layout, top keys {list(obj)[:5]}")
        assert isinstance(obj, list) and obj and isinstance(obj[0], dict), f"{name}: expected a list of records"
        return obj

    def pick(keys, cands, what, required=True):
        for c in cands:
            if c in keys: return c
        if required: raise KeyError(f"no {what} field among {sorted(keys)}")
        return None

    norm = lambda s: re.sub(r"\s+", " ", s).strip().lower()
    md5  = lambda s: hashlib.md5(norm(s).encode()).hexdigest()

    RAW5 = {}
    for split, key in (("train", "Train/train.json"), ("dev", "Train/dev.json"), ("test", "Test/test.json")):
        recs = to_records(read_json_any(P[key]), key); keys = set(recs[0])
        ID_K = pick(keys, ("id", "case_id", "doc_id"), "id")
        TX_K = pick(keys, ("text", "document", "facts"), "text")
        LB_K = pick(keys, ("labels", "label", "statutes"), "labels", required=(split != "test"))
        out = []
        for r in recs:
            t = r[TX_K]; t = " ".join(map(str, t)) if isinstance(t, list) else str(t)
            d = {"id": r[ID_K], "text": t}
            if LB_K and split != "test":
                lab = r[LB_K]; lab = lab if isinstance(lab, list) else [lab]
                d["labels"] = sorted({int(x) for x in lab})
            out.append(d)
        RAW5[split] = out
        print(f"{split:5s}: {len(out):6d} cases | id='{ID_K}' text='{TX_K}' labels='{LB_K}' | other fields: {sorted(keys - {ID_K, TX_K, LB_K})}")

    # ---- any other official Phase-5 file (e.g. statute texts) ----
    extra = [k for k in P if k not in ("Train/train.json", "Train/dev.json", "Test/test.json")]
    print("other official Phase-5 files:", extra or "none")
    for k in extra:
        try:
            obj = read_json_any(P[k])
            smp = list(obj.items())[:1] if isinstance(obj, dict) else obj[:1]
            print(f"  {k}: {type(obj).__name__} with {len(obj)} entries | sample: {str(smp)[:300]}")
        except Exception as e:
            print(f"  {k}: not JSON ({e})")

    # ---- integrity: duplicates inside splits, nothing shared with test ----
    IDS5  = {s: [str(d["id"]) for d in RAW5[s]] for s in RAW5}
    HASH5 = {s: {md5(d["text"]) for d in RAW5[s]} for s in RAW5}
    for s in RAW5:
        dup = [k for k, n in collections.Counter(IDS5[s]).items() if n > 1]
        print(f"{s:5s}: unique ids {len(set(IDS5[s]))} / {len(IDS5[s])} | duplicated ids {len(dup)} | unique texts {len(HASH5[s])}")
    assert len(IDS5["test"]) == len(set(IDS5["test"])), "duplicate ids inside test"
    for a, b in (("train", "dev"), ("train", "test"), ("dev", "test")):
        ni, nt = len(set(IDS5[a]) & set(IDS5[b])), len(HASH5[a] & HASH5[b])
        print(f"overlap {a}-{b}: ids {ni} | identical texts {nt}")
        if b == "test":
            assert ni == 0, f"{a} shares case ids with test"

    # ---- identical facts across / inside splits: official-data property, reported not blocked ----
    by_hash = {s: collections.defaultdict(list) for s in RAW5}
    for s in RAW5:
        for d in RAW5[s]: by_hash[s][md5(d["text"])].append(d)
    for a, b in (("train", "train"), ("train", "dev"), ("train", "test"), ("dev", "test")):
        common = [h for h in by_hash[a] if len(by_hash[a][h]) > 1] if a == b else list(set(by_hash[a]) & set(by_hash[b]))
        if not common: continue
        wl = [len(by_hash[b][h][0]["text"].split()) for h in common]
        msg = f"identical facts {a}-{b}: {len(common)} texts | words per text median {int(np.median(wl))}, max {max(wl)}"
        if b != "test":
            if a == b:
                agree = sum(len({tuple(x["labels"]) for x in by_hash[a][h]}) == 1 for h in common)
            else:
                agree = sum(any(tuple(x["labels"]) == tuple(by_hash[b][h][0]["labels"]) for x in by_hash[a][h]) for h in common)
            msg += f" | same labels {agree}/{len(common)}"
        print(msg)

    # ---- label statistics ----
    LABELS = sorted({l for s in ("train", "dev") for d in RAW5[s] for l in d["labels"]})
    L = len(LABELS)
    for s in ("train", "dev"):
        k = [len(d["labels"]) for d in RAW5[s]]
        print(f"{s:5s}: labels per case mean {np.mean(k):.2f} | min {min(k)} | max {max(k)} | cases with 0 labels {sum(x == 0 for x in k)}")
    ctr = collections.Counter(l for d in RAW5["train"] for l in d["labels"])
    cdv = collections.Counter(l for d in RAW5["dev"] for l in d["labels"])
    fr = sorted(((ctr[l], l) for l in LABELS), reverse=True)
    print(f"{L} distinct labels | train positives per label: max {fr[0][0]} (id {fr[0][1]}) | median {fr[L // 2][0]} | min {fr[-1][0]} (id {fr[-1][1]})")
    print("  labels with fewer than N train positives: " + " | ".join(f"<{t}: {sum(c < t for c, _ in fr)}" for t in (10, 20, 50, 100, 200)))
    print("  labels with fewer than N dev positives:   " + " | ".join(f"<{t}: {sum(cdv[l] < t for l in LABELS)}" for t in (1, 5, 10, 20, 50)))
    print("  labels in dev but never in train:", sorted(set(cdv) - set(ctr)) or "none")
    ftr = np.array([ctr[l] / len(RAW5["train"]) for l in LABELS]); fdv = np.array([cdv[l] / len(RAW5["dev"]) for l in LABELS])
    big = np.argsort(-np.abs(ftr - fdv))[:5]
    print(f"train vs dev label frequency: correlation {np.corrcoef(ftr, fdv)[0, 1]:.3f} | largest shifts: "
          + ", ".join(f"{LABELS[j]}: {ftr[j]:.3f}->{fdv[j]:.3f}" for j in big))

    # ---- length statistics + tokenizer cost on a sample ----
    def pct(xs, q): xs = sorted(xs); return xs[min(len(xs) - 1, int(q * len(xs)))]
    for s in RAW5:
        wl = [len(d["text"].split()) for d in RAW5[s]]
        print(f"{s:5s} words/case p50 {pct(wl, .5)} p90 {pct(wl, .9)} p99 {pct(wl, .99)} max {max(wl)} | total {sum(wl) / 1e6:.1f}M words")
    rng = random.Random(SEED); sample = rng.sample(RAW5["train"], min(200, len(RAW5["train"])))
    lens = [len(tok.encode(d["text"])) for d in sample]
    tpw = sum(lens) / max(1, sum(len(d["text"].split()) for d in sample))
    print(f"tokens/case (sample of {len(sample)}): p50 {pct(lens, .5)} p90 {pct(lens, .9)} max {max(lens)} | {tpw:.2f} tokens/word")


### 5.2 De-duplicate, tokenize once (cached), chunking, TAPT export

In [ ]:
if PHASE == 5:
    # ===================== 5.2  De-duplicate, tokenize once (cached), chunking, TAPT export (plumbing) =====================
    CACHE5 = "p5_tokens.pt"

    def dedup(recs):
        seen, out = set(), []
        for d in recs:
            k = str(d["id"])
            if k not in seen: seen.add(k); out.append(d)
        return out
    DATA5 = {s: (dedup(RAW5[s]) if HP["DEDUP"] else list(RAW5[s])) for s in RAW5}
    for s in DATA5: print(f"{s:5s}: {len(RAW5[s])} rows -> {len(DATA5[s])} cases")

    hits = glob.glob(f"/kaggle/input/**/{CACHE5}", recursive=True) + glob.glob(f"/kaggle/working/{CACHE5}")
    TOK5 = None
    if hits:
        TOK5 = torch.load(hits[0], weights_only=False)
        if TOK5["cfg"]["DEDUP"] != HP["DEDUP"] or any(TOK5["ids"][s] != [str(d["id"]) for d in DATA5[s]] for s in DATA5):
            print("token cache does not match the data -> re-tokenizing"); TOK5 = None
        else:
            print("loaded token cache:", hits[0])
    if TOK5 is None:
        TOK5 = {"cfg": {"DEDUP": HP["DEDUP"]}, "ids": {}, "labels": {}, "offsets": {}, "tokens": {}}
        for s in DATA5:
            t0 = time.time(); seqs = [np.asarray(tok.encode(d["text"]), dtype=np.uint16) for d in DATA5[s]]
            TOK5["ids"][s]     = [str(d["id"]) for d in DATA5[s]]
            TOK5["labels"][s]  = [d.get("labels", []) for d in DATA5[s]]
            TOK5["offsets"][s] = np.concatenate([[0], np.cumsum([len(x) for x in seqs])]).astype(np.int64)
            TOK5["tokens"][s]  = np.concatenate(seqs)
            print(f"{s:5s}: {len(seqs)} cases, {len(TOK5['tokens'][s]) / 1e6:.1f}M tokens, {time.time() - t0:.0f}s")
        torch.save(TOK5, f"/kaggle/working/{CACHE5}")
    ORIG_ID5 = {s: {str(d["id"]): d["id"] for d in DATA5[s]} for s in DATA5}

    def doc_tokens5(split, i):
        o = TOK5["offsets"][split]
        return TOK5["tokens"][split][o[i]:o[i + 1]]

    def chunk_starts(n, Lc, stride):
        if n <= Lc: return [0]
        st = list(range(0, n - Lc + 1, stride))
        if st[-1] + Lc < n: st.append(n - Lc)
        return st

    def select_chunks(st, max_chunks, keep):
        if len(st) <= max_chunks: return st
        if keep == "first": return st[:max_chunks]
        if keep == "last":  return st[-max_chunks:]
        if keep == "both":
            a = (max_chunks + 1) // 2
            return st[:a] + st[len(st) - (max_chunks - a):]
        raise ValueError("HP['KEEP'] must be 'first', 'last' or 'both'")

    def doc_chunks5(split, i):
        x = doc_tokens5(split, i)
        Lc = HP["CHUNK_LEN"]; stride = Lc - int(round(Lc * HP["OVERLAP"]))
        st = select_chunks(chunk_starts(max(len(x), 1), Lc, stride), HP["MAX_CHUNKS"], HP["KEEP"])
        if len(x) == 0: x = np.array([EOS_ID], dtype=np.uint16)
        return [torch.from_numpy(x[a:a + Lc].astype(np.int64)) for a in st]

    # ---- what the chunk settings imply ----
    Lc = HP["CHUNK_LEN"]; stride = Lc - int(round(Lc * HP["OVERLAP"]))
    print(f"\nCHUNK_LEN {Lc} | stride {stride} | MAX_CHUNKS {HP['MAX_CHUNKS']} -> fully covered up to {Lc + (HP['MAX_CHUNKS'] - 1) * stride} tokens")
    for s in DATA5:
        n = np.diff(TOK5["offsets"][s]); nch = np.array([len(chunk_starts(max(int(k), 1), Lc, stride)) for k in n])
        cap = np.minimum(nch, HP["MAX_CHUNKS"])
        used = sum(min(int(k), Lc) if c == 1 else c * Lc for k, c in zip(n, cap))
        print(f"{s:5s}: tokens/case p50 {int(np.median(n))} p90 {int(np.percentile(n, 90))} | chunks/case p50 {int(np.median(nch))} "
              f"p90 {int(np.percentile(nch, 90))} | over the cap {np.mean(nch > HP['MAX_CHUNKS']):.1%} | encoder tokens/epoch {used / 1e6:.1f}M")
        if s == "train":
            print(f"   one train epoch at ~5,100 tok/s (full FT, T4) = {used / 5100 / 3600:.1f} h | at ~5,500 (LoRMA) = {used / 5500 / 3600:.1f} h")

    # ---- TAPT export: Phase-5 facts as token streams for the MLM notebook (train facts -> train, dev facts -> val) ----
    def export_stream(split, path):
        o, t = TOK5["offsets"][split], TOK5["tokens"][split]
        eos = np.array([EOS_ID], dtype=np.uint16)
        np.save(path, np.concatenate([np.concatenate([t[o[i]:o[i + 1]], eos]) for i in range(len(o) - 1)]))
        print(f"wrote {path} ({os.path.getsize(path) / 1e6:.0f} MB)")
    export_stream("train", "/kaggle/working/p5_tapt_train_ids.npy")
    export_stream("dev",   "/kaggle/working/p5_tapt_val_ids.npy")


### 5.3 YOUR CODE — model (`LSIModel`), optional loss, KD student + KD loss
Contract: `__init__(encoder, d_model, n_labels)`, backbone in `self.encoder`, `forward(win_ids, win_lens)` → raw logits `(1, n_labels)`.

In [ ]:
if PHASE == 5:
    class LSIModel(nn.Module):
        def __init__(self, encoder, d_model, n_labels):
            super().__init__()
            self.encoder = encoder
            self.classifier = nn.Linear(d_model, n_labels)

        def forward(self, win_ids, win_lens):
            H = encoder_states(self.encoder, win_ids, win_lens)
            n_chunks, T, d_model = H.shape
            mask = torch.arange(T, device=H.device).unsqueeze(0) < win_lens.unsqueeze(1)
            H_masked = H.masked_fill(~mask.unsqueeze(-1), float('-inf'))
            H_flat = H_masked.view(n_chunks * T, d_model)
            pooled, _ = H_flat.max(dim=0)
            logits = self.classifier(pooled).unsqueeze(0)
            return logits

    def build_lsi_model():
        # main runs: the CONTROL cell's TECHNIQUE (full / lorma / partial ...) is applied here
        enc, d_model = load_backbone()
        return apply_technique(LSIModel(enc, d_model, L)).to(device)

    def build_student5():
        # KD runs only: smaller student, every parameter trains
        enc, d_model = load_backbone()
        initial_layers = len(enc.blocks)
        enc.blocks = enc.blocks[:HP["STUDENT_LAYERS"]]
        assert len(enc.blocks) < initial_layers
        model = LSIModel(enc, d_model, L)
        for p in model.parameters(): p.requires_grad = True
        print(f"KD student: kept {len(enc.blocks)} of {initial_layers} blocks")
        return model.to(device)

    def lsi_loss(logits, y, pw, teacher=None):
        return F.binary_cross_entropy_with_logits(logits, y, pos_weight=pw)

    def kd_loss5(logits, y, pw, teacher=None):
        hard_loss = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pw)
        soft_target = torch.sigmoid(teacher.detach())
        soft_loss = F.binary_cross_entropy_with_logits(logits, soft_target)
        alpha = HP.get("KD_ALPHA", 0.5)
        return (1.0 - alpha) * hard_loss + alpha * soft_loss


### 5.4 Multi-label harness (training, diagnostics, thresholds, metrics, submission)

In [ ]:
if PHASE == 5:
    # ===================== 5.4  Multi-label harness: training, thresholds, metrics, submission (plumbing) =====================
    LAB2I = {l: j for j, l in enumerate(LABELS)}

    def p5_items(split, frac=None, seed=0):
        idx = list(range(len(TOK5["ids"][split])))
        if frac is not None and frac < 1.0:
            idx = sorted(random.Random(seed).sample(idx, int(round(frac * len(idx)))))
        return idx

    def p5_tensors(split, i):
        ch = doc_chunks5(split, i); T = max(len(c) for c in ch)
        x = torch.full((len(ch), T), EOS_ID, dtype=torch.long)
        for k, c in enumerate(ch): x[k, :len(c)] = c
        return x.to(device), torch.tensor([len(c) for c in ch]).to(device)

    def p5_y(split, i):
        y = torch.zeros(1, L)
        for l in TOK5["labels"][split][i]:
            if l in LAB2I: y[0, LAB2I[l]] = 1.0
        return y.to(device)

    def p5_id(split, i): return TOK5["ids"][split][i]

    def label_stats(idx, power):
        cnt = np.zeros(L)
        for i in idx:
            for l in TOK5["labels"]["train"][i]:
                if l in LAB2I: cnt[LAB2I[l]] += 1
        prior = cnt / max(1, len(idx))
        pw = ((len(idx) - cnt) / np.maximum(cnt, 1)) ** power             # power 0 -> plain BCE
        pw = np.where(cnt > 0, pw, 1.0)                                    # a label never seen in train gets no boost
        return torch.tensor(pw, dtype=torch.float32, device=device), prior, pw

    def no_skill_loss(prior, pw):
        """loss of the best CONSTANT prediction per label under pos-weighted BCE (the 'learned nothing' reference)"""
        p = np.clip(prior, 1e-6, 1 - 1e-6); q = pw * p / (pw * p + 1 - p)
        return float(np.mean(-pw * p * np.log(q) - (1 - p) * np.log(1 - q)))

    def bce_posw(logits, y, pw, teacher=None):
        # per-element weighting: works at any batch size, including ONE case per forward pass
        return F.binary_cross_entropy_with_logits(logits, y, pos_weight=pw)

    @torch.no_grad()
    def predict5(model, split, idx):
        model.eval(); out = np.zeros((len(idx), L), np.float32)
        for r, i in enumerate(idx):
            x, lens = p5_tensors(split, i)
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                lg = model(x, lens)
            out[r] = lg.float().reshape(-1).cpu().numpy()
        model.train(); return out                                          # raw logits (n, L)

    def trainable_state(model):
        return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

    # ---- numpy side: metrics, thresholds, submission (no torch needed) ----
    def gold_matrix(split, idx):
        Y = np.zeros((len(idx), L), np.uint8)
        for r, i in enumerate(idx):
            for l in TOK5["labels"][split][i]:
                if l in LAB2I: Y[r, LAB2I[l]] = 1
        return Y

    def eval_cols(Y):
        """label columns the macro average runs over: 'gold' = labels present in the reference, 'all' = every label"""
        return np.arange(L) if HP.get("EVAL_LABELS", "gold") == "all" else np.where(Y.sum(0) > 0)[0]

    def metrics5(Y, Pr):
        Y = Y.astype(np.int64); Pr = Pr.astype(np.int64)
        tp = (Y * Pr).sum(0); fp = ((1 - Y) * Pr).sum(0); fn = (Y * (1 - Pr)).sum(0)
        den = 2 * tp + fp + fn
        f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
        cols = eval_cols(Y)
        return dict(macro=float(f1[cols].mean()) if len(cols) else 0.0,
                    micro=float(2 * tp.sum() / max(1, den.sum())),
                    subset=float((Y == Pr).all(1).mean()), n_cols=int(len(cols)),
                    per_label=f1)

    def best_thr_for(p, y, lo, hi):
        """threshold for ONE label maximising its F1; candidates = the label's own scores inside [lo, hi]"""
        P = int(y.sum())
        if P == 0: return None, 0.0
        o = np.argsort(-p); ps, ys = p[o], y[o].astype(np.int64)
        tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
        f1 = 2 * tp / (tp + fp + P)                       # predict the top-k (k = position + 1) as positive
        last = np.r_[ps[1:] != ps[:-1], True]             # only cut between different scores (ties stay together)
        ok = last & (ps >= lo) & (ps <= hi)
        if not ok.any(): return None, 0.0
        k = int(np.argmax(np.where(ok, f1, -1.0)))
        return float(ps[k]), float(f1[k])

    def apply_thr(Pm, thr, min_labels):
        Pr = (Pm >= thr[None, :]).astype(np.uint8)
        if min_labels:
            short = np.where(Pr.sum(1) < min_labels)[0]
            if len(short):
                top = np.argsort(-Pm[short], 1)[:, :min_labels]
                Pr[short[:, None], top] = 1
        return Pr

    def tune_thresholds(Pm, Y, min_pos, lo, hi, min_labels):
        """one global threshold (best macro-F1) for every label, then a per-label threshold where the label has
        at least min_pos positives in Y. Returns (thresholds, global threshold, number of labels tuned individually)."""
        cand = np.unique(np.quantile(Pm, np.linspace(0, 1, 201)))
        cand = cand[(cand >= lo) & (cand <= hi)]
        if len(cand) == 0: cand = np.array([min(max(0.5, lo), hi)])
        g = float(max(cand, key=lambda t: metrics5(Y, apply_thr(Pm, np.full(L, t), min_labels))["macro"]))
        thr, n = np.full(L, g), 0
        for j in range(L):
            if Y[:, j].sum() >= min_pos:
                t, _ = best_thr_for(Pm[:, j], Y[:, j], lo, hi)
                if t is not None: thr[j] = t; n += 1
        return thr, g, n

    def crossfit_macro(Pm, Y, min_pos, lo, hi, min_labels, seed=0):
        """honest estimate: tune on one half of dev, score the other half, both ways, average"""
        r = np.random.RandomState(seed).permutation(len(Y)); a, b = r[: len(r) // 2], r[len(r) // 2:]
        out = []
        for tr, te in ((a, b), (b, a)):
            thr, _, _ = tune_thresholds(Pm[tr], Y[tr], max(1, min_pos // 2), lo, hi, min_labels)
            out.append(metrics5(Y[te], apply_thr(Pm[te], thr, min_labels))["macro"])
        return float(np.mean(out))

    def sig(x): return 1.0 / (1.0 + np.exp(-x))

    def dev_report(Lg, Y):
        Pm = sig(Lg)
        mp, lo, hi, ml = HP["THR_MIN_POS"], HP["THR_LO"], HP["THR_HI"], HP["MIN_LABELS"]
        raw = metrics5(Y, apply_thr(Pm, np.full(L, 0.5), 0))
        thr, g, nt = tune_thresholds(Pm, Y, mp, lo, hi, ml)
        tun = metrics5(Y, apply_thr(Pm, thr, ml))
        return dict(macro_at_0_5=raw["macro"], labels_per_case_at_0_5=float((Pm >= 0.5).sum(1).mean()),
                    macro_tuned=tun["macro"], micro_tuned=tun["micro"], subset_tuned=tun["subset"],
                    macro_crossfit=crossfit_macro(Pm, Y, mp, lo, hi, ml), global_thr=g, labels_tuned=nt,
                    labels_never_right=int((tun["per_label"][eval_cols(Y)] == 0).sum())), thr

    def write_submission5(Pm_test, thr, min_labels, tag):
        Pr = apply_thr(Pm_test, thr, min_labels)
        ids = TOK5["ids"]["test"]
        pred = [{"id": ORIG_ID5["test"][cid], "pred": [int(LABELS[j]) for j in np.where(Pr[r])[0]]} for r, cid in enumerate(ids)]
        want = [str(d["id"]) for d in RAW5["test"]]
        assert len(pred) == len(set(want)) == len(want) and {str(p["id"]) for p in pred} == set(want), "test id mismatch"
        assert all(len(set(p["pred"])) == len(p["pred"]) for p in pred) and all(isinstance(x, int) for p in pred for x in p["pred"])
        js, zp = "/kaggle/working/prediction.json", f"/kaggle/working/submission_{tag}.zip"
        json.dump(pred, open(js, "w"))
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(js, arcname="prediction.json")
        k = Pr.sum(1)
        print(f"wrote {zp}: {len(pred)} cases | labels/case mean {k.mean():.2f} | empty {int((k == 0).sum())} | "
              f"distinct labels predicted {int((Pr.sum(0) > 0).sum())} of {L}")

    def train5(model, train_idx, dev_idx, loss_fn, teacher=None, *, EPOCHS, LR_ENC, LR_HEAD, WARMUP_FRAC, WD,
               ACCUM_DOCS, POS_W_POWER, TIME_BUDGET_H, EVAL_EVERY=None, CLIP=1.0, seed=0, tag="run"):
        torch.manual_seed(seed); rng = random.Random(seed); train_idx = list(train_idx)
        pw_t, prior, pw = label_stats(train_idx, POS_W_POWER)
        ref = no_skill_loss(prior, pw); true_lpc = float(prior.sum())
        print(f"  no-skill loss reference {ref:.4f} | true labels/case {true_lpc:.2f} | pos_weight min {pw.min():.1f} max {pw.max():.1f}")
        Ydev = gold_matrix("dev", dev_idx) if dev_idx else None
        enc_p  = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("encoder.")]
        head_p = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("encoder.")]
        groups = [g for g in ({"params": enc_p, "lr": LR_ENC}, {"params": head_p, "lr": LR_HEAD}) if g["params"]]
        opt = torch.optim.AdamW(groups, weight_decay=WD)
        steps_ep = math.ceil(len(train_idx) / ACCUM_DOCS); total = EPOCHS * steps_ep; warm = max(1, int(WARMUP_FRAC * total))
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
        scaler = torch.amp.GradScaler(DEV_TYPE, enabled=(DEV_TYPE == "cuda"))
        params = [p for g in groups for p in g["params"]]
        best = {"macro_crossfit": -1.0}; best_state = None; hist = []; n_oom = 0
        t_start = time.time(); stop = False; seen_tok = 0; done = 0
        probe = random.Random(seed + 1).sample(train_idx, min(200, len(train_idx)))
        n_tok_epoch = sum(int(sum(len(c) for c in doc_chunks5("train", i))) for i in probe) / len(probe) * len(train_idx)
        win = dict(l=0.0, n=0, pred=0, true=0, labs=set())

        def dev_check(label):
            nonlocal best, best_state
            if not dev_idx: return
            rep, _ = dev_report(predict5(model, "dev", dev_idx), Ydev); rep["at"] = label; hist.append(rep)
            print(f"  [{label}] dev macro-F1: at 0.5 {rep['macro_at_0_5']:.4f} | thresholds tuned {rep['macro_tuned']:.4f} "
                  f"| cross-fitted (honest) {rep['macro_crossfit']:.4f} | micro {rep['micro_tuned']:.4f} | "
                  f"labels never right {rep['labels_never_right']} | labels/case at 0.5 {rep['labels_per_case_at_0_5']:.2f}")
            if rep["macro_crossfit"] > best["macro_crossfit"]:
                best, best_state = rep, trainable_state(model)
                torch.save(best_state, f"/kaggle/working/weights_{tag}.pt")

        model.train(); k = 0
        for ep in range(EPOCHS):
            rng.shuffle(train_idx); t0 = time.time(); tot = 0.0
            for k, i in enumerate(train_idx):
                x, lens = p5_tensors("train", i); y = p5_y("train", i)
                tch = teacher[p5_id("train", i)].to(device).reshape(1, L) if teacher is not None else None
                try:
                    with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                        lg = model(x, lens)
                    lg = lg.float().reshape(1, L)
                    loss = loss_fn(lg, y, pw_t, teacher=tch) / ACCUM_DOCS
                    scaler.scale(loss).backward(); lv = loss.item() * ACCUM_DOCS; tot += lv
                except torch.cuda.OutOfMemoryError:
                    n_oom += 1; opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                    print(f"  OOM on case {p5_id('train', i)} ({x.shape[0]} chunks) -- skipped ({n_oom} so far)"); continue
                pr = (lg.detach() > 0).reshape(-1)
                win["l"] += lv; win["n"] += 1; win["pred"] += int(pr.sum()); win["true"] += int(y.sum())
                win["labs"].update(torch.where(pr)[0].tolist())
                seen_tok += int(lens.sum()); done += 1
                if (k + 1) % ACCUM_DOCS == 0 or k + 1 == len(train_idx):
                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, CLIP)
                    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
                el = time.time() - t_start
                if done == 50:
                    sp = seen_tok / el; steps_fit = int(TIME_BUDGET_H * 3600 * done / el / ACCUM_DOCS)
                    print(f"  speed check: {sp:,.0f} tok/s | projected epoch {n_tok_epoch / sp / 3600:.2f} h | budget {TIME_BUDGET_H} h"
                          + (f" | GPU mem peak {torch.cuda.max_memory_allocated() / 1e9:.1f} GB" if DEV_TYPE == "cuda" else ""))
                    print(f"  LR schedule: {total} planned steps ({EPOCHS} epochs), warmup {warm} | about {steps_fit} steps fit in the budget")
                    if steps_fit < warm: print("  !! the budget ends INSIDE warmup -- check EPOCHS / WARMUP_FRAC vs TIME_BUDGET_H")
                    elif steps_fit < total: print("  !! the budget ends before the planned schedule finishes -- check EPOCHS vs TIME_BUDGET_H")
                if (k + 1) % 1000 == 0 and win["n"]:
                    print(f"  ep {ep + 1} | {k + 1}/{len(train_idx)} | last-{win['n']} loss {win['l'] / win['n']:.4f} "
                          f"(no-skill {ref:.4f}) | labels/case predicted {win['pred'] / win['n']:.2f} vs true {win['true'] / win['n']:.2f} "
                          f"| distinct labels predicted {len(win['labs'])}/{L} | LR {sched.get_last_lr()[0]:.2e} | {el / 3600:.2f} h")
                    win = dict(l=0.0, n=0, pred=0, true=0, labs=set())
                if EVAL_EVERY and (k + 1) % EVAL_EVERY == 0: dev_check(f"ep{ep + 1} case{k + 1}")
                if el > TIME_BUDGET_H * 3600:
                    print(f"  time budget reached after {k + 1} cases of epoch {ep + 1}"); stop = True; break
            print(f"epoch {ep + 1}/{EPOCHS} | train loss {tot / max(1, k + 1):.4f} | {time.time() - t0:.0f}s | OOM-skipped {n_oom}")
            dev_check(f"end ep{ep + 1}" if not stop else f"stop ep{ep + 1}")
            if stop: break
        if best_state is not None:
            model.load_state_dict(best_state, strict=False)
            print("BEST dev (by cross-fitted macro-F1):", {k_: (round(v, 4) if isinstance(v, float) else v) for k_, v in best.items()})
        return best, hist


### 5.4b Sanity: memorise 32 cases (interactive only; set `SANITY = True`)

In [ ]:
if PHASE == 5:
    # ===================== 5.4b  SANITY (run interactively before any long run; delete or skip in commits) =====================
    # Can the model memorise 32 training cases? Loss must fall far below the no-skill reference and exact matches rise.
    SANITY = False                      # set True only in an interactive session
    if SANITY:
        sub = p5_items("train")[:32]
        m = build_lsi_model() if TECHNIQUE != "kd" else build_student5().to(device)
        m.train()
        opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=HP["LR_HEAD"])
        scaler = torch.amp.GradScaler(DEV_TYPE, enabled=(DEV_TYPE == "cuda"))
        ones = torch.ones(L, device=device)
        _, prior, _ = label_stats(sub, 0.0)
        print(f"no-skill reference on this subset {no_skill_loss(prior, np.ones(L)):.4f} | true labels/case {prior.sum():.2f}")
        for ep in range(15):
            tot, exact, f1s = 0.0, 0, []
            for i in sub:
                x, lens = p5_tensors("train", i); y = p5_y("train", i)
                with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                    lg = m(x, lens)
                loss = bce_posw(lg.float().reshape(1, L), y, ones)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                pr = (lg.detach().float().reshape(-1) > 0).cpu().numpy().astype(int); yy = y.reshape(-1).cpu().numpy().astype(int)
                tot += loss.item(); exact += int((pr == yy).all())
                f1s.append(2 * (pr * yy).sum() / max(1, pr.sum() + yy.sum()))
            print(f"pass {ep + 1:2d}: loss {tot / len(sub):.4f} | exact-match {exact}/{len(sub)} | mean per-case F1 {np.mean(f1s):.3f}")


### 5.5 Run (uses the CONTROL cell)

In [ ]:
if PHASE == 5 and P5_MODE == "train":
    # ===================== 5.5  Run (uses the CONTROL cell) =====================
    P5_TECH = ("full", "partial", "qlora", "lorma", "adalora", "adapters", "prefix", "prompt", "kd")
    assert TECHNIQUE in P5_TECH, f"TECHNIQUE={TECHNIQUE!r} is not on the Phase-5 list {P5_TECH}"
    assert TECHNIQUE in ("full", "lorma", "kd") or TECHNIQUE in EXTRA_TECHNIQUES, f"write '{TECHNIQUE}' into EXTRA_TECHNIQUES first"
    need("EPOCHS", "LR_ENC", "LR_HEAD", "WARMUP_FRAC", "WD", "ACCUM_DOCS", "TIME_BUDGET_H", "KEEP",
         "POS_W_POWER", "THR_MIN_POS", "THR_LO", "THR_HI", "MIN_LABELS", "LOSS5")
    tag5 = f"p5_{ARCH}_{TECHNIQUE}" + (f"_{RUN_NOTE}" if globals().get("RUN_NOTE") else "")
    settings = {k: HP[k] for k in ("EPOCHS", "LR_ENC", "LR_HEAD", "WARMUP_FRAC", "WD", "ACCUM_DOCS", "POS_W_POWER", "TIME_BUDGET_H")}
    settings["EVAL_EVERY"] = HP.get("EVAL_EVERY")
    train_idx = p5_items("train", HP.get("TRAIN_FRAC"), SEED); dev_idx, test_idx = p5_items("dev"), p5_items("test")

    teacher = None
    loss_fn = {"bce": bce_posw, "student": lsi_loss}[HP["LOSS5"]]
    if TECHNIQUE == "kd":
        assert TEACHER_TAG, "set TEACHER_TAG (e.g. 'arch3_full') and attach that run's teacher file"
        tl = torch.load(find_input(f"teacher_logits_p5_{TEACHER_TAG}.pt"), weights_only=False)   # our own file (numpy arrays inside)
        teacher = {k: torch.as_tensor(v) for k, v in tl.items()}
        train_idx = [i for i in train_idx if p5_id("train", i) in teacher]
        model, loss_fn = build_student5().to(device), kd_loss5
        trainable_report(model, "KD student")
    else:
        model = build_lsi_model()
        if HP.get("INIT_WEIGHTS"):                     # continue training from a finished run's best weights
            sd = torch.load(find_input(HP["INIT_WEIGHTS"]), map_location="cpu")
            missing, unexpected = model.load_state_dict(sd, strict=False)
            print(f"continued from {HP['INIT_WEIGHTS']}: {len(sd)} tensors loaded | not in file {len(missing)} | unexpected {len(unexpected)}")
    print(f"{tag5}: train {len(train_idx)} | dev {len(dev_idx)} | test {len(test_idx)} | labels {L} | loss {HP['LOSS5']}")

    t_run = time.time()
    best, hist = train5(model, train_idx, dev_idx, loss_fn, teacher, seed=SEED, tag=tag5, **settings)
    print(f"training time {(time.time() - t_run) / 3600:.2f} h")

    Ydev = gold_matrix("dev", dev_idx)
    dev_lg = predict5(model, "dev", dev_idx)
    rep, thr = dev_report(dev_lg, Ydev)
    print("DEV (best weights):", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in rep.items()})
    test_lg = predict5(model, "test", test_idx)
    torch.save({"ids": [p5_id("dev", i) for i in dev_idx], "logits": dev_lg, "labels": LABELS}, f"/kaggle/working/devlogits_{tag5}.pt")
    torch.save({"ids": [p5_id("test", i) for i in test_idx], "logits": test_lg, "labels": LABELS}, f"/kaggle/working/testlogits_{tag5}.pt")
    np.save(f"/kaggle/working/thresholds_{tag5}.npy", thr)
    json.dump(dict(hist=hist, settings=settings, final=rep, hp={k: v for k, v in HP.items() if not callable(v)}),
              open(f"/kaggle/working/hist_{tag5}.json", "w"), default=str)
    write_submission5(sig(test_lg), thr, HP["MIN_LABELS"], tag5)

    if CACHE_TEACHER_LOGITS:
        t0 = time.time(); tr_lg = predict5(model, "train", train_idx)
        torch.save({p5_id("train", i): tr_lg[r] for r, i in enumerate(train_idx)}, f"/kaggle/working/teacher_logits_p5_{ARCH}_{TECHNIQUE}.pt")
        print(f"saved teacher logits for {len(train_idx)} train cases in {(time.time() - t0) / 3600:.2f} h")


### 5.6 Ensemble of finished runs (CPU)

In [ ]:
if PHASE == 5 and P5_MODE == "ensemble":
    # ===================== 5.6  ENSEMBLE of finished runs + thresholds on the averaged scores (CPU is enough) =====================
    MEMBERS = None    # YOUR choice: tags whose devlogits_/testlogits_ files are attached, e.g. ["p5_arch3_full", "p5_arch2_lorma"]
    WRITE   = False   # True -> write the ensemble submission
    assert MEMBERS, "set MEMBERS"
    dev_by, test_by = {}, {}
    for t in MEMBERS:
        d, te = torch.load(find_input(f"devlogits_{t}.pt"), weights_only=False), torch.load(find_input(f"testlogits_{t}.pt"), weights_only=False)
        assert list(d["labels"]) == list(LABELS), f"{t}: label order differs from this notebook's LABELS"
        dev_by[t]  = dict(zip(d["ids"],  np.asarray(d["logits"])))
        test_by[t] = dict(zip(te["ids"], np.asarray(te["logits"])))
    dev_idx, test_idx = p5_items("dev"), p5_items("test")
    Ydev = gold_matrix("dev", dev_idx)
    for t in MEMBERS:
        rep, _ = dev_report(np.stack([dev_by[t][p5_id("dev", i)] for i in dev_idx]), Ydev)
        print(f"{t:28s} macro tuned {rep['macro_tuned']:.4f} | cross-fitted {rep['macro_crossfit']:.4f} | micro {rep['micro_tuned']:.4f}")
    # average PROBABILITIES, then go back to logits so dev_report / thresholds work unchanged
    def avg_logits(by, split, idx):
        Pm = np.mean([sig(np.stack([by[t][p5_id(split, i)] for i in idx])) for t in MEMBERS], axis=0)
        Pm = np.clip(Pm, 1e-6, 1 - 1e-6); return np.log(Pm / (1 - Pm))
    dlg = avg_logits(dev_by, "dev", dev_idx)
    rep, thr = dev_report(dlg, Ydev)
    print("ENSEMBLE", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in rep.items()})
    if WRITE:
        write_submission5(sig(avg_logits(test_by, "test", test_idx)), thr, HP["MIN_LABELS"], "p5_ensemble_" + "_".join(t.split("_")[1] for t in MEMBERS))


### 5.7 Diagnostics + retrieval (CPU; `P5_MODE = "diag"`)
Per-label F1 by frequency, nearest-training-case similarity, and kNN label retrieval alone / blended with a finished run — every score cross-fitted on dev.

In [ ]:
if PHASE == 5 and P5_MODE == "diag":
    # ===================== 5.7  DIAGNOSTICS + RETRIEVAL (CPU is enough; plumbing) =====================
    # (a) where macro-F1 is lost per label  (b) how close dev/test cases are to training cases
    # (c) how well "copy the labels of the most similar training cases" works alone and blended with a model
    from sklearn.feature_extraction.text import TfidfVectorizer
    DIAG_RUN  = ["p5_arch3_full_tapt"]        # one or more finished runs (attached): their probabilities are averaged, then blended with kNN
    KNN_K     = (25, 50, 100, 200)            # YOUR grid: neighbours used
    KNN_POW   = (4, 8, 16)                    # YOUR grid: neighbour weight = similarity ** power
    BLEND_LAM = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0)   # YOUR grid: final = (1-lam) * model + lam * kNN (1.0 = kNN alone)
    WRITE     = False                         # True -> write the best blend's submission + logits for 5.6

    tr_idx, dv_idx, te_idx = p5_items("train"), p5_items("dev"), p5_items("test")
    Ytr, Ydv = gold_matrix("train", tr_idx), gold_matrix("dev", dv_idx)
    mp, lo, hi, ml = HP["THR_MIN_POS"], HP["THR_LO"], HP["THR_HI"], HP["MIN_LABELS"]
    def logit(P): P = np.clip(P, 1e-6, 1 - 1e-6); return np.log(P / (1 - P))
    def run_probs(kind, split, idx):
        out = []
        for t in DIAG_RUN:
            d = torch.load(find_input(f"{kind}logits_{t}.pt"), weights_only=False)
            assert list(d["labels"]) == list(LABELS), f"{t}: label order differs"
            pos = {k: r for r, k in enumerate(d["ids"])}
            out.append(sig(np.stack([np.asarray(d["logits"])[pos[p5_id(split, i)]] for i in idx])))
        return np.mean(out, 0)
    RUN_NAME = "+".join(DIAG_RUN)

    # ---- (a) per-label F1 of the model, by label frequency ----
    Pdv_m = run_probs("dev", "dev", dv_idx)
    rep, thr = dev_report(logit(Pdv_m), Ydv)
    pl = metrics5(Ydv, apply_thr(Pdv_m, thr, ml))["per_label"]
    ftr = Ytr.mean(0); order = np.argsort(ftr)
    print(f"[a] {RUN_NAME}: dev macro tuned {rep['macro_tuned']:.4f} | cross-fitted {rep['macro_crossfit']:.4f}")
    for name, sl in zip(("rarest quarter", "2nd quarter", "3rd quarter", "commonest quarter"), np.array_split(order, 4)):
        print(f"    {name:17s}: train freq {ftr[sl].min():.3f}-{ftr[sl].max():.3f} | mean dev F1 {pl[sl].mean():.3f}")
    worst = np.argsort(pl)[:10]
    print("    10 worst labels  id: F1 (train positives): " + ", ".join(f"{LABELS[j]}: {pl[j]:.2f} ({int(Ytr[:, j].sum())})" for j in worst))
    lsets = {tuple(np.where(r)[0]) for r in Ytr}
    print(f"    dev cases whose exact label SET occurs in train: {np.mean([tuple(np.where(r)[0]) in lsets for r in Ydv]):.1%}")

    # ---- (b) lexical nearest neighbours (TF-IDF over your BPE token ids, fitted on TRAIN only) ----
    t0 = time.time()
    def docs(split, idx): return [" ".join(map(str, doc_tokens5(split, i)[:4000].tolist())) for i in idx]
    vec = TfidfVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=500000, dtype=np.float32)
    Xtr = vec.fit_transform(docs("train", tr_idx)); Xdv = vec.transform(docs("dev", dv_idx)); Xte = vec.transform(docs("test", te_idx))
    def topk(Xq, K, bs=1000):
        S_all, I_all = [], []
        for a in range(0, Xq.shape[0], bs):
            S = (Xq[a:a + bs] @ Xtr.T).toarray()
            I = np.argpartition(-S, K, axis=1)[:, :K]; Sk = np.take_along_axis(S, I, 1); o = np.argsort(-Sk, 1)
            I_all.append(np.take_along_axis(I, o, 1)); S_all.append(np.take_along_axis(Sk, o, 1))
        return np.vstack(S_all), np.vstack(I_all)
    Kmax = max(KNN_K); Sdv, Idv = topk(Xdv, Kmax); Ste, Ite = topk(Xte, Kmax)
    print(f"[b] TF-IDF + neighbours in {time.time() - t0:.0f}s | vocab {len(vec.vocabulary_):,}")
    for name, S in (("dev", Sdv), ("test", Ste)):
        s1 = S[:, 0]
        print(f"    {name:4s} nearest-train similarity: p50 {np.median(s1):.3f} | >0.95 {np.mean(s1 > .95):.1%} | >0.8 {np.mean(s1 > .8):.1%} | >0.6 {np.mean(s1 > .6):.1%}")
    for t in (0.95, 0.8, 0.6):
        m = Sdv[:, 0] > t
        if m.sum():
            same = np.mean([(Ytr[Idv[r, 0]] == Ydv[r]).all() for r in np.where(m)[0]])
            print(f"    dev cases with a train neighbour >{t}: {m.sum()} | identical label set with that neighbour {same:.1%}")

    # ---- (c) kNN alone, then blended with the model; every score is the honest cross-fitted macro-F1 ----
    def knn_P(S, I, K, p):
        W = (np.maximum(S[:, :K], 0) ** p + 1e-9).astype(np.float32)          # float32 keeps K=200 within memory
        return (W[:, :, None] * Ytr[I[:, :K]]).sum(1) / W.sum(1, keepdims=True)
    best_knn = None
    for K in KNN_K:
        for p in KNN_POW:
            r, _ = dev_report(logit(knn_P(Sdv, Idv, K, p)), Ydv)
            print(f"[c] kNN alone K={K:2d} pow={p}: cross-fitted macro {r['macro_crossfit']:.4f} (tuned {r['macro_tuned']:.4f})")
            if best_knn is None or r["macro_crossfit"] > best_knn[0]: best_knn = (r["macro_crossfit"], K, p)
    _, K, p = best_knn; Pdv_k = knn_P(Sdv, Idv, K, p)
    best = (rep["macro_crossfit"], 0.0)
    for lam in BLEND_LAM:
        r, _ = dev_report(logit((1 - lam) * Pdv_m + lam * Pdv_k), Ydv)
        print(f"    blend lam={lam:.1f} (K={K}, pow={p}): cross-fitted macro {r['macro_crossfit']:.4f} (tuned {r['macro_tuned']:.4f})")
        if r["macro_crossfit"] > best[0]: best = (r["macro_crossfit"], lam)
    print(f"BEST: lam={best[1]} K={K} pow={p} -> cross-fitted dev macro {best[0]:.4f} (model alone {rep['macro_crossfit']:.4f})")

    if WRITE and best[1] > 0:
        lam = best[1]; tag = "p5_" + "_".join(t.replace("p5_", "") for t in DIAG_RUN)[:100] + "_knn"
        Ldv = logit((1 - lam) * Pdv_m + lam * Pdv_k)
        Lte = logit((1 - lam) * run_probs("test", "test", te_idx) + lam * knn_P(Ste, Ite, K, p))
        r, thr_b = dev_report(Ldv, Ydv)
        torch.save({"ids": [p5_id("dev", i) for i in dv_idx], "logits": Ldv.astype(np.float32), "labels": LABELS}, f"/kaggle/working/devlogits_{tag}.pt")
        torch.save({"ids": [p5_id("test", i) for i in te_idx], "logits": Lte.astype(np.float32), "labels": LABELS}, f"/kaggle/working/testlogits_{tag}.pt")
        write_submission5(sig(Lte), thr_b, ml, tag)


### 5.8 Retrieval v2 + TF-IDF linear model + 3-way blend (`P5_MODE = "diag2"`)

In [ ]:
if PHASE == 5 and P5_MODE == "diag2":
    # ===================== 5.8  RETRIEVAL v2 + TF-IDF LINEAR MODEL + 3-WAY BLEND (CPU is enough; plumbing) =====================
    # 1) kNN whose index also holds labelled DEV cases (dev queries only see the OTHER half of dev; test sees train + all dev)
    # 2) a second lexical model: one logistic regression per statute on the same TF-IDF features
    # 3) blend  w_model * neural + w_knn * kNN + w_lin * linear  on a weight grid; every score = honest cross-fitted dev macro-F1
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.multiclass import OneVsRestClassifier
    MODEL_RUNS   = ["p5_arch3_full_tapt"]   # finished neural runs (attached); their probabilities are averaged
    KNN_K        = (50, 100, 200)           # YOUR grid: neighbours
    KNN_POW      = (4, 8, 16)               # YOUR grid: neighbour weight = similarity ** power
    DEV_IN_INDEX = True                     # also retrieve from labelled dev cases (never a case's own half)
    LIN_C        = 4.0                      # YOUR value: inverse regularisation strength of each per-statute logistic regression
    STEP         = 0.1                      # grid step for the 3 blend weights (they sum to 1)
    WRITE        = True                     # write the best blend's submission + logits

    mp, lo, hi, ml = HP["THR_MIN_POS"], HP["THR_LO"], HP["THR_HI"], HP["MIN_LABELS"]
    tr_idx, dv_idx, te_idx = p5_items("train"), p5_items("dev"), p5_items("test")
    Ytr, Ydv = gold_matrix("train", tr_idx), gold_matrix("dev", dv_idx)
    n_tr, n_dv = len(tr_idx), len(dv_idx)
    def logit(P): P = np.clip(P, 1e-6, 1 - 1e-6); return np.log(P / (1 - P))
    def cf(P): return crossfit_macro(P, Ydv, mp, lo, hi, ml)          # honest score of a probability matrix on dev

    # ---- neural model(s) ----
    def run_probs(kind, split, idx):
        out = []
        for t in MODEL_RUNS:
            d = torch.load(find_input(f"{kind}logits_{t}.pt"), weights_only=False)
            assert list(d["labels"]) == list(LABELS), f"{t}: label order differs"
            pos = {k: r for r, k in enumerate(d["ids"])}
            out.append(sig(np.stack([np.asarray(d["logits"])[pos[p5_id(split, i)]] for i in idx])))
        return np.mean(out, 0)
    Pdv_m, Pte_m = run_probs("dev", "dev", dv_idx), run_probs("test", "test", te_idx)
    print(f"neural {'+'.join(MODEL_RUNS)}: cross-fitted dev macro {cf(Pdv_m):.4f}")

    # ---- TF-IDF (fitted on TRAIN only) ----
    t0 = time.time()
    def docs(split, idx): return [" ".join(map(str, doc_tokens5(split, i)[:4000].tolist())) for i in idx]
    vec = TfidfVectorizer(token_pattern=r"\S+", ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=500000, dtype=np.float32)
    Xtr = vec.fit_transform(docs("train", tr_idx)); Xdv = vec.transform(docs("dev", dv_idx)); Xte = vec.transform(docs("test", te_idx))
    print(f"TF-IDF in {time.time() - t0:.0f}s")

    # ---- kNN with train (+ dev) as the index ----
    t0 = time.time(); Kmax = max(KNN_K)
    fold = np.random.RandomState(1).permutation(n_dv) % 2          # dev halves for retrieval
    Yidx = np.vstack([Ytr, Ydv])                                    # label rows: train first, then dev (offset n_tr)
    def topk(Xq, Xi, K, offset=0, qfold=None, bs=1000):
        S_all, I_all = [], []
        for a in range(0, Xq.shape[0], bs):
            S = (Xq[a:a + bs] @ Xi.T).toarray()
            if qfold is not None: S[qfold[a:a + bs, None] == fold[None, :]] = -1.0     # a dev query never sees its own half (or itself)
            I = np.argpartition(-S, K, axis=1)[:, :K]
            S_all.append(np.take_along_axis(S, I, 1)); I_all.append(I + offset)
        return np.vstack(S_all), np.vstack(I_all)
    def merge(parts, K):
        S = np.hstack([p[0] for p in parts]); I = np.hstack([p[1] for p in parts])
        o = np.argsort(-S, 1)[:, :K]
        return np.take_along_axis(S, o, 1), np.take_along_axis(I, o, 1)
    dv_parts = [topk(Xdv, Xtr, Kmax)]; te_parts = [topk(Xte, Xtr, Kmax)]
    if DEV_IN_INDEX:
        dv_parts.append(topk(Xdv, Xdv, min(Kmax, n_dv // 2 - 1), offset=n_tr, qfold=fold))
        te_parts.append(topk(Xte, Xdv, min(Kmax, n_dv - 1), offset=n_tr))
    Sdv, Idv = merge(dv_parts, Kmax); Ste, Ite = merge(te_parts, Kmax)
    print(f"neighbours in {time.time() - t0:.0f}s | index = train" + (" + dev" if DEV_IN_INDEX else ""))
    def knn_P(S, I, K, p):
        W = (np.maximum(S[:, :K], 0) ** p + 1e-9).astype(np.float32)
        return (W[:, :, None] * Yidx[I[:, :K]]).sum(1) / W.sum(1, keepdims=True)
    best_knn = None
    for K in KNN_K:
        for p in KNN_POW:
            s = cf(knn_P(Sdv, Idv, K, p)); print(f"kNN K={K:3d} pow={p:2d}: cross-fitted {s:.4f}")
            if best_knn is None or s > best_knn[0]: best_knn = (s, K, p)
    _, K, p = best_knn; Pdv_k, Pte_k = knn_P(Sdv, Idv, K, p), knn_P(Ste, Ite, K, p)

    # ---- TF-IDF logistic regression, one per statute (trained on TRAIN) ----
    t0 = time.time()
    lin = OneVsRestClassifier(LogisticRegression(C=LIN_C, solver="liblinear", max_iter=300), n_jobs=-1).fit(Xtr, Ytr)
    Pdv_l, Pte_l = lin.predict_proba(Xdv), lin.predict_proba(Xte)
    print(f"linear model in {time.time() - t0:.0f}s | cross-fitted {cf(Pdv_l):.4f}")

    # ---- 3-way blend on a weight grid ----
    t0 = time.time(); res = []
    g = np.round(np.arange(0, 1 + 1e-9, STEP), 4)
    for wm in g:
        for wk in g:
            wl = round(1 - wm - wk, 4)
            if wl < -1e-9: continue
            res.append((cf(wm * Pdv_m + wk * Pdv_k + wl * Pdv_l), wm, wk, wl))
    res.sort(key=lambda r: r[0], reverse=True)
    print(f"blend grid ({len(res)} weightings) in {time.time() - t0:.0f}s | top 5:")
    for s, wm, wk, wl in res[:5]: print(f"    {s:.4f}  neural {wm:.1f} | kNN {wk:.1f} (K={K}, pow={p}) | linear {wl:.1f}")
    s, wm, wk, wl = res[0]
    print(f"BEST: cross-fitted dev macro {s:.4f}  (neural alone {cf(Pdv_m):.4f})")

    if WRITE:
        tag = "p5_" + "_".join(t.replace("p5_", "") for t in MODEL_RUNS)[:90] + "_knnlin"
        Ldv = logit(wm * Pdv_m + wk * Pdv_k + wl * Pdv_l); Lte = logit(wm * Pte_m + wk * Pte_k + wl * Pte_l)
        r, thr_b = dev_report(Ldv, Ydv)
        print("final dev (thresholds tuned on all of dev):", {k_: (round(v, 4) if isinstance(v, float) else v) for k_, v in r.items()})
        torch.save({"ids": [p5_id("dev", i) for i in dv_idx], "logits": Ldv.astype(np.float32), "labels": LABELS}, f"/kaggle/working/devlogits_{tag}.pt")
        torch.save({"ids": [p5_id("test", i) for i in te_idx], "logits": Lte.astype(np.float32), "labels": LABELS}, f"/kaggle/working/testlogits_{tag}.pt")
        write_submission5(sig(Lte), thr_b, ml, tag)


### 5.9 More levers: per-statute stacking; linear model also trained on dev; re-blend (`P5_MODE = "diag2"`)

In [ ]:
if PHASE == 5 and P5_MODE == "diag2":
    # ===================== 5.9  MORE LEVERS on top of 5.8 (stacking; linear model also trained on dev) =====================
    RUN_STACK      = True    # per-statute stacking of neural / kNN / linear (~5 min)
    RUN_LINEAR_DEV = True    # linear model trained on train + dev (~90 min), then re-blend and re-stack
    np.savez("/kaggle/working/lexical_probs.npz", Pdv_k=Pdv_k, Pte_k=Pte_k, Pdv_l=Pdv_l, Pte_l=Pte_l)   # backup of 5.8's predictions
    # helpers (plumbing)
    from sklearn.linear_model import LogisticRegression
    from sklearn.multiclass import OneVsRestClassifier
    from scipy.sparse import vstack as svstack
    BEST_SO_FAR = res[0][0]                              # 5.8's best cross-fitted dev macro (0.4187)

    def save_and_write(Ldv, Lte, tag):
        r, thr_b = dev_report(Ldv, Ydv)
        torch.save({"ids": [p5_id("dev", i) for i in dv_idx], "logits": Ldv.astype(np.float32), "labels": LABELS}, f"/kaggle/working/devlogits_{tag}.pt")
        torch.save({"ids": [p5_id("test", i) for i in te_idx], "logits": Lte.astype(np.float32), "labels": LABELS}, f"/kaggle/working/testlogits_{tag}.pt")
        write_submission5(sig(Lte), thr_b, ml, tag)

    def blend_grid(Pd_m, Pd_k, Pd_l, Pt_m, Pt_k, Pt_l, tag):
        """fixed weights (sum 1) for neural / kNN / linear, chosen by cross-fitted dev macro-F1; writes a zip only if it beats BEST_SO_FAR"""
        global BEST_SO_FAR
        out = []
        for wm in g:
            for wk in g:
                wl = round(1 - wm - wk, 4)
                if wl < -1e-9: continue
                out.append((cf(wm * Pd_m + wk * Pd_k + wl * Pd_l), wm, wk, wl))
        out.sort(key=lambda r: r[0], reverse=True); s, wm, wk, wl = out[0]
        print(f"[blend {tag}] best {s:.4f}  neural {wm:.1f} | kNN {wk:.1f} | linear {wl:.1f}   (best so far {BEST_SO_FAR:.4f})")
        if s > BEST_SO_FAR:
            BEST_SO_FAR = s; save_and_write(logit(wm * Pd_m + wk * Pd_k + wl * Pd_l), logit(wm * Pt_m + wk * Pt_k + wl * Pt_l), tag)
        return s

    def stack(Pd_m, Pd_k, Pd_l, Pt_m, Pt_k, Pt_l, tag, C=1.0):
        """STACKING: per statute, a 3-input logistic regression on dev learns how much to trust neural / kNN / linear for THAT statute.
        Dev predictions are out-of-fold (each half scored by a model fitted on the other half) -> the score stays honest."""
        global BEST_SO_FAR
        def F(Pm, Pk, Pl, j): return np.stack([logit(Pm[:, j]), logit(Pk[:, j]), logit(Pl[:, j])], 1)
        sf = np.random.RandomState(2).permutation(n_dv) % 2
        OOF = np.zeros_like(Pd_m); PT = np.zeros_like(Pt_m)
        for j in range(L):
            for f in (0, 1):
                a, b = sf != f, sf == f
                m = LogisticRegression(C=C, max_iter=1000).fit(F(Pd_m[a], Pd_k[a], Pd_l[a], j), Ydv[a, j])
                OOF[b, j] = m.predict_proba(F(Pd_m[b], Pd_k[b], Pd_l[b], j))[:, 1]
            m = LogisticRegression(C=C, max_iter=1000).fit(F(Pd_m, Pd_k, Pd_l, j), Ydv[:, j])
            PT[:, j] = m.predict_proba(F(Pt_m, Pt_k, Pt_l, j))[:, 1]
        s = cf(OOF)
        print(f"[stack {tag}] out-of-fold cross-fitted {s:.4f}   (best so far {BEST_SO_FAR:.4f})")
        if s > BEST_SO_FAR:
            BEST_SO_FAR = s; save_and_write(logit(OOF), logit(PT), tag)
        return s

    def linear_with_dev(C, class_weight=None):
        """the linear model also trained on labelled DEV: dev scored out-of-fold (fit on train + other dev half), test from train + all dev"""
        lf = np.random.RandomState(3).permutation(n_dv) % 2
        Pd = np.zeros((n_dv, L)); t0 = time.time()
        mk = lambda: OneVsRestClassifier(LogisticRegression(C=C, class_weight=class_weight, solver="liblinear", max_iter=300), n_jobs=-1)
        for f in (0, 1):
            a = lf != f
            Pd[lf == f] = mk().fit(svstack([Xtr, Xdv[a]]), np.vstack([Ytr, Ydv[a]])).predict_proba(Xdv[lf == f])
            print(f"  half {f + 1}/2 done ({time.time() - t0:.0f}s)")
        Pt = mk().fit(svstack([Xtr, Xdv]), np.vstack([Ytr, Ydv])).predict_proba(Xte)
        print(f"linear (train + dev) C={C} cw={class_weight}: cross-fitted {cf(Pd):.4f} | train-only linear {cf(Pdv_l):.4f} | {time.time() - t0:.0f}s")
        return Pd, Pt

    if RUN_STACK:
        stack(Pdv_m, Pdv_k, Pdv_l, Pte_m, Pte_k, Pte_l, "p5_stack")
    if RUN_LINEAR_DEV:
        Pdv_l3, Pte_l3 = linear_with_dev(LIN_C)
        np.savez("/kaggle/working/linear_dev_probs.npz", Pdv_l3=Pdv_l3, Pte_l3=Pte_l3)
        blend_grid(Pdv_m, Pdv_k, Pdv_l3, Pte_m, Pte_k, Pte_l3, "p5_knnlin_dev")
        stack(Pdv_m, Pdv_k, Pdv_l3, Pte_m, Pte_k, Pte_l3, "p5_stack_dev")
    print(f"FINAL best cross-fitted dev macro {BEST_SO_FAR:.4f} -> upload the LAST submission_*.zip written above")


### 5.10 KD as neural member, 2nd TF-IDF view, 4-way blend + stacking (`P5_MODE = "diag3"`)
Needs the **phase5-diag2 output** (`lexical_probs.npz`, `linear_dev_probs.npz`), R5's and KD's `devlogits_`/`testlogits_`.

In [ ]:
if PHASE == 5 and P5_MODE == "diag3":
    # ===================== 5.10  MORE LEVERS: better neural member (KD), a 2nd TF-IDF view, 4-way blend + stacking (CPU; plumbing) =====================
    # Reuses diag2's saved predictions (attach the phase5-diag2 OUTPUT): lexical_probs.npz (kNN) + linear_dev_probs.npz (linear, train+dev)
    import itertools
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.multiclass import OneVsRestClassifier
    from scipy.sparse import vstack as svstack
    NEURAL_SETS = [["p5_arch3_full_tapt"], ["p5_arch4_kd"], ["p5_arch3_full_tapt", "p5_arch4_kd"]]   # YOUR candidates (their logits must be attached)
    RUN_LIN2  = True                       # 2nd linear model on a different TF-IDF view (~2 h); False = only the fast part
    LIN2_NGRAM, LIN2_MAXF = (1, 3), 1_000_000   # YOUR 2nd view: word n-gram range over BPE tokens, vocabulary size
    LIN2_C, LIN2_CW = 4.0, "balanced"      # YOUR values: inverse regularisation, class weighting
    STEP = 0.1                             # blend-weight grid step

    mp, lo, hi, ml = HP["THR_MIN_POS"], HP["THR_LO"], HP["THR_HI"], HP["MIN_LABELS"]
    tr_idx, dv_idx, te_idx = p5_items("train"), p5_items("dev"), p5_items("test")
    Ytr, Ydv = gold_matrix("train", tr_idx), gold_matrix("dev", dv_idx); n_dv = len(dv_idx)
    def logit(P): P = np.clip(P, 1e-6, 1 - 1e-6); return np.log(P / (1 - P))
    def cf(P): return crossfit_macro(P, Ydv, mp, lo, hi, ml)

    lex = np.load(find_input("lexical_probs.npz")); Pdv_k, Pte_k = lex["Pdv_k"], lex["Pte_k"]
    lin = np.load(find_input("linear_dev_probs.npz")); Pdv_l, Pte_l = lin["Pdv_l3"], lin["Pte_l3"]
    assert Pdv_k.shape == Pdv_l.shape == (n_dv, L) and Pte_k.shape == Pte_l.shape == (len(te_idx), L), "saved predictions do not match this data"
    print(f"loaded: kNN cross-fitted {cf(Pdv_k):.4f} | linear(train+dev) cross-fitted {cf(Pdv_l):.4f}")

    def neural(runs, kind, split, idx):
        out = []
        for t in runs:
            d = torch.load(find_input(f"{kind}logits_{t}.pt"), weights_only=False)
            assert list(d["labels"]) == list(LABELS), f"{t}: label order differs"
            pos = {k: r for r, k in enumerate(d["ids"])}
            out.append(sig(np.stack([np.asarray(d["logits"])[pos[p5_id(split, i)]] for i in idx])))
        return np.mean(out, 0)

    BEST = {"s": -1.0}
    def save_and_write(Ldv, Lte, tag):
        r, thr_b = dev_report(Ldv, Ydv)
        torch.save({"ids": [p5_id("dev", i) for i in dv_idx], "logits": Ldv.astype(np.float32), "labels": LABELS}, f"/kaggle/working/devlogits_{tag}.pt")
        torch.save({"ids": [p5_id("test", i) for i in te_idx], "logits": Lte.astype(np.float32), "labels": LABELS}, f"/kaggle/working/testlogits_{tag}.pt")
        write_submission5(sig(Lte), thr_b, ml, tag)
    def consider(s, Pd, Pt, tag, what):
        print(f"  {s:.4f}  {what}" + ("   <- new best, zip written" if s > BEST["s"] else ""))
        if s > BEST["s"]:
            BEST.update(s=s, tag=tag); save_and_write(logit(Pd), logit(Pt), tag)
    def blend(comps_d, comps_t, names, tag):
        n = len(comps_d); steps = int(round(1 / STEP)); out = []
        for c in itertools.product(range(steps + 1), repeat=n - 1):
            if sum(c) > steps: continue
            w = np.array(list(c) + [steps - sum(c)]) / steps
            out.append((cf(sum(wi * P for wi, P in zip(w, comps_d))), w))
        out.sort(key=lambda r: r[0], reverse=True); s, w = out[0]
        consider(s, sum(wi * P for wi, P in zip(w, comps_d)), sum(wi * P for wi, P in zip(w, comps_t)), tag,
                 "blend " + " | ".join(f"{nm} {wi:.1f}" for nm, wi in zip(names, w)))
    def stack(comps_d, comps_t, names, tag, C=1.0):
        sf = np.random.RandomState(2).permutation(n_dv) % 2
        OOF = np.zeros((n_dv, L)); PT = np.zeros((len(te_idx), L))
        for j in range(L):
            Fd = np.stack([logit(P[:, j]) for P in comps_d], 1); Ft = np.stack([logit(P[:, j]) for P in comps_t], 1)
            for f in (0, 1):
                a, b = sf != f, sf == f
                OOF[b, j] = LogisticRegression(C=C, max_iter=1000).fit(Fd[a], Ydv[a, j]).predict_proba(Fd[b])[:, 1]
            PT[:, j] = LogisticRegression(C=C, max_iter=1000).fit(Fd, Ydv[:, j]).predict_proba(Ft)[:, 1]
        consider(cf(OOF), OOF, PT, tag, "stack " + " + ".join(names))

    # ---- fast part: which neural member(s) help most next to kNN + linear ----
    t0 = time.time(); NEU = {}
    for runs in NEURAL_SETS:
        nm = "+".join(r.replace("p5_", "") for r in runs)
        NEU[nm] = (neural(runs, "dev", "dev", dv_idx), neural(runs, "test", "test", te_idx))
        print(f"[neural {nm}] alone cross-fitted {cf(NEU[nm][0]):.4f}")
        blend([NEU[nm][0], Pdv_k, Pdv_l], [NEU[nm][1], Pte_k, Pte_l], [nm, "kNN", "linear"], f"p5_d3_{nm.replace('+', '_')}_blend")
    for nm in NEU:
        stack([NEU[nm][0], Pdv_k, Pdv_l], [NEU[nm][1], Pte_k, Pte_l], [nm, "kNN", "linear"], f"p5_d3_{nm.replace('+', '_')}_stack")
    print(f"fast part done in {time.time() - t0:.0f}s | best so far {BEST['s']:.4f} ({BEST.get('tag')})")

    # ---- slow part: 2nd TF-IDF view + its own linear model (also trained on dev, scored out-of-fold) ----
    if RUN_LIN2:
        t0 = time.time()
        def docs(split, idx): return [" ".join(map(str, doc_tokens5(split, i)[:4000].tolist())) for i in idx]
        vec2 = TfidfVectorizer(token_pattern=r"\S+", ngram_range=LIN2_NGRAM, min_df=2, sublinear_tf=True, max_features=LIN2_MAXF, dtype=np.float32)
        X2tr = vec2.fit_transform(docs("train", tr_idx)); X2dv = vec2.transform(docs("dev", dv_idx)); X2te = vec2.transform(docs("test", te_idx))
        print(f"TF-IDF view 2 {LIN2_NGRAM}, {X2tr.shape[1]:,} features in {time.time() - t0:.0f}s")
        mk = lambda: OneVsRestClassifier(LogisticRegression(C=LIN2_C, class_weight=LIN2_CW, solver="liblinear", max_iter=300), n_jobs=-1)
        lf = np.random.RandomState(3).permutation(n_dv) % 2; Pdv_l2 = np.zeros((n_dv, L))
        for f in (0, 1):
            a = lf != f
            Pdv_l2[lf == f] = mk().fit(svstack([X2tr, X2dv[a]]), np.vstack([Ytr, Ydv[a]])).predict_proba(X2dv[lf == f])
            print(f"  linear-2 half {f + 1}/2 done ({time.time() - t0:.0f}s)")
        Pte_l2 = mk().fit(svstack([X2tr, X2dv]), np.vstack([Ytr, Ydv])).predict_proba(X2te)
        np.savez("/kaggle/working/linear2_probs.npz", Pdv_l2=Pdv_l2, Pte_l2=Pte_l2)
        print(f"linear-2 cross-fitted {cf(Pdv_l2):.4f} | {time.time() - t0:.0f}s")
        for nm in NEU:
            blend([NEU[nm][0], Pdv_k, Pdv_l, Pdv_l2], [NEU[nm][1], Pte_k, Pte_l, Pte_l2], [nm, "kNN", "linear", "linear-2"], f"p5_d3_{nm.replace('+', '_')}_blend4")
            stack([NEU[nm][0], Pdv_k, Pdv_l, Pdv_l2], [NEU[nm][1], Pte_k, Pte_l, Pte_l2], [nm, "kNN", "linear", "linear-2"], f"p5_d3_{nm.replace('+', '_')}_stack4")
    print(f"FINAL best cross-fitted dev macro {BEST['s']:.4f} -> upload submission_{BEST.get('tag')}.zip")


# PHASE 6 — Abstractive Summarization
Run order: 6.1 (data + diagnostics, CPU is enough).

### 6.0 Input-selection helpers (extract-then-abstract)

In [ ]:
if PHASE == 6:
    # ===================== 6.0  Input-selection helpers (plumbing; used by 6.1 report and 6.2) =====================
    # Extract-then-abstract: the model sees only SRC_LEN tokens, so WHICH sentences enter the prefix matters.
    #   "lead"      first words of the document
    #   "lead_tail" half from the start, half from the end
    #   "extract"   unsupervised: sentences closest (TF-IDF cosine) to the document's own centroid
    #   "oracle"    supervised: a logistic-regression sentence scorer trained to predict which sentences
    #               a greedy ROUGE-style oracle picks against the reference summary (Nallapati et al. 2017;
    #               extract-then-abstract with a Transformer LM: Pilault et al. 2020). Trained on training docs only.
    from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
    from sklearn.linear_model import LogisticRegression
    import scipy.sparse as sp
    SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")
    STOP6 = set(ENGLISH_STOP_WORDS)
    def words_of(s): return re.findall(r"[a-z0-9]+", s.lower())
    def content_words(s): return [w for w in words_of(s) if w not in STOP6 and len(w) > 1]

    def split_sents(doc, max_words=60):
        out = []
        for s in SENT_SPLIT.split(doc):
            w = s.split()
            if not w: continue
            for i in range(0, len(w), max_words):          # very long "sentences" (no full stops) cut into pieces
                out.append(" ".join(w[i:i + max_words]))
        return out

    def oracle_labels(sents, summary, max_ratio=3.0):
        """Greedy oracle: repeatedly add the sentence that most improves (unigram F + bigram F) of the selected set
        against the summary (set-based ROUGE-1/2 approximation); stop when nothing improves."""
        sw = content_words(summary); Su, Sb = set(sw), set(zip(sw, sw[1:]))
        if not Su: return np.zeros(len(sents), dtype=np.int8)
        U = [set(content_words(s)) for s in sents]
        B = [set(zip(c, c[1:])) for c in (content_words(s) for s in sents)]
        selU, selB, covU, covB = set(), set(), 0, 0
        chosen, best_f, words, limit = [], 0.0, 0, max_ratio * len(summary.split())
        def f(cov, tot, ref): return 0.0 if cov == 0 else 2 * (cov / tot) * (cov / ref) / (cov / tot + cov / ref)
        while words < limit:
            cand, cand_f = None, best_f
            for i in range(len(sents)):
                if i in chosen: continue
                nu, nb = U[i] - selU, B[i] - selB
                cu, cb = covU + len(nu & Su), covB + len(nb & Sb)
                tu, tb = len(selU) + len(nu), len(selB) + len(nb)
                sc = f(cu, max(1, tu), len(Su)) + f(cb, max(1, tb), max(1, len(Sb)))
                if sc > cand_f: cand, cand_f = i, sc
            if cand is None: break
            chosen.append(cand); best_f = cand_f
            selU |= U[cand]; selB |= B[cand]
            covU, covB = len(selU & Su), len(selB & Sb); words += len(sents[cand].split())
        y = np.zeros(len(sents), dtype=np.int8); y[chosen] = 1
        return y

    class Selector:
        def __init__(self, mode): self.mode, self.vec, self.clf = mode, None, None

        def fit(self, docs, max_docs=3000, seed=0):
            if self.mode in ("lead", "lead_tail"): return self
            t0 = time.time()
            self.vec = TfidfVectorizer(sublinear_tf=True, min_df=2, max_features=200000, stop_words="english").fit(d["document"] for d in docs)
            print(f"  selector: TF-IDF on {len(docs)} training docs ({time.time() - t0:.0f}s)")
            if self.mode == "oracle":
                Xs, ys = [], []
                for d in random.Random(seed).sample(docs, min(max_docs, len(docs))):
                    sents = split_sents(d["document"])
                    if len(sents) < 3: continue
                    Xs.append(self._feats(sents)); ys.append(oracle_labels(sents, d["summary"]))
                X, y = sp.vstack(Xs).tocsr(), np.concatenate(ys)
                self.clf = LogisticRegression(solver="liblinear", max_iter=1000).fit(X, y)
                print(f"  selector: oracle scorer trained on {len(Xs)} docs / {X.shape[0]:,} sentences "
                      f"({y.mean():.1%} oracle-positive) in {time.time() - t0:.0f}s")
            return self

        def _feats(self, sents):
            M = self.vec.transform(sents)
            c = np.asarray(M.mean(axis=0)).ravel()
            cen = (M @ c) / (np.sqrt(M.multiply(M).sum(1)).A.ravel() * (np.linalg.norm(c) + 1e-9) + 1e-9)
            n = len(sents); pos = np.zeros((n, 10)); pos[np.arange(n), np.minimum(9, (np.arange(n) * 10) // n)] = 1
            ln = np.log1p([len(s.split()) for s in sents])[:, None]
            return sp.hstack([M, sp.csr_matrix(np.hstack([pos, ln, cen[:, None]]))]).tocsr()

        def scores(self, sents):
            if self.mode == "extract":
                F_ = self._feats(sents); return F_[:, -1].toarray().ravel()      # centrality column
            return self.clf.decision_function(self._feats(sents))

        def select(self, doc, word_budget):
            """-> (head_text, tail_text or None), about word_budget words in total, original order kept."""
            w = doc.split()
            if len(w) <= word_budget: return doc, None
            if self.mode == "lead": return " ".join(w[:int(word_budget * 1.2)]), None
            if self.mode == "lead_tail":
                h = word_budget // 2
                return " ".join(w[:int(h * 1.2)]), " ".join(w[-int(h * 1.2):])
            sents = split_sents(doc)
            if len(sents) < 3: return " ".join(w[:int(word_budget * 1.2)]), None
            sc = self.scores(sents)
            keep, used = [], 0
            for i in np.argsort(-sc):
                n = len(sents[i].split())
                if used + n > word_budget and keep: continue
                keep.append(i); used += n
                if used >= word_budget * 0.97: break
            return " ".join(sents[i] for i in sorted(keep)), None

    def coverage(text, summary, first_words=None):
        """share of the summary's distinct content words that appear in `text` (optionally: only the summary's first N words)"""
        s = summary if first_words is None else " ".join(summary.split()[:first_words])
        S = set(content_words(s))
        return len(S & set(content_words(text))) / max(1, len(S))


### 6.1 Data, lengths, coverage diagnostics, lead baselines

In [ ]:
if PHASE == 6 and P6_MODE == "data":
    # ===================== 6.1  Data, lengths, coverage diagnostics (CPU; plumbing) =====================
    import hashlib
    RUN_BASELINES = True      # lead-N extractive baselines scored with ROUGE (+ BERTScore if available) on a held-out sample
    N_EVAL        = 300       # held-out train documents used for the baseline scores

    def read_json_any(path):
        try: return json.load(open(path))
        except json.JSONDecodeError: return [json.loads(l) for l in open(path) if l.strip()]
    def to_records(obj):
        if isinstance(obj, dict):
            vals = list(obj.values())
            if vals and all(isinstance(v, dict) for v in vals): return [dict(id=k, **v) for k, v in obj.items()]
            return next(v for v in vals if isinstance(v, list))
        return obj
    def pick(keys, cands):
        for c in cands:
            if c in keys: return c
        return None
    md5 = lambda s: hashlib.md5(re.sub(r"\s+", " ", s).strip().lower().encode()).hexdigest()
    def pct(xs, q): xs = sorted(xs); return xs[min(len(xs) - 1, int(q * len(xs)))]

    RAW6 = {}
    for split, key in (("train", "Train/train.json"), ("test", "Test/test.json")):
        recs = to_records(read_json_any(P[key])); keys = set(recs[0])
        ID_K, DOC_K, SUM_K = pick(keys, ("id", "doc_id", "case_id")), pick(keys, ("document", "text", "judgment")), pick(keys, ("summary", "abstract"))
        out = []
        for r in recs:
            d = r[DOC_K]; d = " ".join(map(str, d)) if isinstance(d, list) else str(d)
            item = {"id": r[ID_K], "document": d}
            if split == "train":
                s = r[SUM_K]; item["summary"] = " ".join(map(str, s)) if isinstance(s, list) else str(s)
            out.append(item)
        RAW6[split] = out
        print(f"{split:5s}: {len(out)} docs | id='{ID_K}' document='{DOC_K}' summary='{SUM_K}' | other fields {sorted(keys - {ID_K, DOC_K, SUM_K})}")
    for s in RAW6:
        ids = [str(d["id"]) for d in RAW6[s]]
        print(f"{s:5s}: unique ids {len(set(ids))}/{len(ids)} | unique documents {len({md5(d['document']) for d in RAW6[s]})}")
    print("train-test shared ids:", len({str(d['id']) for d in RAW6['train']} & {str(d['id']) for d in RAW6['test']}),
          "| identical documents:", len({md5(d['document']) for d in RAW6['train']} & {md5(d['document']) for d in RAW6['test']}))

    # ---- lengths (words, and BPE tokens on a sample) ----
    rng = random.Random(SEED)
    for s in RAW6:
        w = [len(d["document"].split()) for d in RAW6[s]]
        print(f"{s:5s} document words: p10 {pct(w, .1)} p50 {pct(w, .5)} p90 {pct(w, .9)} p99 {pct(w, .99)} max {max(w)}")
    sw = [len(d["summary"].split()) for d in RAW6["train"]]
    print(f"train summary words: p10 {pct(sw, .1)} p50 {pct(sw, .5)} p90 {pct(sw, .9)} max {max(sw)}")
    ratio = [len(d["summary"].split()) / max(1, len(d["document"].split())) for d in RAW6["train"]]
    print(f"summary/document length ratio: p10 {pct(ratio, .1):.3f} p50 {pct(ratio, .5):.3f} p90 {pct(ratio, .9):.3f}")
    smp = rng.sample(RAW6["train"], min(200, len(RAW6["train"])))
    dt = [len(tok.encode(d["document"])) for d in smp]; st = [len(tok.encode(d["summary"])) for d in smp]
    print(f"BPE tokens (sample of {len(smp)}): document p50 {pct(dt, .5)} p90 {pct(dt, .9)} max {max(dt)} | summary p50 {pct(st, .5)} p90 {pct(st, .9)} max {max(st)}")
    for ctx in (1024,):
        fit = np.mean([a + b + 1 <= ctx for a, b in zip(dt, st)])
        print(f"document + summary fit in {ctx} tokens: {fit:.1%} of sampled training pairs")

    # ---- where in the document does the summary's content come from? (word overlap per tenth of the document) ----
    def toks(s): return re.findall(r"[a-z0-9]+", s.lower())
    stop = set("the of and to in a is that for on by with as was be it this which or an are at from has have not were been had his her their its said".split())
    cover = np.zeros(10); n = 0
    for d in rng.sample(RAW6["train"], min(1000, len(RAW6["train"]))):
        dw = toks(d["document"]); sv = {w for w in toks(d["summary"]) if w not in stop}
        if len(dw) < 50 or not sv: continue
        for k, part in enumerate(np.array_split(np.array(dw, dtype=object), 10)):
            cover[k] += len(sv & set(part.tolist())) / len(sv)
        n += 1
    print("share of summary content words found in each tenth of the document (start -> end):")
    print("   " + " ".join(f"{c / n:.2f}" for c in cover))

    # ---- extractive lead baselines on held-out training docs: floor scores + effect of length ----
    if RUN_BASELINES:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score", "bert-score"], check=False)
        from rouge_score import rouge_scorer
        ev = rng.sample(RAW6["train"], min(N_EVAL, len(RAW6["train"])))
        refs = [d["summary"] for d in ev]
        med = int(np.median(sw))
        cands = {f"lead-{k}w": [" ".join(d["document"].split()[:k]) for d in ev] for k in (med // 2, med, 2 * med)}
        sc = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
        try:
            from bert_score import score as bscore
            have_bs = True
        except Exception as e:
            have_bs = False; print("BERTScore unavailable:", e)
        for name, hyp in cands.items():
            r = [sc.score(a, b) for a, b in zip(refs, hyp)]
            line = f"{name:10s}: ROUGE-1 F {np.mean([x['rouge1'].fmeasure for x in r]):.4f} | ROUGE-L F {np.mean([x['rougeL'].fmeasure for x in r]):.4f}"
            if have_bs:
                P_, R_, F_ = bscore(hyp, refs, lang="en", verbose=False, batch_size=16, device=DEV_TYPE)
                line += f" | BERTScore F1 {F_.mean().item():.4f} (P {P_.mean().item():.3f} R {R_.mean().item():.3f})"
            print(line)

    # ---- input selection report: how much of the summary's content can each selection put in front of the model? ----
    # (held-out sample of up to 300 training docs; the oracle scorer is trained on the OTHER training docs)
    RUN_SELECT_REPORT = True
    if RUN_SELECT_REPORT:
        pool = RAW6["train"][:]; random.Random(1).shuffle(pool)
        n_rep = min(300, len(pool) // 4); rep, fit_docs = pool[:n_rep], pool[n_rep:]
        tpw = sum(len(tok.encode(" ".join(d["document"].split()[:1500]))) for d in rep[:40]) / sum(min(1500, len(d["document"].split())) for d in rep[:40])
        print(f"BPE tokens per document word: {tpw:.2f}  (word budget = SRC_LEN / {tpw:.2f})")
        orc = Selector("oracle").fit(fit_docs)
        ext = Selector("extract"); ext.vec = orc.vec
        sels = {"lead": Selector("lead"), "lead_tail": Selector("lead_tail"), "extract": ext, "oracle": orc}
        full = np.mean([coverage(d["document"], d["summary"]) for d in rep])
        full4 = np.mean([coverage(d["document"], d["summary"], 400) for d in rep])
        print(f"summary content words present ANYWHERE in the document: {full:.3f} (first 400 summary words: {full4:.3f})")
        print("coverage of summary content words by the selected input   [all summary words | first 400 summary words]")
        for budget in (250, 400, 600):
            row = []
            for name, s in sels.items():
                cv = [(coverage(" ".join(t for t in s.select(d["document"], budget) if t), d["summary"]),
                       coverage(" ".join(t for t in s.select(d["document"], budget) if t), d["summary"], 400)) for d in rep]
                row.append(f"{name} {np.mean([a for a, _ in cv]):.3f} | {np.mean([b for _, b in cv]):.3f}")
            print(f"  {budget:4d} words:  " + "   ".join(row))


### 6.2 Held-out split, input selection, tokenization (cached)

In [ ]:
if PHASE == 6 and P6_MODE in ("train", "predict", "combine"):
    # ===================== 6.2  Held-out split + input selection + tokenization (plumbing) =====================
    # Every slot below is YOURS in the CONTROL cell (P6 = dict(...)). This cell only does what you chose.
    import hashlib, pickle
    need6("VAL_N")
    CTX = 1024

    def _read(path):
        try: obj = json.load(open(path))
        except json.JSONDecodeError: obj = [json.loads(l) for l in open(path) if l.strip()]
        if isinstance(obj, dict):
            vals = list(obj.values())
            obj = [dict(id=k, **v) for k, v in obj.items()] if vals and all(isinstance(v, dict) for v in vals) \
                  else next(v for v in vals if isinstance(v, list))
        out = []
        for r in obj:
            idk = next(k for k in ("id", "doc_id", "case_id") if k in r)
            dk = next(k for k in ("document", "text", "judgment") if k in r)
            d = r[dk]; d = " ".join(map(str, d)) if isinstance(d, list) else str(d)
            item = {"id": r[idk], "document": d}
            sk = next((k for k in ("summary", "abstract") if k in r), None)
            if sk:
                s = r[sk]; item["summary"] = " ".join(map(str, s)) if isinstance(s, list) else str(s)
            out.append(item)
        return out

    train_all, test6 = _read(P["Train/train.json"]), _read(P["Test/test.json"])
    # held-out split: fixed seed (same split in every run with the same VAL_N, so held-out scores are comparable);
    # exact-duplicate documents are kept on the same side so they cannot leak across the split
    _h = lambda s: hashlib.md5(re.sub(r"\s+", " ", s).strip().lower().encode()).hexdigest()
    groups = collections.defaultdict(list)
    for d in train_all: groups[_h(d["document"])].append(d)
    keys = sorted(groups); random.Random(1234).shuffle(keys)
    val6, tr6 = [], []
    for k in keys: (val6 if len(val6) < P6["VAL_N"] else tr6).extend(groups[k])
    print(f"train {len(tr6)} | held-out {len(val6)} | test {len(test6)} | duplicate-document groups {sum(len(g) > 1 for g in groups.values())}")

if PHASE == 6 and P6_MODE in ("train", "predict"):
    need6("SELECT", "SRC_LEN", "TGT_LEN", "PREFIX_BIDIR")
    assert P6["SRC_LEN"] + 1 + P6["TGT_LEN"] + 1 <= CTX, "SRC_LEN + 1 (separator) + TGT_LEN + 1 (EOS) must fit in 1024"
    assert P6["SELECT"] in ("lead", "lead_tail", "extract", "oracle"), "SELECT must be 'lead', 'lead_tail', 'extract' or 'oracle'"

    # tokens per word (turns the BPE budget SRC_LEN into a word budget for selection)
    _s = random.Random(0).sample(tr6, min(40, len(tr6)))
    TPW = sum(len(tok.encode(" ".join(d["document"].split()[:1500]))) for d in _s) / sum(min(1500, len(d["document"].split())) for d in _s)
    WORD_BUDGET = int(P6["SRC_LEN"] / TPW)
    print(f"BPE tokens per word {TPW:.2f} -> SRC_LEN {P6['SRC_LEN']} tokens ~ {WORD_BUDGET} words")

    def encode_item(d, sel):
        head, tail = sel.select(d["document"], WORD_BUDGET)
        if tail is None: src = tok.encode(head)[:P6["SRC_LEN"]]
        else:
            half = P6["SRC_LEN"] // 2
            src = tok.encode(head)[:half] + tok.encode(tail)[-(P6["SRC_LEN"] - half):]
        item = {"id": d["id"], "src": src}
        if "summary" in d:
            t = tok.encode(d["summary"])
            item["tgt"] = t + [EOS_ID] if len(t) <= P6["TGT_LEN"] else t[:P6["TGT_LEN"]]   # EOS only when the whole summary fits
            item["ref"] = d["summary"]
        return item

    cache_name = f"p6data_{P6['SELECT']}_{P6['SRC_LEN']}_{P6['TGT_LEN']}_v{P6['VAL_N']}.pkl"
    hit = glob.glob(f"/kaggle/input/**/{cache_name}", recursive=True)
    if hit:
        D6 = pickle.load(open(hit[0], "rb")); print("loaded tokenized data from", hit[0])
    else:
        t0 = time.time()
        SEL6 = Selector(P6["SELECT"]).fit(tr6)                 # fitted on the training split only (no held-out / test summaries)
        D6 = {}
        for name, docs in (("train", tr6), ("val", val6), ("test", test6)):
            D6[name] = [encode_item(d, SEL6) for d in docs]
            print(f"  tokenized {name}: {len(docs)} docs ({time.time() - t0:.0f}s so far)", flush=True)
        pickle.dump(D6, open(cache_name, "wb"))
        print("saved", cache_name, "(attach this notebook's output to later runs with the same settings to skip this step)")
    print(f"held-out coverage of summary content by the model's input: "
          f"{np.mean([coverage(tok.decode([t for t in it['src'] if t < tok.vocab_size]), it['ref']) for it in D6['val'][:200]]):.3f}")
    _fit = np.mean([it["tgt"][-1] == EOS_ID for it in D6["train"]])
    print(f"src tokens p50 {pct([len(i['src']) for i in D6['train']], .5)} | tgt tokens p50 {pct([len(i['tgt']) for i in D6['train']], .5)} "
          f"| whole summary fits in TGT_LEN: {_fit:.1%} of training pairs")


### 6.3 YOUR CODE — third technique, KD student, KD loss

In [ ]:
if PHASE == 6:
    # ===================== 6.3  YOUR CODE — Phase-6 slots =====================
    # (a) The third adaptation technique (Phase-6 list: Houlsby / Pfeiffer adapters, prefix, prompt, LoRA, QLoRA, AdaLoRA, LoRMA, KD).
    #     Write it like your LoRMALinear + an apply_<name>(model) function, then register it:
    #         EXTRA_TECHNIQUES["houlsby"] = apply_houlsby
    #     apply_technique() then calls it on SummLM; everything you want trained must be requires_grad=True,
    #     and the parameter names you create must contain a word your apply function un-freezes (as "lora"/"lorma" do).

    # (b) KD student (TECHNIQUE = "kd"): same idea as your build_student5, but returning SummLM(enc).
    def build_student6(arch=None):
        raise NotImplementedError("YOUR CODE: build the smaller KD student (see your build_student5) and return SummLM(enc).to(device)")

    # (c) KD loss for generation. Called once per training document with
    #       s_logits (n, V)  student logits at the n summary positions
    #       t_logits (n, V)  teacher logits at the same positions (no grad)
    #       y        (n,)    reference next-token ids
    #     return a scalar loss. P6["KD_T"], P6["KD_ALPHA"] are your values in the CONTROL cell.
    def kd_loss6(s_logits, t_logits, y):
        raise NotImplementedError("YOUR CODE: KD loss (soft teacher targets + hard reference tokens)")


### 6.4 Harness: SummLM wrapper, training loop, cached decoding (greedy / beam), metrics, submission writer

In [ ]:
if PHASE == 6:
    # ===================== 6.4  Summarization harness (plumbing: wrapper, training, decoding, metrics, submission) =====================
    import contextlib, copy

    class SummLM(nn.Module):
        """Wrapper only: your Phase-1 LM unchanged under .encoder, so apply_technique() works as in Phases 3-5."""
        def __init__(self, enc):
            super().__init__(); self.encoder = enc

    def target_logits(enc, arch, idx, P, bidir):
        """idx (1, L) = src + [EOS sep] + tgt. Returns logits at positions P-1 .. L-2, i.e. the predictions of tgt tokens.
        bidir=True: the P prefix tokens attend to each other (your prefix_mask); else plain causal."""
        L = idx.size(1)
        if arch == "arch1":
            out = enc(idx); out = out[0] if isinstance(out, (tuple, list)) else out
            return out[:, P - 1:L - 1]
        mask = prefix_mask(L, torch.tensor([P], device=idx.device), idx.device) if bidir else None
        x = enc.tok_emb(idx)
        for block in enc.blocks:
            if HP.get("GRAD_CKPT") and enc.training and torch.is_grad_enabled():
                x = torch.utils.checkpoint.checkpoint(block, x, mask, use_reentrant=False)
            else:
                x = block(x, mask)
        return enc.head(enc.norm_f(x[:, P - 1:L - 1]))      # head only where the loss is taken (saves memory)

    def seq_of(it):
        ids = it["src"] + [EOS_ID] + it["tgt"]
        return torch.tensor([ids], device=device), len(it["src"]) + 1

    def moe_aux(enc):
        return sum(b.ffn.aux_loss for b in enc.blocks if getattr(b, "is_moe", False))

    @contextlib.contextmanager
    def swapped_hp(saved):
        old = dict(HP); HP.update(saved)
        try: yield
        finally: HP.clear(); HP.update(old)

    # ---------------- saving / loading a finished run ----------------
    def trainable_state(model):
        return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

    def save_run(model, fname, extra=None):
        torch.save(dict(arch=ARCH, technique=TECHNIQUE, hp=dict(HP), p6=dict(P6), state=trainable_state(model),
                        **(extra or {})), fname)
        print(f"saved {fname} ({os.path.getsize(fname) / 1e6:.0f} MB)")

    def load_run(fname):
        """Rebuild a saved Phase-6 model (teacher for KD, or 'predict' mode). Its backbone checkpoint must be attached."""
        ck = torch.load(find_input(fname), map_location="cpu", weights_only=False)
        with swapped_hp(ck["hp"]):
            if ck["technique"] == "kd":
                keep = P6["STUDENT_LAYERS"]; P6["STUDENT_LAYERS"] = ck["p6"]["STUDENT_LAYERS"]
                model = build_student6(ck["arch"]); P6["STUDENT_LAYERS"] = keep
            else:
                enc, _ = load_backbone(ck["arch"], "lm")
                model = apply_technique(SummLM(enc), ck["technique"])
        res = model.load_state_dict(ck["state"], strict=False)
        assert not res.unexpected_keys, f"saved weights do not match the rebuilt model: {res.unexpected_keys[:5]}"
        assert set(ck["state"]) <= {n for n, _ in model.named_parameters()}
        print(f"loaded {fname}: {ck['arch']} + {ck['technique']} | {len(ck['state'])} tensors | prefix bidirectional: {ck['p6']['PREFIX_BIDIR']}")
        return model.to(device).eval(), ck

    # ---------------- teacher-forced held-out loss ----------------
    @torch.no_grad()
    def heldout_loss(model, arch, items, bidir):
        model.eval(); tot = n = 0
        for it in items:
            idx, P = seq_of(it)
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=DEV_TYPE == "cuda"):
                lg = target_logits(model.encoder, arch, idx, P, bidir)
            tot += F.cross_entropy(lg[0].float(), idx[0, P:], reduction="sum").item(); n += idx.size(1) - P
        model.train(); return tot / max(1, n)

    # ---------------- training ----------------
    def train6(model, arch, teacher=None, t_arch=None, t_bidir=None, items=None, updates=None):
        """One document per forward (no padding); ACCUM documents per optimizer update.
        teacher != None -> KD: your kd_loss6 on (student, teacher) logits at the summary positions."""
        items = items if items is not None else D6["train"]
        bidir = P6["PREFIX_BIDIR"]
        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(params, lr=P6["LR"], weight_decay=P6["WD"])
        total_upd = updates or max(1, int(len(items) * P6["EPOCHS"] / P6["ACCUM"]))
        warm = max(1, int(P6["WARMUP_FRAC"] * total_upd))
        def lr_mult(u):                       # linear warm-up, then linear decay to 0
            return (u + 1) / warm if u < warm else max(0.0, (total_upd - u) / max(1, total_upd - warm))
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_mult)
        scaler = torch.amp.GradScaler("cuda", enabled=DEV_TYPE == "cuda")
        val_items = D6["val"][:P6["VAL_LOSS_N"]] if updates is None else []
        best, best_state, seen, upd, t0, run_loss = float("inf"), None, 0, 0, time.time(), []
        model.train(); opt.zero_grad(set_to_none=True)
        print(f"training {arch} | {len(items)} docs x {P6['EPOCHS']} epochs | {total_upd} updates of {P6['ACCUM']} docs | warmup {warm}")
        order = []
        while upd < total_upd:
            if not order:
                order = list(range(len(items))); random.Random(SEED + seen).shuffle(order)
            it = items[order.pop()]
            idx, P = seq_of(it)
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=DEV_TYPE == "cuda"):
                lg = target_logits(model.encoder, arch, idx, P, bidir)[0].float()
                y = idx[0, P:]
                if teacher is None:
                    loss = F.cross_entropy(lg, y)
                else:
                    with torch.no_grad():
                        tl = target_logits(teacher.encoder, t_arch, idx, P, t_bidir)[0].float()
                    loss = kd_loss6(lg, tl, y)
                if arch == "arch4" and P6.get("MOE_AUX_W"):
                    loss = loss + P6["MOE_AUX_W"] * moe_aux(model.encoder)
            scaler.scale(loss / P6["ACCUM"]).backward()
            run_loss.append(loss.item()); seen += 1
            if seen % P6["ACCUM"] == 0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, P6["CLIP"])
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); upd += 1
                if upd % 20 == 0 or upd == 1:
                    el = time.time() - t0
                    print(f"  upd {upd}/{total_upd} | loss {np.mean(run_loss[-20 * P6['ACCUM']:]):.4f} | lr {sched.get_last_lr()[0]:.2e} | "
                          f"{el / 60:.1f} min | ETA {el / upd * (total_upd - upd) / 60:.0f} min", flush=True)
            if val_items and seen % P6["EVAL_EVERY"] == 0:
                vl = heldout_loss(model, arch, val_items, bidir)
                tag = ""
                if vl < best:
                    best, best_state, tag = vl, trainable_state(model), "  <- best"
                print(f"  [held-out] after {seen} docs: loss {vl:.4f} (ppl {math.exp(vl):.1f}){tag}", flush=True)
            if updates is None and (time.time() - t0) / 3600 > P6["TIME_BUDGET_H"]:
                print(f"time budget {P6['TIME_BUDGET_H']} h reached after {upd} updates"); break
        if val_items:                                          # final check (end of schedule or time budget)
            vl = heldout_loss(model, arch, val_items, bidir)
            print(f"  [held-out] final: loss {vl:.4f}")
            if vl < best: best, best_state = vl, trainable_state(model)
        if best_state is not None:
            model.load_state_dict(best_state, strict=False); print(f"restored best held-out loss {best:.4f}")
        return model

    # ---------------- decoding (KV cache for arch2/3/4; full recompute for arch1) ----------------
    class KVDecoder:
        """Re-runs YOUR blocks' arithmetic incrementally (same weights, same RoPE tables, same GQA) with a key/value cache.
        Checked against your forward() by the self-test in 6.5."""
        def __init__(self, enc, bidir): self.enc, self.bidir = enc, bidir
        def _attn(self, a, x, pos, kv):
            B, T, C = x.shape
            q = a.q_proj(x).view(B, T, a.n_heads, a.d_head).transpose(1, 2)
            k = a.k_proj(x).view(B, T, a.n_kv_heads, a.d_head).transpose(1, 2)
            v = a.v_proj(x).view(B, T, a.n_kv_heads, a.d_head).transpose(1, 2)
            cos, sin = a.cos[pos:pos + T], a.sin[pos:pos + T]
            q, k = A3.apply_rope(q, cos, sin), A3.apply_rope(k, cos, sin)
            if kv is not None: k, v = torch.cat([kv[0], k], 2), torch.cat([kv[1], v], 2)
            kk, vv = k.repeat_interleave(a.n_rep, 1), v.repeat_interleave(a.n_rep, 1)
            s = (q * a.scale) @ kk.transpose(-2, -1)
            if T > 1:
                assert pos == 0
                if not self.bidir: s = s.masked_fill(~a.causal_mask[:T, :T], float("-inf"))
            out = (F.softmax(s.float(), -1).to(vv.dtype) @ vv).transpose(1, 2).reshape(B, T, C)
            return a.out_proj(out), (k, v)
        def run(self, ids, pos, cache):
            x, new = self.enc.tok_emb(ids), []
            for i, b in enumerate(self.enc.blocks):
                h, kv = self._attn(b.attn, b.norm1(x), pos, cache[i] if cache else None)
                x = x + h; x = x + b.ffn(b.norm2(x)); new.append(kv)
            return self.enc.head(self.enc.norm_f(x[:, -1])).float(), new
        @staticmethod
        def reorder(cache, ix): return [(k[ix], v[ix]) for k, v in cache]

    class FullDecoder:
        """arch1: no cache, the whole sequence is recomputed each step (slower)."""
        def __init__(self, enc, bidir): self.enc = enc
        def run(self, ids, pos, cache):
            seq = ids if cache is None else torch.cat([cache, ids], 1)
            out = self.enc(seq); out = out[0] if isinstance(out, (tuple, list)) else out
            return out[:, -1].float(), seq
        @staticmethod
        def reorder(cache, ix): return cache[ix]

    def _banned(seq, n):
        if not n or len(seq) < n - 1: return []
        pre = tuple(seq[len(seq) - n + 1:]) if n > 1 else ()
        return [seq[i + n - 1] for i in range(len(seq) - n + 1) if tuple(seq[i:i + n - 1]) == pre]

    def _adjust(logits, seqs, D, src_ids=None, span_map=None):
        for i, s in enumerate(seqs):
            if D["REP_PENALTY"] and D["REP_PENALTY"] != 1.0 and s:
                u = torch.tensor(sorted(set(s)), device=logits.device); r = logits[i, u]
                logits[i, u] = torch.where(r > 0, r / D["REP_PENALTY"], r * D["REP_PENALTY"])   # generated tokens only (copying from the document stays free)
            if D.get("COPY_BIAS") and src_ids is not None:
                logits[i, src_ids] += D["COPY_BIAS"]                       # every token that occurs in the model's input gets +COPY_BIAS
            if D.get("SPAN_BIAS") and span_map is not None and len(s) >= D["SPAN_N"]:
                nxt = span_map.get(tuple(s[-D["SPAN_N"]:]))               # the last SPAN_N generated tokens also occur in the input ->
                if nxt: logits[i, list(nxt)] += D["SPAN_BIAS"]             # boost the token(s) that follow them there (continue copying a phrase)
            b = _banned(s, D["NO_REPEAT_NGRAM"])
            if b: logits[i, b] = float("-inf")
            eos = logits[i, EOS_ID].clone()
            logits[i, tok.vocab_size:] = float("-inf")       # ids the tokenizer cannot decode (MASK row) are never emitted ...
            logits[i, EOS_ID] = eos                          # ... except EOS, which ends the summary
            if len(s) < D["MIN_NEW"]: logits[i, EOS_ID] = float("-inf")
        return logits

    @torch.no_grad()
    def generate(enc, arch, src, bidir, D):
        Dec = FullDecoder if arch == "arch1" else KVDecoder
        dec = Dec(enc, bidir)
        prefix = src + [EOS_ID]; P = len(prefix)
        max_new = min(D["MAX_NEW"], CTX - P)
        src_ids = torch.tensor(sorted(set(t for t in src if t < tok.vocab_size)), device=device) if D.get("COPY_BIAS") else None
        span_map = None
        if D.get("SPAN_BIAS"):
            n = D["SPAN_N"]; span_map = collections.defaultdict(set)
            for j in range(len(src) - n):
                span_map[tuple(src[j:j + n])].add(src[j + n])
        nb = D["BEAMS"] if D["DECODE"] == "beam" else 1
        with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=DEV_TYPE == "cuda"):
            logits, cache = dec.run(torch.tensor([prefix], device=device), 0, None)
            seqs, scores, done = [[]], torch.zeros(1, device=device), []
            for t in range(max_new):
                logits = _adjust(logits, seqs, D, src_ids, span_map)
                if nb == 1:
                    nxt = int(logits[0].argmax())
                    if nxt == EOS_ID: break
                    seqs[0].append(nxt)
                    ix = None
                    feed = torch.tensor([[nxt]], device=device)
                else:
                    lp = scores[:, None] + F.log_softmax(logits, -1)
                    top = lp.view(-1).topk(2 * nb)
                    new_seqs, new_sc, ix_l = [], [], []
                    for sc, flat in zip(top.values.tolist(), top.indices.tolist()):
                        bi, tk = divmod(flat, lp.size(1))
                        if tk == EOS_ID:
                            done.append((seqs[bi], sc / max(1, len(seqs[bi]) + 1) ** D["LEN_PENALTY"]))
                            continue
                        new_seqs.append(seqs[bi] + [tk]); new_sc.append(sc); ix_l.append(bi)
                        if len(new_seqs) == nb: break
                    if len(done) >= nb or not new_seqs: break
                    seqs, scores = new_seqs, torch.tensor(new_sc, device=device)
                    ix = torch.tensor(ix_l, device=device)
                    feed = torch.tensor([[s[-1]] for s in seqs], device=device)
                if ix is not None: cache = dec.reorder(cache, ix)
                logits, cache = dec.run(feed, P + t, cache)
        if nb == 1: return seqs[0]
        pool = done + [(s, sc / max(1, len(s)) ** D["LEN_PENALTY"]) for s, sc in zip(seqs, scores.tolist())]
        return max(pool, key=lambda z: z[1])[0]

    def detok(ids): return tok.decode([i for i in ids if i < tok.vocab_size]).strip()

    def trim_last(text):
        """P6["TRIM"]: drop an unfinished last sentence (generation stopped by MAX_NEW mid-sentence)."""
        m = list(re.finditer(r"[.?!](?=\s|$)", text))
        return text[:m[-1].end()].strip() if m else text

    def decode_set(model, arch, items, bidir, label, out_json=None):
        D = {k: P6[k] for k in ("DECODE", "BEAMS", "MIN_NEW", "MAX_NEW", "NO_REPEAT_NGRAM", "REP_PENALTY", "LEN_PENALTY")}
        D.update({k: P6.get(k) for k in ("COPY_BIAS", "SPAN_BIAS", "SPAN_N")})        # copy-biased decoding (None/0 = off)
        if D["SPAN_BIAS"]: assert D["SPAN_N"], "SPAN_BIAS needs SPAN_N (how many matching tokens start a copied phrase)"
        if D["COPY_BIAS"] or D["SPAN_BIAS"]: print(f"  copy-biased decoding: COPY_BIAS={D['COPY_BIAS']} SPAN_BIAS={D['SPAN_BIAS']} SPAN_N={D['SPAN_N']}")
        model.eval(); out, t0 = [], time.time()
        for i, it in enumerate(items):
            txt = detok(generate(model.encoder, arch, it["src"], bidir, D))
            out.append({"id": it["id"], "summary": trim_last(txt) if P6.get("TRIM") else txt})
            if i in (4, 49) or (i + 1) % 200 == 0 or i + 1 == len(items):
                el = time.time() - t0
                print(f"  [{label}] {i + 1}/{len(items)} | {el / (i + 1):.1f} s/doc | ETA {el / (i + 1) * (len(items) - i - 1) / 60:.0f} min", flush=True)
                if out_json: json.dump(out, open(out_json, "w"))
        return out

    def score6(preds, refs):
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score", "bert-score"], check=False)
        from rouge_score import rouge_scorer
        from bert_score import score as bscore
        sc = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
        r = [sc.score(a, b) for a, b in zip(refs, preds)]
        P_, R_, F_ = bscore(preds, refs, lang="en", verbose=False, batch_size=16, device=DEV_TYPE)
        res = dict(bertscore_f1=F_.mean().item(), bertscore_p=P_.mean().item(), bertscore_r=R_.mean().item(),
                   rouge1_f=float(np.mean([x["rouge1"].fmeasure for x in r])), rougeL_f=float(np.mean([x["rougeL"].fmeasure for x in r])),
                   words_pred=float(np.mean([len(p.split()) for p in preds])), words_ref=float(np.mean([len(x.split()) for x in refs])),
                   empty=int(sum(not p.strip() for p in preds)))
        print("  " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in res.items()))
        return res

    def write_submission6(preds, tag):
        ids = [d["id"] for d in test6]
        got = {p["id"]: p["summary"] for p in preds}
        assert len(preds) == len(ids) and set(got) == set(ids), "ids do not match test.json"
        empty = sum(not s.strip() for s in got.values())
        if empty: print(f"WARNING: {empty} empty summaries")
        json.dump([{"id": i, "summary": got[i]} for i in ids], open("prediction.json", "w"), ensure_ascii=False)
        zname = f"submission_{tag}.zip"
        with zipfile.ZipFile(zname, "w", zipfile.ZIP_DEFLATED) as z: z.write("prediction.json", arcname="prediction.json")
        with zipfile.ZipFile(zname) as z: assert z.namelist() == ["prediction.json"]
        print(f"wrote {zname}: {len(ids)} summaries, prediction.json at the zip root")


### 6.5 Run (`P6_MODE = "train"`) and 6.6 re-decode (`P6_MODE = "predict"`)

In [ ]:
if PHASE == 6 and P6_MODE == "train":
    # ===================== 6.5  Run (uses the CONTROL cell) =====================
    need6("LR", "WARMUP_FRAC", "WD", "CLIP", "ACCUM", "EPOCHS", "TIME_BUDGET_H", "EVAL_EVERY", "VAL_LOSS_N",
          "DECODE", "MIN_NEW", "MAX_NEW", "NO_REPEAT_NGRAM", "REP_PENALTY", "VAL_GEN_N")
    if P6["DECODE"] == "beam": need6("BEAMS", "LEN_PENALTY")
    assert P6["SRC_LEN"] + 1 + P6["MAX_NEW"] <= CTX, "SRC_LEN + 1 + MAX_NEW must fit in 1024"
    assert not (ARCH == "arch1" and P6["PREFIX_BIDIR"]), "arch1 is causal-only: PREFIX_BIDIR must be False"
    if CKPT_KIND != "lm": print(f"NOTE: CKPT_KIND = '{CKPT_KIND}' -- Phase-6 runs load the 'lm' checkpoint (load_backbone(ARCH, 'lm'))")
    TAG6 = f"p6_{ARCH}_{TECHNIQUE}" + (f"_{RUN_NOTE}" if RUN_NOTE else "")
    teacher = t_arch = t_bidir = None
    if TECHNIQUE == "kd":
        need6("TEACHER_FILE", "STUDENT_LAYERS", "KD_T", "KD_ALPHA")
        teacher, tck = load_run(P6["TEACHER_FILE"])
        for p in teacher.parameters(): p.requires_grad = False
        t_arch, t_bidir = tck["arch"], tck["p6"]["PREFIX_BIDIR"]
        for k in ("SELECT", "SRC_LEN", "TGT_LEN"):
            assert tck["p6"][k] == P6[k], f"teacher was trained with {k}={tck['p6'][k]}; use the same input setting for the KD student"
        model = build_student6(ARCH)
    elif P6.get("INIT_FROM"):
        # continue training a saved Phase-6 run (same arch, technique and input settings); its weights are the starting point
        model, ck0 = load_run(P6["INIT_FROM"])
        assert ck0["arch"] == ARCH and ck0["technique"] == TECHNIQUE, f"{P6['INIT_FROM']} is {ck0['arch']}+{ck0['technique']}, CONTROL says {ARCH}+{TECHNIQUE}"
        for k in ("SELECT", "SRC_LEN", "TGT_LEN", "PREFIX_BIDIR"):
            assert ck0["p6"][k] == P6[k], f"{P6['INIT_FROM']} was trained with {k}={ck0['p6'][k]}: use the same value"
        model.train()
        print(f"continuing from {P6['INIT_FROM']} ({sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable)")
    else:
        enc, _ = load_backbone(ARCH, "lm")
        model = apply_technique(SummLM(enc)).to(device)
    print(f"{TAG6}: prefix bidirectional {P6['PREFIX_BIDIR']} | input {P6['SELECT']} {P6['SRC_LEN']} tok | target {P6['TGT_LEN']} tok")

    # self-test: cached decoder == your forward() (teacher-forced logits on one held-out document)
    if ARCH != "arch1":
        with torch.no_grad():
            model.eval(); it = dict(D6["val"][0]); it["tgt"] = it["tgt"][:8]
            idx, P_ = seq_of(it)
            full = target_logits(model.encoder, ARCH, idx, P_, P6["PREFIX_BIDIR"])[0].float()
            dec = KVDecoder(model.encoder, P6["PREFIX_BIDIR"])
            lg, cache = dec.run(idx[:, :P_], 0, None); rows = [lg[0]]
            for j in range(len(it["tgt"]) - 1):
                lg, cache = dec.run(idx[:, P_ + j:P_ + j + 1], P_ + j, cache); rows.append(lg[0])
            err = (torch.stack(rows) - full).abs().max().item() / full.abs().max().item()
            print(f"self-test cached decoding vs forward(): max relative difference {err:.2e}")
            assert err < 1e-3, "cached decoder does not match the model -- do not continue"
            model.train()

    if P6["SANITY"]:
        # interactive only: memorise 8 documents; loss should fall towards 0 and the greedy output should copy the reference
        few = D6["train"][:8]
        _acc = P6["ACCUM"]; P6["ACCUM"] = 1
        train6(model, ARCH, teacher, t_arch, t_bidir, items=few, updates=120)
        P6["ACCUM"] = _acc
        out = decode_set(model, ARCH, few[:1], P6["PREFIX_BIDIR"], "sanity")[0]["summary"]
        print("GENERATED:", out[:400]); print("REFERENCE:", few[0]["ref"][:400])
        print("SANITY run finished -- set SANITY=False before committing")
    else:
        model = train6(model, ARCH, teacher, t_arch, t_bidir)
        save_run(model, f"{TAG6}.pt")
        del teacher; torch.cuda.empty_cache() if DEV_TYPE == "cuda" else None
        test_preds = decode_set(model, ARCH, D6["test"], P6["PREFIX_BIDIR"], "test", f"test_preds_{TAG6}.json")
        write_submission6(test_preds, TAG6)
        val_items = D6["val"][:P6["VAL_GEN_N"]]
        val_preds = decode_set(model, ARCH, val_items, P6["PREFIX_BIDIR"], "held-out", f"val_preds_{TAG6}.json")
        print(f"held-out scores ({len(val_items)} docs):")
        res = score6([p["summary"] for p in val_preds], [it["ref"] for it in val_items])
        json.dump(dict(tag=TAG6, scores=res, p6=P6, hp=HP), open(f"scores_{TAG6}.json", "w"), default=str, indent=1)
        print("example:", val_preds[0]["summary"][:500])

if PHASE == 6 and P6_MODE == "predict":
    # ===================== 6.6  Re-decode a saved run with the current decoding settings (no training) =====================
    need6("LOAD", "DECODE", "MIN_NEW", "MAX_NEW", "NO_REPEAT_NGRAM", "REP_PENALTY", "VAL_GEN_N")
    model, ck = load_run(P6["LOAD"])
    for k in ("SELECT", "SRC_LEN", "TGT_LEN", "PREFIX_BIDIR"):
        assert ck["p6"][k] == P6[k], f"{P6['LOAD']} was trained with {k}={ck['p6'][k]}: set the same value in P6"
    TAG6 = f"p6_{ck['arch']}_{ck['technique']}_pred" + (f"_{RUN_NOTE}" if RUN_NOTE else "")
    test_preds = decode_set(model, ck["arch"], D6["test"], P6["PREFIX_BIDIR"], "test", f"test_preds_{TAG6}.json")
    write_submission6(test_preds, TAG6)
    val_items = D6["val"][:P6["VAL_GEN_N"]]
    val_preds = decode_set(model, ck["arch"], val_items, P6["PREFIX_BIDIR"], "held-out", f"val_preds_{TAG6}.json")
    res = score6([p["summary"] for p in val_preds], [it["ref"] for it in val_items])
    json.dump(dict(tag=TAG6, scores=res, p6=P6), open(f"scores_{TAG6}.json", "w"), default=str, indent=1)


### 6.7 System combination (`P6_MODE = "combine"`): ROUGE-MBR over finished runs

In [ ]:
if PHASE == 6 and P6_MODE == "combine":
    # ===================== 6.7  System combination: ROUGE-based MBR over finished runs (plumbing, CPU or GPU) =====================
    # For every document, pick the candidate summary that agrees most with the OTHER systems' summaries
    # (Minimum Bayes Risk selection, Kumar & Byrne 2004; Freitag et al. 2022). The utility is ROUGE only --
    # no pretrained model is used to choose, so the submitted system stays your own models.
    # Attach the outputs of the runs listed in P6["COMBINE"] (they contain test_preds_<tag>.json and val_preds_<tag>.json).
    need6("COMBINE", "MBR_UTILITY")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score", "bert-score"], check=False)
    from rouge_score import rouge_scorer
    _rs = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    def util(a, b):
        r = _rs.score(b, a)
        return {"rouge1": r["rouge1"].fmeasure, "rougeL": r["rougeL"].fmeasure,
                "mix": (r["rouge1"].fmeasure + r["rougeL"].fmeasure) / 2}[P6["MBR_UTILITY"]]
    def mbr(cands):
        idx = [i for i, c in enumerate(cands) if c.strip()]
        if len(idx) <= 2: return max(idx or [0], key=lambda i: len(cands[i]))
        sc = {i: np.mean([util(cands[i], cands[j]) for j in idx if j != i]) for i in idx}
        return max(sc, key=sc.get)
    def load_preds(kind, tag):
        return {p["id"]: p["summary"] for p in json.load(open(find_input(f"{kind}_preds_{tag}.json")))}

    tags = list(P6["COMBINE"])
    T = {t: load_preds("test", t) for t in tags}
    test_ids = [d["id"] for d in test6]
    for t in tags: assert set(T[t]) == set(test_ids), f"{t}: test predictions do not cover test.json"
    V = {t: load_preds("val", t) for t in tags}
    common = [d for d in val6 if all(d["id"] in V[t] for t in tags)]
    assert common, "no held-out document decoded by every run -- were the runs made with the same VAL_N?"
    print(f"{len(tags)} systems | {len(common)} held-out docs decoded by all of them")

    picks = [mbr([V[t][d["id"]] for t in tags]) for d in common]
    print("held-out picks per system:", {t: picks.count(i) for i, t in enumerate(tags)})
    refs = [d["summary"] for d in common]
    for i, t in enumerate(tags):
        print(f"{t}:"); score6([V[t][d["id"]] for d in common], refs)
    print(f"MBR combination ({P6['MBR_UTILITY']}):")
    res = score6([V[tags[k]][d["id"]] for k, d in zip(picks, common)], refs)

    tp = [mbr([T[t][i] for t in tags]) for i in test_ids]
    TAG6 = f"p6_mbr_{len(tags)}sys" + (f"_{RUN_NOTE}" if RUN_NOTE else "")
    write_submission6([{"id": i, "summary": T[tags[k]][i]} for k, i in zip(tp, test_ids)], TAG6)
    json.dump(dict(tag=TAG6, systems=tags, scores=res, test_picks={t: tp.count(i) for i, t in enumerate(tags)}),
              open(f"scores_{TAG6}.json", "w"), indent=1)


### 6.8 Diagnostics (`P6_MODE = "diag"`): length, repetition, copying, ROUGE, BERTScore P/R/F vs copy-the-lead baselines

In [ ]:
if PHASE == 6 and P6_MODE == "diag":
    # ===================== 6.8  Diagnostics on finished runs (plumbing: no training; CPU or GPU) =====================
    # Reads val_preds_<tag>.json of each run in P6["COMBINE"] (attach those outputs), rebuilds the same held-out
    # split as 6.2 (VAL_N must match the runs) and prints, per system, next to copy-the-lead baselines:
    #   length, finished last sentence, repetition, how much is copied from the document, ROUGE, BERTScore P/R/F,
    #   and how often each system beats a length-matched lead baseline on the same document.
    need6("VAL_N", "COMBINE")
    import hashlib, subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score", "bert-score"], check=False)
    from rouge_score import rouge_scorer
    try:
        from bert_score import score as bscore
    except Exception:
        bscore = None; print("bert-score unavailable -> BERTScore columns skipped")

    def _read6(path):
        obj = json.load(open(path))
        if isinstance(obj, dict):
            vals = list(obj.values())
            obj = [dict(id=k, **v) for k, v in obj.items()] if vals and all(isinstance(v, dict) for v in vals) \
                  else next(v for v in vals if isinstance(v, list))
        out = []
        for r in obj:
            idk = next(k for k in ("id", "doc_id", "case_id") if k in r)
            d = r[next(k for k in ("document", "text", "judgment") if k in r)]
            s = r.get("summary", r.get("abstract", ""))
            out.append({"id": r[idk], "document": " ".join(map(str, d)) if isinstance(d, list) else str(d),
                        "summary": " ".join(map(str, s)) if isinstance(s, list) else str(s)})
        return out
    _all = _read6(P["Train/train.json"])
    _h = lambda s: hashlib.md5(re.sub(r"\s+", " ", s).strip().lower().encode()).hexdigest()
    _g = collections.defaultdict(list)
    for d in _all: _g[_h(d["document"])].append(d)
    _k = sorted(_g); random.Random(1234).shuffle(_k)
    val6d = []
    for k in _k:
        if len(val6d) >= P6["VAL_N"]: break
        val6d.extend(_g[k])
    DOC = {d["id"]: d for d in val6d}

    tags = list(P6["COMBINE"])
    PRED = {t: {p["id"]: p["summary"] for p in json.load(open(find_input(f"val_preds_{t}.json")))} for t in tags}
    ids = [d["id"] for d in val6d if all(d["id"] in PRED[t] for t in tags)]
    assert ids, "no held-out id shared by all runs and this split -- same VAL_N as the runs?"
    refs = [DOC[i]["summary"] for i in ids]
    print(f"{len(tags)} systems | {len(ids)} held-out docs | reference words mean {np.mean([len(r.split()) for r in refs]):.0f}")

    def lead(i, n): return " ".join(DOC[i]["document"].split()[:n])
    def ngrams(w, n): return [tuple(w[j:j + n]) for j in range(len(w) - n + 1)]
    _rs = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

    def stats(name, preds, docs_ids):
        W = [p.split() for p in preds]
        n = [len(w) for w in W]
        lw = [words_of(p) for p in preds]
        rep = [1 - len(set(ngrams(w, 3))) / max(1, len(ngrams(w, 3))) for w in lw]
        copy4 = []; lead4 = []
        for w, i in zip(lw, docs_ids):
            g = ngrams(w, 4)
            if not g: copy4.append(0.0); lead4.append(0.0); continue
            dw = words_of(DOC[i]["document"]); D = set(ngrams(dw, 4)); L = set(ngrams(dw[:700], 4))
            copy4.append(sum(x in D for x in g) / len(g)); lead4.append(sum(x in L for x in g) / len(g))
        fin = np.mean([bool(re.search(r"[.?!]\s*$", p.strip())) for p in preds])
        r = [_rs.score(a, b) for a, b in zip(refs, preds)]
        row = dict(name=name, words=f"{np.mean(n):.0f} (p10 {pct(n, .1)} / p50 {pct(n, .5)} / p90 {pct(n, .9)})",
                   ends_sentence=f"{fin:.0%}", repeated_3grams=f"{np.mean(rep):.1%}",
                   copied_4grams_from_doc=f"{np.mean(copy4):.1%}", of_which_in_first_700_words=f"{np.mean(lead4):.1%}",
                   rouge1=f"{np.mean([x['rouge1'].fmeasure for x in r]):.4f}", rougeL=f"{np.mean([x['rougeL'].fmeasure for x in r]):.4f}")
        return row

    systems = {t: [PRED[t][i] for i in ids] for t in tags}
    for t in tags:                                       # copy-the-lead baselines with the SAME word count as each system
        systems[f"lead-matched[{t}]"] = [lead(i, max(1, len(PRED[t][i].split()))) for i in ids]
    systems["lead-638w"] = [lead(i, 638) for i in ids]

    rows = [stats(name, preds, ids) for name, preds in systems.items()]
    BS = {}
    if bscore is not None:
        for name, preds in systems.items():
            P_, R_, F_ = bscore(preds, refs, lang="en", verbose=False, batch_size=16, device=DEV_TYPE)
            BS[name] = (P_.numpy(), R_.numpy(), F_.numpy())
        for row in rows:
            P_, R_, F_ = BS[row["name"]]
            row.update(bert_P=f"{P_.mean():.4f}", bert_R=f"{R_.mean():.4f}", bert_F=f"{F_.mean():.4f}")
    print()
    for row in rows:
        print(row.pop("name")); print("   " + " | ".join(f"{k} {v}" for k, v in row.items()))
    if BS:
        print("\nper-document BERTScore F1: system vs its length-matched lead baseline (same documents)")
        for t in tags:
            s, l = BS[t][2], BS[f"lead-matched[{t}]"][2]
            print(f"  {t}: beats lead on {np.mean(s > l):.0%} of docs | mean difference {np.mean(s - l):+.4f} | "
                  f"system F1 p10 {np.percentile(s, 10):.3f} / p50 {np.median(s):.3f} / p90 {np.percentile(s, 90):.3f}")
    print("\nexamples (first 70 words)")
    for i in ids[:2]:
        print(f"\n=== {i}\nREFERENCE: {' '.join(DOC[i]['summary'].split()[:70])}\nLEAD     : {lead(i, 70)}")
        for t in tags: print(f"{t}: {' '.join(PRED[t][i].split()[:70])}")


# PHASE 7 — Prior Case Retrieval
_Section added when the phase opens. Data files are already listed in `PHASE_FILES`._

### 7.0 Helpers

In [ ]:
if PHASE == 7:
    # ===================== 7.0  Phase-7 helpers (plumbing): loading, metric, TF-IDF ranking, chunking =====================
    import scipy.sparse as sp
    from sklearn.feature_extraction.text import TfidfVectorizer

    def _txt(v):
        if isinstance(v, dict):
            v = v.get("text", v.get("document", v.get("sentences", "")))
        if isinstance(v, list): return " ".join(map(str, v)), len(v)
        return str(v), None

    def _show(name, obj):
        if isinstance(obj, dict):
            k = next(iter(obj)); v = obj[k]
            print(f"{name}: dict with {len(obj)} keys | first key {k!r} -> {type(v).__name__}"
                  + (f" keys {list(v)[:6]}" if isinstance(v, dict) else ""))
        elif isinstance(obj, list):
            v = obj[0]; print(f"{name}: list of {len(obj)} | first item {type(v).__name__}" + (f" keys {list(v)[:8]}" if isinstance(v, dict) else ""))

    def load7(path, name, verbose=True):
        """-> {id(str): {"text": str, "n_sent": int|None, "rel": [ids] or None}}"""
        obj = json.load(open(path))
        if verbose: _show(name, obj)
        if isinstance(obj, dict) and len(obj) <= 3 and any(isinstance(v, list) and v and isinstance(v[0], dict) for v in obj.values()):
            obj = next(v for v in obj.values() if isinstance(v, list) and v and isinstance(v[0], dict))
        out = {}
        items = obj.items() if isinstance(obj, dict) else ((r[next(k for k in ("id", "query_id", "candidate_id", "case_id", "doc_id") if k in r)], r) for r in obj)
        for k, v in items:
            t, n = _txt(v)
            rel = v.get("relevant_candidates") if isinstance(v, dict) else None
            out[str(k)] = {"text": t, "n_sent": n, "rel": [str(x) for x in rel] if rel is not None else None}
        return out

    def load_all7(verbose=True):
        S = {}
        for split, qf, cf in (("train", "Train/train.json", "Train/train_candidates.json"),
                              ("dev", "Dev/dev.json", "Dev/dev_candidates.json"),
                              ("test", "Test/test.json", "Test/test_candidates.json")):
            S[split] = (load7(P[qf], f"{split} queries", verbose), load7(P[cf], f"{split} candidates", verbose))
        return S

    def micro7(ranked, Q, K=7):
        """official metric: micro P/R/F1 over all queries, top-K predicted vs the full gold set"""
        tp = sum(len(set(ranked[q][:K]) & set(Q[q]["rel"])) for q in ranked)
        npred = K * len(ranked); ngold = sum(len(Q[q]["rel"]) for q in ranked)
        p_, r_ = tp / max(1, npred), tp / max(1, ngold)
        return p_, r_, (2 * p_ * r_ / (p_ + r_) if p_ + r_ else 0.0)

    def recall_at(ranked, Q, K):
        return sum(len(set(ranked[q][:K]) & set(Q[q]["rel"])) for q in ranked) / max(1, sum(len(Q[q]["rel"]) for q in ranked))

    def tfidf_rank(Q, C, K):
        """TF-IDF (uni+bigram, sublinear tf) fitted on the split's own pool; the query itself is never returned.
        -> {qid: [(cid, score), ...] top K, best first}"""
        cids, qids = list(C), list(Q)
        vec = TfidfVectorizer(sublinear_tf=True, stop_words="english", ngram_range=(1, 2), min_df=2, max_features=300000)
        Cm = vec.fit_transform(C[c]["text"] for c in cids); Qm = vec.transform(Q[q]["text"] for q in qids)
        S = (Qm @ Cm.T).toarray()
        idx = {c: j for j, c in enumerate(cids)}
        out = {}
        for i, q in enumerate(qids):
            s = S[i]
            if q in idx: s[idx[q]] = -np.inf
            top = np.argsort(-s)[:K]
            out[q] = [(cids[j], float(s[j])) for j in top]
        return out

    def write_prediction7(ranked, Q, C, fname, zip_name=None):
        out = {}
        for q in Q:
            top = [c for c in ranked[q] if c in C and c != q][:7]
            assert len(top) == 7 and len(set(top)) == 7, f"query {q}: could not fill 7 unique candidates"
            out[q] = top
        json.dump(out, open(fname, "w"))
        msg = f"wrote {fname}: {len(out)} queries x 7 unique pool candidates"
        if zip_name:
            with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z: z.write(fname, arcname="prediction.json")
            with zipfile.ZipFile(zip_name) as z: assert z.namelist() == ["prediction.json"]
            msg += f" -> {zip_name} (prediction.json at the zip root)"
        print(msg)


### 7.1 Data + TF-IDF baseline (`P7_MODE = "data"`)

In [ ]:
if PHASE == 7 and P7_MODE == "data":
    # ===================== 7.1  Data, statistics and lexical baselines (plumbing, CPU) =====================
    try:
        from huggingface_hub import HfApi
        from kaggle_secrets import UserSecretsClient
        _tk = UserSecretsClient().get_secret("HF_TOKEN")
        print("official Phase-7 files:", [f for f in HfApi().list_repo_files(REPO, repo_type="dataset", token=_tk) if f.startswith("Phase_7_PCR/")])
    except Exception as e:
        print("(could not list the HF folder:", e, ")")
    S7 = load_all7()
    print("\n=== sizes")
    for split, (Q, C) in S7.items():
        line = f"{split:5s}: {len(Q):5d} queries | pool {len(C):6d} candidates | queries also inside their own pool: {sum(q in C for q in Q)}"
        if all(v["rel"] is not None for v in Q.values()):
            r = [len(v["rel"]) for v in Q.values()]
            line += (f"\n       relevant per query: mean {np.mean(r):.2f} | p10 {pct(r, .1)} p50 {pct(r, .5)} p90 {pct(r, .9)} max {max(r)} "
                     f"| queries with 0 relevant {sum(x == 0 for x in r)}"
                     f"\n       best possible micro-F1@7 (perfect top-7): {2 * sum(min(7, x) for x in r) / (7 * len(r) + sum(r)):.3f}")
        print(line)
    print("\n=== TF-IDF baseline (micro-F1@7, R@K)")
    for split in ("train", "dev"):
        Q, C = S7[split]; t0 = time.time(); tr = tfidf_rank(Q, C, 200); ranked = {q: [c for c, _ in v] for q, v in tr.items()}
        p_, r_, f_ = micro7(ranked, Q)
        print(f"  {split:5s}: micro-F1@7 {f_:.4f} (P {p_:.4f} R {r_:.4f}) | " + " ".join(f"R@{K} {recall_at(ranked, Q, K):.3f}" for K in (7, 20, 50, 100, 200)) + f" | {time.time() - t0:.0f}s")
    Q, C = S7["test"]; tr = tfidf_rank(Q, C, 50)
    write_prediction7({q: [c for c, _ in v] for q, v in tr.items()}, Q, C, "baseline_tfidf_test_prediction.json", "sub_p7_tfidf_baseline.zip")
    print("(lexical baseline = diagnostic / format check, not a Phase-1-model submission)")


### 7.2 Retrieval harness: chunked embeddings, InfoNCE training, dev fusion, submission

In [ ]:
if PHASE == 7:
    # ===================== 7.2  Retrieval harness (plumbing) =====================
    # Pipeline: TF-IDF top-TOPK per query (candidate generation)  ->  your encoder embeds query and candidates
    # (chunked, POOL-ed over the chunks' token states, L2-normalised)  ->  cosine similarity  ->  per-query
    # z-scored fusion  a*TF-IDF + (1-a)*neural, with a chosen on DEV by an exhaustive grid over [0, 1]  ->  top-7.
    # Training: InfoNCE over (query, one relevant candidate, N_NEG hard negatives = TF-IDF top-NEG_TOP non-relevant).
    import multiprocessing as mp, pickle, contextlib

    def need7(*keys):
        missing = [k for k in keys if P7.get(k) is None]
        assert not missing, f"fill these in P7 (CONTROL cell) first: {missing}"

    class RetModel(nn.Module):
        """Wrapper only: your Phase-1 backbone unchanged under .encoder (so apply_technique works as before)."""
        def __init__(self, enc): super().__init__(); self.encoder = enc

    @contextlib.contextmanager
    def swapped_hp(saved):
        old = dict(HP); HP.update(saved)
        try: yield
        finally: HP.clear(); HP.update(old)

    # ---------------- tokenisation into chunks (parallel, cached) ----------------
    def _chunk_ids(ids, L, stride, n_max):
        out, s = [], 0
        while s < len(ids) and len(out) < n_max:
            out.append(ids[s:s + L])
            if s + L >= len(ids): break
            s += stride
        return out

    def _tok_job(args):
        did, head, tail = args
        return did, tok.encode(head), (tok.encode(tail) if tail else None)

    def build_chunks(texts, L, overlap, n_max, keep, tpw):
        """texts {id: str} -> {id: [chunk ids, ...]} (at most n_max chunks; keep "first" | "both" = start + end)"""
        stride = max(1, int(L * (1 - overlap)))
        max_tok = L + (n_max - 1) * stride
        jobs = []
        for d, t in texts.items():
            w = t.split(); need_w = int(max_tok / tpw * 1.15) + 20
            if keep == "both" and len(w) > need_w:
                hw = int((L + (-(-n_max // 2) - 1) * stride) / tpw * 1.15) + 20
                tw = int((L + (n_max // 2 - 1) * stride) / tpw * 1.15) + 20 if n_max >= 2 else 0
                jobs.append((d, " ".join(w[:hw]), " ".join(w[-tw:]) if tw else None))
            else:
                jobs.append((d, " ".join(w[:need_w]), None))
        t0 = time.time(); out = {}
        with mp.get_context("fork").Pool(max(1, os.cpu_count() or 1)) as pool:
            for i, (d, h, t) in enumerate(pool.imap_unordered(_tok_job, jobs, chunksize=8)):
                if t is None:
                    out[d] = _chunk_ids(h, L, stride, n_max)
                else:
                    nh = -(-n_max // 2); nt = n_max - nh
                    out[d] = _chunk_ids(h, L, stride, nh) + _chunk_ids(t, L, stride, 10 ** 6)[-nt:]
                if (i + 1) % 1000 == 0: print(f"  tokenised {i + 1}/{len(jobs)} docs ({time.time() - t0:.0f}s)", flush=True)
        print(f"tokenised {len(out)} docs into chunks of {L} (stride {stride}, max {n_max}, keep {keep}) in {time.time() - t0:.0f}s")
        return out

    # ---------------- encoding ----------------
    def window_states(enc, arch, idx, lens, bidir):
        if arch == "arch1":
            return A1.states_a1(enc, idx)
        return backbone_states(enc, idx, lens if bidir else None)

    def embed_batch(model, arch, docs_chunks, bidir, pool):
        """docs_chunks: list (per doc) of list of chunk id-lists -> (n_docs, d) L2-normalised (with grad if training)"""
        wins, owner = [], []
        for di, ch in enumerate(docs_chunks):
            for c in ch: wins.append(c); owner.append(di)
        L = max(len(w) for w in wins)
        idx = torch.zeros(len(wins), L, dtype=torch.long, device=device)
        for i, w in enumerate(wins): idx[i, :len(w)] = torch.tensor(w, device=device)
        lens = torch.tensor([len(w) for w in wins], device=device)
        H = window_states(model.encoder, arch, idx, lens, bidir).float()               # (n_win, L, d)
        m = (torch.arange(L, device=device)[None, :] < lens[:, None]).float()[..., None]
        own = torch.tensor(owner, device=device); nd = len(docs_chunks)
        if pool == "mean":
            s = torch.zeros(nd, H.size(-1), device=device).index_add(0, own, (H * m).sum(1))
            n = torch.zeros(nd, device=device).index_add(0, own, m.squeeze(-1).sum(1))
            v = s / n[:, None].clamp_min(1)
        else:   # "max"
            hm = (H + (m - 1) * 1e4).amax(1)                                              # (n_win, d), pads excluded
            v = torch.full((nd, H.size(-1)), -1e4, device=device).index_reduce(0, own, hm, "amax", include_self=True)
        return F.normalize(v, dim=-1)

    @torch.no_grad()
    def embed_all(model, arch, ids, chunks, n_chunks, bidir, pool, win_batch=8):
        model.eval(); out = {}; t0 = time.time()
        order = sorted(ids, key=lambda d: -len(chunks[d]))
        batch, nwin = [], 0
        def flush():
            if not batch: return
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=DEV_TYPE == "cuda"):
                V = embed_batch(model, arch, [chunks[d][:n_chunks] for d in batch], bidir, pool)
            for d, v in zip(batch, V): out[d] = v.float().cpu()
        for i, d in enumerate(order):
            k = min(n_chunks, len(chunks[d]))
            if nwin + k > win_batch and batch: flush(); batch.clear(); nwin = 0
            batch.append(d); nwin += k
            if (i + 1) % 500 == 0: print(f"  embedded {i + 1}/{len(order)} docs ({time.time() - t0:.0f}s)", flush=True)
        flush()
        print(f"embedded {len(out)} docs ({n_chunks} chunks max) in {time.time() - t0:.0f}s")
        return out

    # ---------------- saving / loading runs ----------------
    def trainable_state(model):
        return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

    def save_run7(model, fname):
        torch.save(dict(arch=ARCH, technique=TECHNIQUE, hp=dict(HP), p7=dict(P7), state=trainable_state(model)), fname)
        print(f"saved {fname} ({os.path.getsize(fname) / 1e6:.0f} MB)")

    def load_run7(fname):
        ck = torch.load(find_input(fname), map_location="cpu", weights_only=False)
        with swapped_hp(ck["hp"]):
            if ck["technique"] == "kd":
                keep = P7.get("STUDENT_LAYERS"); P7["STUDENT_LAYERS"] = ck["p7"]["STUDENT_LAYERS"]
                model = build_student7(ck["arch"], ck["p7"]["CKPT"]); P7["STUDENT_LAYERS"] = keep
            else:
                enc, _ = load_backbone(ck["arch"], ck["p7"]["CKPT"])
                model = apply_technique(RetModel(enc), ck["technique"])
        res = model.load_state_dict(ck["state"], strict=False)
        assert not res.unexpected_keys, f"saved weights do not match the rebuilt model: {res.unexpected_keys[:5]}"
        print(f"loaded {fname}: {ck['arch']} + {ck['technique']} | pool {ck['p7']['POOL']} | bidirectional {ck['p7']['BIDIR_ENC']} | tau {ck['p7']['TAU']}")
        return model.to(device).eval(), ck

    # ---------------- training ----------------
    def train7(model, arch, pairs, hardneg, chunks, teacher=None, t_ck=None, updates=None):
        need7("LR", "WARMUP_FRAC", "WD", "CLIP", "ACCUM", "EPOCHS", "TIME_BUDGET_H", "TRAIN_CHUNKS", "N_NEG", "TAU", "POOL", "BIDIR_ENC")
        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(params, lr=P7["LR"], weight_decay=P7["WD"])
        total = updates or max(1, int(len(pairs) * P7["EPOCHS"] / P7["ACCUM"]))
        warm = max(1, int(P7["WARMUP_FRAC"] * total))
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda u: (u + 1) / warm if u < warm else max(0.0, (total - u) / max(1, total - warm)))
        scaler = torch.amp.GradScaler("cuda", enabled=DEV_TYPE == "cuda")
        rng = random.Random(SEED); order, seen, upd, t0, hist = [], 0, 0, time.time(), []
        print(f"training {arch} | {len(pairs)} (query, relevant) pairs x {P7['EPOCHS']} epochs | {total} updates of {P7['ACCUM']} | "
              f"{P7['N_NEG']} hard negatives | {P7['TRAIN_CHUNKS']} chunk(s) per document | warmup {warm}")
        model.train(); opt.zero_grad(set_to_none=True)
        while upd < total:
            if not order: order = list(range(len(pairs))); rng.shuffle(order)
            q, pos = pairs[order.pop()]
            pool_ = hardneg[q]
            negs = rng.sample(pool_, min(P7["N_NEG"], len(pool_)))
            docs = [q, pos] + negs
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=DEV_TYPE == "cuda"):
                V = embed_batch(model, arch, [chunks[d][:P7["TRAIN_CHUNKS"]] for d in docs], P7["BIDIR_ENC"], P7["POOL"])
                logits = (V[1:] @ V[0]).float()[None, :] / P7["TAU"]
                y = torch.zeros(1, dtype=torch.long, device=device)
                if teacher is None:
                    loss = F.cross_entropy(logits, y)
                else:
                    tp = t_ck["p7"]
                    with torch.no_grad():
                        TV = embed_batch(teacher, t_ck["arch"], [chunks[d][:P7["TRAIN_CHUNKS"]] for d in docs], tp["BIDIR_ENC"], tp["POOL"])
                        t_logits = (TV[1:] @ TV[0]).float()[None, :] / tp["TAU"]
                    loss = kd_loss7(logits, t_logits, y)
            scaler.scale(loss / P7["ACCUM"]).backward(); hist.append(loss.item()); seen += 1
            if seen % P7["ACCUM"] == 0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, P7["CLIP"])
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); upd += 1
                if upd % 20 == 0 or upd == 1:
                    el = time.time() - t0
                    print(f"  upd {upd}/{total} | loss {np.mean(hist[-20 * P7['ACCUM']:]):.4f} | lr {sched.get_last_lr()[0]:.2e} | "
                          f"{el / 60:.1f} min | ETA {el / upd * (total - upd) / 60:.0f} min", flush=True)
            if updates is None and (time.time() - t0) / 3600 > P7["TIME_BUDGET_H"]:
                print(f"time budget {P7['TIME_BUDGET_H']} h reached after {upd} updates"); break
        return model

    # ---------------- re-ranking, fusion on dev, test submission ----------------
    def zs(x):
        x = np.asarray(x, dtype=np.float64); s = x.std()
        return (x - x.mean()) / s if s > 0 else x * 0

    def fused_rank(lists, neural, a):
        out = {}
        for q, lst in lists.items():
            cids = [c for c, _ in lst]
            t = zs([s for _, s in lst]); n = zs([float(neural[q] @ neural[c]) for c in cids])
            f = a * t + (1 - a) * n
            out[q] = [cids[j] for j in np.argsort(-f)]
        return out

    def evaluate_and_submit(model, arch, chunks, S7, lists, tag, bidir, pool):
        need7("MAX_CHUNKS")
        Qd, Cd = S7["dev"]; Qt, Ct = S7["test"]
        dev_ids = set(Qd) | set(Cd)                     # whole dev pool (saved for the later fusion lab)
        Ed = embed_all(model, arch, dev_ids, chunks, P7["MAX_CHUNKS"], bidir, pool)
        grid = [round(x, 2) for x in np.arange(0, 1.0001, 0.05)]
        res = {a: micro7(fused_rank(lists["dev"], Ed, a), Qd) for a in grid}
        best_a = max(grid, key=lambda a: res[a][2])
        print(f"DEV micro-F1@7 | TF-IDF only (a=1): {res[1.0][2]:.4f} | your model only (a=0): {res[0.0][2]:.4f} | "
              f"best fusion a={best_a}: {res[best_a][2]:.4f} (P {res[best_a][0]:.4f} R {res[best_a][1]:.4f})")
        print("  fusion curve: " + " ".join(f"{a}:{res[a][2]:.3f}" for a in grid))
        test_ids = set(Qt) | set(Ct)                    # whole test pool
        Et = embed_all(model, arch, test_ids, chunks, P7["MAX_CHUNKS"], bidir, pool)
        torch.save({"dev": {d: v.half() for d, v in Ed.items()}, "test": {d: v.half() for d, v in Et.items()},
                    "arch": arch, "tag": tag}, f"emb_{tag}.pt")
        print(f"saved emb_{tag}.pt (document embeddings of the whole dev and test pools, for the fusion lab)")
        write_prediction7(fused_rank(lists["test"], Et, best_a), Qt, Ct, f"prediction_{tag}.json", f"submission_{tag}.zip")
        write_prediction7(fused_rank(lists["test"], Et, 0.0), Qt, Ct, f"prediction_{tag}_neural_only.json", f"submission_{tag}_neural_only.zip")
        json.dump(dict(tag=tag, best_a=best_a, dev={str(a): res[a] for a in grid}, p7=P7, hp=HP),
                  open(f"scores_{tag}.json", "w"), default=str, indent=1)


### 7.3 KD student + KD loss (your code, ported)

In [ ]:
if PHASE == 7:
    # ===================== 7.3  KD student + KD loss =====================
    # STUDENT'S CODE, ported to Phase 7 by Claude with mechanical changes only (for the LLM declaration):
    #   build_student7 = the student's build_student5 (load_backbone(arch, kind), P7 instead of HP, RetModel instead of LSIModel)
    #   kd_loss7       = the student's Phase-3/4 kd_loss (cross-entropy + temperature-softened KL x T^2, alpha mix);
    #                    the "classes" are the query's candidates [relevant, negatives...], y = index of the relevant one
    def build_student7(arch=None, kind=None):
        # KD runs only: smaller student, every parameter trains
        enc, d_model = load_backbone(arch or ARCH, kind or P7["CKPT"])
        initial_layers = len(enc.blocks)
        enc.blocks = enc.blocks[:P7["STUDENT_LAYERS"]]
        assert len(enc.blocks) < initial_layers
        model = RetModel(enc)
        for p in model.parameters(): p.requires_grad = True
        print(f"KD student: kept {len(enc.blocks)} of {initial_layers} blocks")
        return model.to(device)

    def kd_loss7(s_logits, t_logits, y):
        hard_loss = torch.nn.functional.cross_entropy(s_logits, y)

        T = P7["KD_T"]
        alpha = P7["KD_ALPHA"]

        student_log_probs = torch.nn.functional.log_softmax(s_logits / T, dim=-1)
        teacher_probs = torch.nn.functional.softmax(t_logits / T, dim=-1)

        soft_loss = torch.nn.functional.kl_div(
            student_log_probs,
            teacher_probs,
            reduction='batchmean'
        ) * (T ** 2)

        return (1.0 - alpha) * hard_loss + alpha * soft_loss


### 7.4 Run (`P7_MODE = "train"` / `"predict"`)

In [ ]:
if PHASE == 7 and P7_MODE in ("train", "predict"):
    # ===================== 7.4  Run: (train) -> dev fusion -> test submission =====================
    need7("CKPT", "CHUNK_LEN", "OVERLAP", "MAX_CHUNKS", "KEEP", "TOPK")
    S7 = load_all7(verbose=False)
    texts = {}
    for split, (Q, C) in S7.items():
        for d, v in list(Q.items()) + list(C.items()): texts.setdefault(d, v["text"])
    _s = random.Random(0).sample(list(texts.values()), min(30, len(texts)))
    TPW = sum(len(tok.encode(" ".join(t.split()[:1500]))) for t in _s) / sum(min(1500, len(t.split())) for t in _s)
    cache = f"p7chunks_{P7['CHUNK_LEN']}_{P7['OVERLAP']}_{P7['MAX_CHUNKS']}_{P7['KEEP']}.pkl"
    hit = glob.glob(f"/kaggle/input/**/{cache}", recursive=True)
    if hit:
        CH = pickle.load(open(hit[0], "rb")); print("loaded chunks from", hit[0])
    else:
        CH = build_chunks(texts, P7["CHUNK_LEN"], P7["OVERLAP"], P7["MAX_CHUNKS"], P7["KEEP"], TPW)
        pickle.dump(CH, open(cache, "wb")); print("saved", cache)
    LISTS = {s: tfidf_rank(*S7[s], P7["TOPK"]) for s in ("dev", "test")}
    Qd = S7["dev"][0]
    print(f"TF-IDF top-{P7['TOPK']}: dev recall {recall_at({q: [c for c, _ in v] for q, v in LISTS['dev'].items()}, Qd, P7['TOPK']):.3f} "
          f"(ceiling for re-ranking) | dev micro-F1@7 {micro7({q: [c for c, _ in v] for q, v in LISTS['dev'].items()}, Qd)[2]:.4f}")
    TAG7 = f"p7_{ARCH}_{TECHNIQUE}" + (f"_{RUN_NOTE}" if RUN_NOTE else "")

    if P7_MODE == "predict":
        need7("LOAD")
        model, ck = load_run7(P7["LOAD"])
        TAG7 = f"p7_{ck['arch']}_{ck['technique']}_pred" + (f"_{RUN_NOTE}" if RUN_NOTE else "")
        evaluate_and_submit(model, ck["arch"], CH, S7, LISTS, TAG7, ck["p7"]["BIDIR_ENC"], ck["p7"]["POOL"])
    else:
        need7("BIDIR_ENC", "POOL", "TRAIN_CHUNKS", "N_NEG", "NEG_TOP", "TAU")
        assert not (ARCH == "arch1" and P7["BIDIR_ENC"]), "arch1 is causal-only: BIDIR_ENC must be False"
        Qtr, Ctr = S7["train"]
        trl = tfidf_rank(Qtr, Ctr, P7["NEG_TOP"] + max(len(v["rel"]) for v in Qtr.values()))
        HN = {q: [c for c, _ in trl[q] if c not in set(Qtr[q]["rel"])][:P7["NEG_TOP"]] for q in Qtr}
        PAIRS = [(q, p) for q in Qtr for p in Qtr[q]["rel"] if p in CH and p != q and HN[q]]
        print(f"{len(PAIRS)} training pairs from {sum(1 for q in Qtr if Qtr[q]['rel'])} queries | hard negatives per query up to {P7['NEG_TOP']}")
        teacher = t_ck = None
        if TECHNIQUE == "kd":
            need7("TEACHER_FILE", "STUDENT_LAYERS", "KD_T", "KD_ALPHA")
            teacher, t_ck = load_run7(P7["TEACHER_FILE"])
            for p in teacher.parameters(): p.requires_grad = False
            model = build_student7(ARCH)
        else:
            enc, _ = load_backbone(ARCH, P7["CKPT"])
            model = apply_technique(RetModel(enc)).to(device)
        print(f"{TAG7}: ckpt {P7['CKPT']} | bidirectional {P7['BIDIR_ENC']} | pool {P7['POOL']} | tau {P7['TAU']}")
        if P7["SANITY"]:
            few = PAIRS[:8]; _acc = P7["ACCUM"]; P7["ACCUM"] = 1
            train7(model, ARCH, few, HN, CH, teacher, t_ck, updates=80)
            P7["ACCUM"] = _acc
            print("SANITY finished -- loss should fall towards 0 on these 8 pairs; set SANITY=False before committing")
        else:
            model = train7(model, ARCH, PAIRS, HN, CH, teacher, t_ck)
            save_run7(model, f"{TAG7}.pt")
            del teacher; torch.cuda.empty_cache() if DEV_TYPE == "cuda" else None
            evaluate_and_submit(model, ARCH, CH, S7, LISTS, TAG7, P7["BIDIR_ENC"], P7["POOL"])
